## Blockchain Resilience Infrastructure

In [1]:
# ================================================================
# COMPLETE BRI QUALITATIVE CODING PIPELINE
# INPUT FILE: Themes.xlsx
# PARTICIPANTS: P01-P26
# ================================================================

import pandas as pd
from pathlib import Path

# ================================================================
# 1. FIND INPUT FILE AUTOMATICALLY
# ================================================================

possible_files = [
    Path("Themes.xlsx"),
    Path("Themes.xls"),
    Path("Themes.xlsm")
]

input_file = None

for f in possible_files:
    if f.exists():
        input_file = f
        break

if input_file is None:
    raise FileNotFoundError(
        "Themes file was not found. Put 'Themes.xlsx' in the same "
        "folder as your Jupyter notebook/Python script."
    )

print("=" * 70)
print("INPUT FILE FOUND")
print("=" * 70)
print(input_file.resolve())


# ================================================================
# 2. READ EXCEL WORKBOOK
# ================================================================

excel_file = pd.ExcelFile(input_file)

print("\nSheets found:")
print(excel_file.sheet_names)


# ================================================================
# 3. EXTRACT BRI FROM P01-P26
# ================================================================

original_rows = []
raw_rows = []

for sheet in excel_file.sheet_names:

    sheet_name = str(sheet).strip().upper()

    # ------------------------------------------------------------
    # Convert sheet name into participant number
    # Accepts:
    # P01, P1, 01, 1
    # Ignores:
    # Sheet1, Coding Dictionary, etc.
    # ------------------------------------------------------------

    if sheet_name.startswith("P"):
        number_part = sheet_name[1:]
    else:
        number_part = sheet_name

    try:
        participant_number = int(number_part)
    except (ValueError, TypeError):
        continue

    # Only process P01-P26
    if not 1 <= participant_number <= 26:
        continue

    participant = f"P{participant_number:02d}"

    print(f"Processing {sheet} -> {participant}")

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    # ------------------------------------------------------------
    # Find BRI row
    # Assumes:
    # column 2 = construct
    # column 3 = themes
    # ------------------------------------------------------------

    found_bri = False

    for _, row in df.iterrows():

        if len(row) < 3:
            continue

        construct = str(row.iloc[1]).strip().upper()

        if construct == "BRI":

            found_bri = True

            original_text = str(row.iloc[2]).strip()

            # Original participant response
            original_rows.append({
                "Participant": participant,
                "Construct": "BRI",
                "Original_BRI_Themes": original_text
            })

            # Split comma-separated themes
            themes = original_text.split(",")

            for theme in themes:

                theme = str(theme).strip()

                if theme and theme.lower() != "nan":

                    raw_rows.append({
                        "Participant": participant,
                        "Construct": "BRI",
                        "Raw_Theme": theme
                    })

            break

    if not found_bri:
        print(f"   WARNING: BRI not found in {participant}")


# ================================================================
# 4. CREATE RAW BRI DATAFRAME
# ================================================================

original_bri = pd.DataFrame(original_rows)

raw_bri = pd.DataFrame(raw_rows)

print("\n" + "=" * 70)
print("STAGE 1 — RAW BRI EXTRACTION")
print("=" * 70)

print(
    "Participants found:",
    original_bri["Participant"].nunique()
)

print(
    "Raw BRI observations:",
    len(raw_bri)
)


# ================================================================
# 5. CLEAN RAW THEMES
# ================================================================

clean_bri = raw_bri.copy()

clean_bri["Theme_Key"] = (
    clean_bri["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove empty values
clean_bri = clean_bri[
    clean_bri["Theme_Key"].notna()
    & (clean_bri["Theme_Key"] != "")
    & (clean_bri["Theme_Key"] != "nan")
].copy()

# Remove duplicate mention of the same theme by same participant
clean_bri = clean_bri.drop_duplicates(
    subset=["Participant", "Theme_Key"]
)

print("\n" + "=" * 70)
print("STAGE 2 — CLEANING")
print("=" * 70)

print(
    "Unique participant × raw-theme observations:",
    len(clean_bri)
)


# ================================================================
# 6. BRI NORMALIZATION DICTIONARY
# ================================================================

bri_map = {

    # ---------------- SHARED RECORDS ----------------
    "common records": "Shared records",
    "shared records": "Shared records",
    "cross-party records": "Shared records",

    # ---------------- INFORMATION CONSISTENCY ----------------
    "information consistency": "Information consistency",
    "consistency": "Information consistency",
    "version consistency": "Information consistency",

    # ---------------- INFORMATION INTEGRITY ----------------
    "information integrity": "Information integrity",
    "trusted records": "Information integrity",

    # ---------------- TRACEABILITY ----------------
    "traceability": "End-to-end traceability",
    "cross-organizational traceability": "End-to-end traceability",
    "end-to-end traceability": "End-to-end traceability",

    # ---------------- TRANSACTION HISTORY ----------------
    "transaction history": "Transaction history",
    "shared transaction history": "Transaction history",
    "transaction sequence": "Transaction history",

    # ---------------- PRODUCT / PROVENANCE ----------------
    "product history": "Product/provenance history",
    "provenance": "Product/provenance history",
    "recall traceability": "Product/provenance history",

    # ---------------- HISTORICAL RECORD CONTINUITY ----------------
    "shared history": "Historical record continuity",
    "common history": "Historical record continuity",
    "continuous history": "Historical record continuity",

    # ---------------- EVENT / DISRUPTION ----------------
    "event tracing": "Event/disruption tracing",
    "disruption tracing": "Event/disruption tracing",
    "disruption identification": "Event/disruption tracing",
    "event sequence": "Event/disruption tracing",
    "shipment reconstruction": "Event/disruption tracing",

    # ---------------- INFORMATION TRAIL ----------------
    "information trail": "Information trail",
    "information history": "Information trail",

    # ---------------- INTERORGANIZATIONAL VISIBILITY ----------------
    "common visibility": "Interorganizational visibility",
    "interorganizational visibility": "Interorganizational visibility",
    "multi-party visibility": "Interorganizational visibility",
    "shared visibility": "Interorganizational visibility",
    "cross-party visibility": "Interorganizational visibility",
    "partner visibility": "Interorganizational visibility",

    # ---------------- PRODUCT / BATCH / LOT ----------------
    "batch tracking": "Product/batch/lot tracking",
    "batch visibility": "Product/batch/lot tracking",
    "lot visibility": "Product/batch/lot tracking",
    "product tracking": "Product/batch/lot tracking",

    # ---------------- SHIPMENT / MOVEMENT ----------------
    "shipment tracking": "Shipment/movement tracking",
    "movement tracking": "Shipment/movement tracking",
    "status and movement tracking": "Shipment/movement tracking",

    # ---------------- STATUS / LOCATION ----------------
    "status tracking": "Status/location tracking",
    "ownership/location tracking": "Status/location tracking",
    "status history": "Status/location tracking",

    # ---------------- TRANSACTION VISIBILITY ----------------
    "transaction visibility": "Transaction visibility",

    # ---------------- SHARED INFORMATION ----------------
    "shared information": "Shared information",

    # ---------------- VERIFICATION ----------------
    "verification": "Verification",
    "shared verification": "Verification",
    "shipment verification": "Verification",

    # ---------------- DISTINCT THEMES ----------------
    "immutability": "Immutability",
    "integration": "Integration",
    "interorganizational trust": "Interorganizational trust",
    "accountability": "Accountability",
    "reduced disputes": "Reduced disputes",
    "transparency": "Transparency",
    "visibility": "Visibility"
}


# ================================================================
# 7. APPLY NORMALIZATION
# ================================================================

clean_bri["Normalized_Theme"] = (
    clean_bri["Theme_Key"].map(bri_map)
)

unmapped = clean_bri[
    clean_bri["Normalized_Theme"].isna()
].copy()

print("\n" + "=" * 70)
print("STAGE 3 — NORMALIZATION")
print("=" * 70)

print(
    "Raw/cleaned themes:",
    len(clean_bri)
)

print(
    "Normalized themes:",
    clean_bri["Normalized_Theme"].nunique()
)

print(
    "Unmapped themes:",
    len(unmapped)
)

if len(unmapped) > 0:

    print("\nWARNING — THESE THEMES NEED CODING:")
    print(
        unmapped[
            ["Raw_Theme", "Theme_Key"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 8. MERGE / KEEP DECISION
# ================================================================

decision_map = {

    "common records": "Merge",
    "shared records": "Keep",
    "cross-party records": "Merge",

    "information consistency": "Keep",
    "consistency": "Merge",
    "version consistency": "Merge",

    "information integrity": "Keep",
    "trusted records": "Merge",

    "traceability": "Merge",
    "cross-organizational traceability": "Merge",
    "end-to-end traceability": "Keep",

    "transaction history": "Keep",
    "shared transaction history": "Merge",
    "transaction sequence": "Merge",

    "product history": "Merge",
    "provenance": "Keep",
    "recall traceability": "Merge",

    "shared history": "Merge",
    "common history": "Merge",
    "continuous history": "Merge",

    "event tracing": "Keep",
    "disruption tracing": "Merge",
    "disruption identification": "Merge",
    "event sequence": "Merge",
    "shipment reconstruction": "Merge",

    "information trail": "Keep",
    "information history": "Merge",

    "common visibility": "Merge",
    "interorganizational visibility": "Keep",
    "multi-party visibility": "Merge",
    "shared visibility": "Merge",
    "cross-party visibility": "Merge",
    "partner visibility": "Merge",

    "batch tracking": "Keep",
    "batch visibility": "Merge",
    "lot visibility": "Merge",
    "product tracking": "Merge",

    "shipment tracking": "Keep",
    "movement tracking": "Merge",
    "status and movement tracking": "Merge",

    "status tracking": "Keep",
    "ownership/location tracking": "Merge",
    "status history": "Merge",

    "transaction visibility": "Keep",

    "shared information": "Keep",

    "verification": "Keep",
    "shared verification": "Merge",
    "shipment verification": "Merge",

    "immutability": "Keep",
    "integration": "Keep",
    "interorganizational trust": "Keep",
    "accountability": "Keep",
    "reduced disputes": "Keep",
    "transparency": "Keep",
    "visibility": "Keep"
}

clean_bri["Decision"] = (
    clean_bri["Theme_Key"].map(decision_map)
)

uncoded_decisions = clean_bri[
    clean_bri["Decision"].isna()
].copy()

clean_bri["Reason"] = clean_bri["Decision"].map({

    "Merge":
        "Conceptually overlaps with other raw expressions representing the same underlying theme.",

    "Keep":
        "Retained as a distinct conceptual aspect of BRI."
})


# ================================================================
# 9. PARTICIPANT × NORMALIZED THEME MATRIX
# ================================================================

coded_bri = clean_bri[
    clean_bri["Normalized_Theme"].notna()
].copy()

coded_bri = coded_bri.drop_duplicates(
    subset=[
        "Participant",
        "Normalized_Theme"
    ]
)

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix = pd.crosstab(
    coded_bri["Normalized_Theme"],
    coded_bri["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)


# ================================================================
# 10. FREQUENCY AND PERCENTAGE
# ================================================================

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"] /
    len(participants) *
    100
).round(1)

summary = (
    matrix[
        ["Frequency", "Percentage"]
    ]
    .sort_values(
        "Frequency",
        ascending=False
    )
    .reset_index()
)

summary = summary.rename(
    columns={
        "Normalized_Theme": "Theme"
    }
)


# ================================================================
# 11. NORMALIZED CODING DICTIONARY
# ================================================================

coding_dictionary = (
    clean_bri[
        [
            "Raw_Theme",
            "Theme_Key",
            "Normalized_Theme",
            "Decision",
            "Reason"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["Normalized_Theme", "Raw_Theme"]
    )
)


# ================================================================
# 12. CREATE OUTPUT EXCEL
# ================================================================

output_file = Path(
    "BRI_Complete_Qualitative_Coding.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # Original participant responses
    original_bri.to_excel(
        writer,
        sheet_name="01_Original_BRI",
        index=False
    )

    # Raw extracted themes
    raw_bri.to_excel(
        writer,
        sheet_name="02_Raw_BRI",
        index=False
    )

    # Cleaned themes
    clean_bri[
        [
            "Participant",
            "Construct",
            "Raw_Theme",
            "Theme_Key"
        ]
    ].to_excel(
        writer,
        sheet_name="03_Cleaned_BRI",
        index=False
    )

    # Normalization/coding dictionary
    coding_dictionary.to_excel(
        writer,
        sheet_name="04_Normalized_Coding",
        index=False
    )

    # Participant × theme matrix
    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix"
    )

    # Frequency and percentage
    summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    # Unmapped themes
    if len(unmapped) > 0:

        unmapped[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key"
            ]
        ].drop_duplicates().to_excel(
            writer,
            sheet_name="07_Unmapped_Check",
            index=False
        )

    else:

        pd.DataFrame({
            "Status": [
                "All BRI themes were successfully normalized."
            ]
        }).to_excel(
            writer,
            sheet_name="07_Unmapped_Check",
            index=False
        )

    # Decision check
    if len(uncoded_decisions) > 0:

        uncoded_decisions[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key",
                "Normalized_Theme"
            ]
        ].drop_duplicates().to_excel(
            writer,
            sheet_name="08_Decision_Check",
            index=False
        )

    else:

        pd.DataFrame({
            "Status": [
                "All BRI themes have a Merge/Keep decision."
            ]
        }).to_excel(
            writer,
            sheet_name="08_Decision_Check",
            index=False
        )


# ================================================================
# 13. FINAL REPORT
# ================================================================

print("\n" + "=" * 70)
print("BRI QUALITATIVE CODING COMPLETED")
print("=" * 70)

print("Participants detected:",
      original_bri["Participant"].nunique())

print("Raw BRI observations:",
      len(raw_bri))

print("Cleaned observations:",
      len(clean_bri))

print("Normalized themes:",
      clean_bri["Normalized_Theme"].nunique())

print("Merge decisions:",
      (clean_bri["Decision"] == "Merge").sum())

print("Keep decisions:",
      (clean_bri["Decision"] == "Keep").sum())

print("Unmapped themes:",
      len(unmapped))

print("Themes without decision:",
      len(uncoded_decisions))

print("\nTOP BRI THEMES")
print(
    summary.to_string(index=False)
)

print("\n" + "=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(output_file.resolve())

INPUT FILE FOUND
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Themes.xlsx

Sheets found:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26']
Processing 1 -> P01
Processing 2 -> P02
Processing 3 -> P03
Processing 4 -> P04
Processing 5 -> P05
Processing 6 -> P06
Processing 7 -> P07
Processing 8 -> P08
Processing 9 -> P09
Processing 10 -> P10
Processing 11 -> P11
Processing 12 -> P12
Processing 13 -> P13
Processing 14 -> P14
Processing 15 -> P15
Processing 16 -> P16
Processing 17 -> P17
Processing 18 -> P18
Processing 19 -> P19
Processing 20 -> P20
Processing 21 -> P21
Processing 22 -> P22
Processing 23 -> P23
Processing 24 -> P24
Processing 25 -> P25
Processing 26 -> P26

STAGE 1 — RAW BRI EXTRACTION
Participants found: 26
Raw BRI observations: 87

STAGE 2 — CLEANING
Unique participant × raw-theme

In [ ]:
# ================================================================
# COMPLETE BRI QUALITATIVE ANALYSIS
# ================================================================
#
# INPUT:
#     BRI_Complete_Qualitative_Coding.xlsx
#
# OUTPUT:
#     BRI_FINAL_QUALITATIVE_ANALYSIS_20260827_104106.xlsx
#
# This code:
#   1. Reads the completed BRI coding workbook
#   2. Preserves the original responses
#   3. Preserves raw themes
#   4. Preserves cleaned themes
#   5. Preserves normalized coding
#   6. Rebuilds the participant × theme matrix
#   7. Recalculates theme frequency and percentage
#   8. Creates a ranked theme summary
#   9. Creates a coding audit
#  10. Creates participant coverage
#  11. Creates construct-level statistics
#  12. Creates a final BRI evidence table
#  13. Saves EVERYTHING into one Excel workbook
#
# ================================================================


import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND INPUT FILE
# ================================================================

input_file = Path("BRI_Complete_Qualitative_Coding.xlsx")

# If exact filename is not found, search for an Excel file
# containing "BRI" in its name.

if not input_file.exists():

    possible_files = list(
        Path(".").glob("*BRI*.xlsx")
    )

    if len(possible_files) == 0:

        raise FileNotFoundError(
            "\nNo BRI Excel file was found.\n"
            "Please make sure the file is in the same folder "
            "as your Python notebook/script and is named:\n\n"
            "BRI_Complete_Qualitative_Coding.xlsx"
        )

    elif len(possible_files) == 1:

        input_file = possible_files[0]

    else:

        print("Multiple BRI files were found:")

        for i, f in enumerate(possible_files):
            print(f"{i}: {f.name}")

        raise ValueError(
            "\nMore than one BRI Excel file was found. "
            "Keep only the correct one in the folder."
        )


print("=" * 70)
print("BRI QUALITATIVE ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")

for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. READ EXISTING BRI SHEETS
# ================================================================

# ------------------------------------------------
# Original responses
# ------------------------------------------------

original_df = pd.read_excel(
    input_file,
    sheet_name="01_Original_BRI"
)


# ------------------------------------------------
# Raw themes
# ------------------------------------------------

raw_df = pd.read_excel(
    input_file,
    sheet_name="02_Raw_BRI"
)


# ------------------------------------------------
# Cleaned themes
# ------------------------------------------------

clean_df = pd.read_excel(
    input_file,
    sheet_name="03_Cleaned_BRI"
)


# ------------------------------------------------
# Normalized coding
# ------------------------------------------------

coding_df = pd.read_excel(
    input_file,
    sheet_name="04_Normalized_Coding"
)


# ================================================================
# 4. STANDARDIZE COLUMN NAMES
# ================================================================

# Make sure the important columns are correctly named.

original_df.columns = [
    str(c).strip()
    for c in original_df.columns
]

raw_df.columns = [
    str(c).strip()
    for c in raw_df.columns
]

clean_df.columns = [
    str(c).strip()
    for c in clean_df.columns
]

coding_df.columns = [
    str(c).strip()
    for c in coding_df.columns
]


# ================================================================
# 5. BASIC DATA CLEANING
# ================================================================

# Clean theme keys

if "Theme_Key" in clean_df.columns:

    clean_df["Theme_Key"] = (
        clean_df["Theme_Key"]
        .astype(str)
        .str.strip()
        .str.lower()
    )


coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .astype(str)
    .str.strip()
    .str.lower()
)


coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .astype(str)
    .str.strip()
)


coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype(str)
    .str.strip()
)


coding_df["Decision"] = (
    coding_df["Decision"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 6. CHECK PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .astype(str)
    .unique()
)

print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 7. CHECK FOR UNMAPPED THEMES
# ================================================================

unmapped = coding_df[
    coding_df["Normalized_Theme"].isin(
        ["", "nan", "None"]
    )
].copy()


# Also check for missing values

unmapped = coding_df[
    coding_df["Normalized_Theme"].isna()
    |
    (
        coding_df["Normalized_Theme"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
].copy()


# ================================================================
# 8. REBUILD PARTICIPANT-LEVEL NORMALIZED DATA
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 9. CHECK FOR MISSING MAPPINGS
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
    |
    (
        coded_df["Normalized_Theme"]
        .astype(str)
        .str.strip()
        .isin(["", "nan", "None"])
    )
].copy()


if len(missing_mapping) > 0:

    print("\nWARNING:")
    print(
        "Some cleaned themes do not have a normalized theme."
    )

    print(
        missing_mapping[
            [
                "Participant",
                "Raw_Theme",
                "Theme_Key"
            ]
        ].to_string(index=False)
    )

else:

    print(
        "\nAll cleaned BRI themes have a normalized theme."
    )


# ================================================================
# 10. REMOVE DUPLICATE PARTICIPANT-THEME COMBINATIONS
# ================================================================

# A participant mentioning the same normalized theme several
# times should count ONCE for participant-level prevalence.

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# ================================================================
# 11. CREATE PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# Ensure P01-P26 appear in correct order

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]


matrix = matrix.reindex(
    columns=expected_participants,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 12. FREQUENCY AND PERCENTAGE
# ================================================================

matrix["Frequency"] = matrix[
    expected_participants
].sum(axis=1)


matrix["Percentage"] = (
    matrix["Frequency"]
    / len(expected_participants)
    * 100
).round(1)


# ================================================================
# 13. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 14. FINAL THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_BRI_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 15. ADD INTERPRETIVE PREVALENCE CATEGORY
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = theme_summary[
    "Percentage_of_Experts"
].apply(
    prevalence_category
)


# ================================================================
# 16. CODING AUDIT
# ================================================================

coding_audit = coding_df.copy()


coding_audit["Normalized_Theme"] = (
    coding_audit["Normalized_Theme"]
    .astype(str)
    .str.strip()
)


# Count how many participants mentioned each normalized theme

theme_participant_counts = (
    participant_theme
    .groupby("Normalized_Theme")
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_audit.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit["Experts_Mentioning"] = (
    coding_audit["Experts_Mentioning"]
    .fillna(0)
    .astype(int)
)


coding_audit["Percentage_of_Experts"] = (
    coding_audit["Experts_Mentioning"]
    / len(expected_participants)
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 17. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 18. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .groupby("Normalized_Theme")
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),
        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),
        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = normalization_summary.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / len(expected_participants)
    * 100
).round(1)


normalization_summary = normalization_summary.sort_values(
    "Experts_Mentioning",
    ascending=False
).reset_index(drop=True)


# ================================================================
# 19. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        expected_participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 20. CONSTRUCT-LEVEL STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["BRI"],

    "Participants": [
        len(expected_participants)
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 21. FINAL BRI EVIDENCE TABLE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_BRI_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Experts_Mentioning":
            "Experts_Mentioning",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence["Prevalence_Category"] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 22. CREATE QUALITY-CHECK TABLE
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme combinations",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        len(expected_participants),

        len(original_df),

        len(raw_df),

        coding_df["Theme_Key"].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        (
            len(coded_df)
            -
            len(participant_theme)
        ),

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if len(expected_participants) == 26
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]

})


# ================================================================
# 23. CREATE OUTPUT FILE
# ================================================================

output_file = Path(
    "BRI_FINAL_QUALITATIVE_ANALYSIS_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


# ================================================================
# 24. WRITE COMPLETE EXCEL WORKBOOK
# ================================================================

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # ------------------------------------------------
    # Original evidence
    # ------------------------------------------------

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    # ------------------------------------------------
    # Raw coding
    # ------------------------------------------------

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------
    # Cleaned coding
    # ------------------------------------------------

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------
    # Complete coding dictionary
    # ------------------------------------------------

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    # ------------------------------------------------
    # Participant × theme matrix
    # ------------------------------------------------

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )


    # ------------------------------------------------
    # Theme summary
    # ------------------------------------------------

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------
    # Normalization summary
    # ------------------------------------------------

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    # ------------------------------------------------
    # Decision summary
    # ------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------
    # Participant coverage
    # ------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------
    # Construct statistics
    # ------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )


    # ------------------------------------------------
    # Final evidence table
    # ------------------------------------------------

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_BRI_Evidence",
        index=False
    )


    # ------------------------------------------------
    # Quality checks
    # ------------------------------------------------

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------
    # Unmapped themes
    # ------------------------------------------------

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 25. FINAL CONSOLE REPORT
# ================================================================

print("\n")
print("=" * 70)
print("BRI ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    len(expected_participants)
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Final normalized BRI themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df["Decision"]
        == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df["Decision"]
        == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 26. SHOW FINAL BRI THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL BRI THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_BRI_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 27. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print("\nThe complete BRI analysis workbook has been created.")

BRI QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\BRI_Complete_Qualitative_Coding.xlsx

Sheets found:
 - 01_Original_BRI
 - 02_Raw_BRI
 - 03_Cleaned_BRI
 - 04_Normalized_Coding
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Unmapped_Check
 - 08_Decision_Check

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

All cleaned BRI themes have a normalized theme.


BRI ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 87
Cleaned theme observations: 87
Unique raw themes: 55
Final normalized BRI themes: 23
Keep decisions: 25
Merge decisions: 35
Unmapped themes: 0


FINAL BRI THEMES
               Final_BRI_Theme  Experts_Mentioning  Expert_Preva

In [ ]:
# =====================================================================
# BRI — FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS
# =====================================================================
#
# INPUT:
# BRI_FINAL_QUALITATIVE_ANALYSIS_20260827_104106.xlsx
#
# OUTPUT:
# BRI_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_104251.xlsx
#
# PURPOSE:
# 1. Read completed BRI qualitative analysis
# 2. Extract final BRI evidence
# 3. Remove duplicate themes caused by capitalization/spacing
# 4. Organize themes into BRI content domains
# 5. Calculate expert prevalence
# 6. Produce theme-to-domain evidence tables
# 7. Produce quality-control checks
#
# IMPORTANT:
# The six BRI domains generated here are CONTENT DOMAINS.
# They are NOT automatically treated as six statistical dimensions
# in the later PLS-SEM measurement model.
# =====================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# =====================================================================
# 1. INPUT FILE
# =====================================================================

TARGET = "BRI_FINAL_QUALITATIVE_ANALYSIS_20260827_104106"


# Search locations
search_locations = [
    Path("/mnt/data"),
    Path("."),
    Path.cwd(),
    Path.home() / "Downloads",
    Path.home() / "Documents",
    Path.home() / "Desktop"
]


possible_files = []


for location in search_locations:

    if not location.exists():
        continue

    try:

        for ext in [".xlsx", ".xlsm", ".xls"]:

            possible_files.extend(
                location.rglob(TARGET + ext)
            )

    except Exception:
        pass


possible_files = list(
    dict.fromkeys(
        [p.resolve() for p in possible_files]
    )
)


# =====================================================================
# 2. FALLBACK SEARCH
# =====================================================================

if len(possible_files) == 0:

    print("\nExact filename not found.")
    print("Searching for another BRI qualitative workbook...\n")

    candidates = []

    for location in search_locations:

        if not location.exists():
            continue

        try:

            for p in location.rglob("*.xlsx"):

                try:

                    xls_test = pd.ExcelFile(
                        p,
                        engine="openpyxl"
                    )

                    sheet_names_lower = [
                        str(s).lower()
                        for s in xls_test.sheet_names
                    ]

                    if any(
                        (
                            "final_bri_evidence" in s
                            or
                            ("final" in s and "bri" in s and "evidence" in s)
                        )
                        for s in sheet_names_lower
                    ):

                        candidates.append(
                            p.resolve()
                        )

                except Exception:
                    pass

        except Exception:
            pass


    if len(candidates) > 0:

        possible_files = candidates


# =====================================================================
# 3. STOP IF INPUT NOT FOUND
# =====================================================================

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\n\nINPUT FILE NOT FOUND.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Please put the Excel workbook in the same folder as "
        "your notebook or in Downloads/Desktop.\n"
    )


INPUT_FILE = possible_files[0]


print("=" * 90)
print("BRI FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS")
print("=" * 90)

print("\nInput file found:")
print(INPUT_FILE)


# =====================================================================
# 4. READ WORKBOOK
# =====================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)


print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# =====================================================================
# 5. LOAD ALL SHEETS
# =====================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# =====================================================================
# 6. HELPER FUNCTIONS
# =====================================================================

def find_sheet(keyword):

    keyword = keyword.lower()

    for s in sheets.keys():

        if keyword in str(s).lower():

            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def numeric_value(x):

    try:

        return float(
            str(x)
            .replace("%", "")
            .strip()
        )

    except:

        return np.nan


def prevalence_category(x):

    x = numeric_value(x)

    if pd.isna(x):
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# =====================================================================
# 7. IDENTIFY ORIGINAL SHEETS
# =====================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_BRI_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


# =====================================================================
# 8. FIND FINAL BRI EVIDENCE SHEET
# =====================================================================

if final_sheet is None:

    for s in sheets.keys():

        sl = str(s).lower()

        if (
            "final" in sl
            and "bri" in sl
            and "evidence" in sl
        ):

            final_sheet = s
            break


if final_sheet is None:

    raise ValueError(
        "\nCould not find the BRI final evidence sheet.\n"
        "Expected something similar to:\n"
        "11_Final_BRI_Evidence"
    )


print(
    "\nFinal evidence sheet:",
    final_sheet
)


# =====================================================================
# 9. LOAD FINAL BRI EVIDENCE
# =====================================================================

final_evidence = sheets[
    final_sheet
].copy()


final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print("\nFinal evidence columns:")

for i, c in enumerate(
    final_evidence.columns,
    start=1
):

    print(
        f"{i}. {c}"
    )


# =====================================================================
# 10. FIND BRI THEME COLUMN
# =====================================================================

theme_col = None


possible_theme_columns = [

    "Final_BRI_Theme",

    "Final_BRI_Themes",

    "Final_Theme",

    "Theme",

    "Cleaned_Theme",

    "Normalized_Theme"

]


for c in possible_theme_columns:

    if c in final_evidence.columns:

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "theme" in cl
            and "bri" in cl
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "theme" in str(c).lower():

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "\nCould not identify the BRI theme column."
    )


print(
    "\nTheme column detected:",
    theme_col
)


# =====================================================================
# 11. STANDARDIZE THEME COLUMN
# =====================================================================

final_evidence[
    "Final_BRI_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# =====================================================================
# 12. IDENTIFY EXPERT / PREVALENCE COLUMNS
# =====================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "expert" in cl
            and "mention" in cl
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if "Expert_Prevalence_%" not in final_evidence.columns:

    for c in final_evidence.columns:

        cl = str(c).lower()

        if (
            "prevalence" in cl
            and (
                "%" in str(c)
                or "percent" in cl
            )
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# =====================================================================
# 13. REMOVE EMPTY THEMES
# =====================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.strip()
    != ""
].copy()


# =====================================================================
# 14. DETERMINE PARTICIPANT COUNT AUTOMATICALLY
# =====================================================================

n_participants = None


# First: participant matrix
if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()

    participant_columns = []

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.fullmatch(
            r"P\d+",
            cstr,
            flags=re.IGNORECASE
        ):

            participant_columns.append(c)


    if len(participant_columns) > 0:

        n_participants = len(
            participant_columns
        )


# Second: participant coverage
if n_participants is None and coverage_sheet is not None:

    coverage = sheets[
        coverage_sheet
    ].copy()

    for c in coverage.columns:

        cl = str(c).lower()

        if (
            "participant" in cl
            and (
                "id" in cl
                or "code" in cl
            )
        ):

            vals = (
                coverage[c]
                .dropna()
                .astype(str)
                .str.strip()
            )

            vals = vals[
                vals.str.match(
                    r"^P\d+$",
                    case=False
                )
            ]

            if len(vals) > 0:

                n_participants = (
                    vals.nunique()
                )

                break


# Third: use maximum observed expert count
if n_participants is None:

    if "Experts_Mentioning" in final_evidence.columns:

        observed = pd.to_numeric(
            final_evidence[
                "Experts_Mentioning"
            ],
            errors="coerce"
        )

        observed_max = observed.max()

        if not pd.isna(observed_max):

            # This is only a fallback.
            # It should NOT normally be used because
            # participant matrix should determine N.
            n_participants = int(observed_max)


# Final safeguard
if n_participants is None:

    raise ValueError(
        "\nCould not determine the number of participants automatically.\n"
        "Please check the Participant Matrix or Participant Coverage sheet."
    )


print(
    "\nParticipants detected:",
    n_participants
)


# =====================================================================
# 15. PREVALENCE CATEGORY
# =====================================================================

if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# =====================================================================
# 16. REMOVE DUPLICATES
# =====================================================================

final_evidence[
    "_normalized_theme"
] = (
    final_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


if "Experts_Mentioning" in final_evidence.columns:

    final_evidence[
        "_expert_sort"
    ] = pd.to_numeric(
        final_evidence[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)

else:

    final_evidence[
        "_expert_sort"
    ] = 0


final_evidence = (
    final_evidence
    .sort_values(
        "_expert_sort",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "_normalized_theme"
        ],
        keep="first"
    )
    .copy()
)


final_evidence = (
    final_evidence
    .drop(
        columns=[
            "_normalized_theme",
            "_expert_sort"
        ],
        errors="ignore"
    )
    .reset_index(drop=True)
)


# =====================================================================
# 17. SELECT FINAL EVIDENCE COLUMNS
# =====================================================================

preferred_columns = [

    "Final_BRI_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]


evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_bri_evidence = final_evidence[
    evidence_columns
].copy()


# =====================================================================
# 18. ADD RANK SAFELY
# =====================================================================

if "Rank" in final_bri_evidence.columns:

    final_bri_evidence = (
        final_bri_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_bri_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_bri_evidence) + 1
    )
)


# =====================================================================
# 19. BRI CONTENT-DOMAIN MAPPING
# =====================================================================
#
# IMPORTANT:
# These are CONTENT DOMAINS derived from qualitative evidence.
#
# They are NOT automatically six measurement dimensions.
#
# The BRI qualitative analysis identified:
#
# 1. Traceability & Provenance
# 2. Interorganizational Information Sharing
# 3. Data Integrity & Verification
# 4. Transaction Transparency & History
# 5. Governance, Trust & Accountability
# 6. Blockchain-System Integration
#
# =====================================================================

def map_bri_dimension(theme):

    t = str(theme).strip().lower()


    # ------------------------------------------------------------
    # 1. TRACEABILITY & PROVENANCE
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "traceability",
            "trace",
            "provenance",
            "product history",
            "historical record continuity",
            "batch",
            "lot tracking",
            "product tracking",
            "shipment tracking",
            "movement tracking",
            "status tracking",
            "location tracking",
            "event tracing",
            "disruption tracing"

        ]
    ):

        return "Traceability & Provenance"


    # ------------------------------------------------------------
    # 2. INTERORGANIZATIONAL INFORMATION SHARING
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "shared records",
            "shared record",
            "shared information",
            "information sharing",
            "interorganizational visibility",
            "inter-organizational visibility",
            "interorganizational information",
            "information sharing across",
            "transaction visibility",
            "visibility across",
            "partner visibility"

        ]
    ):

        return "Interorganizational Information Sharing"


    # ------------------------------------------------------------
    # 3. DATA INTEGRITY & VERIFICATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "verification",
            "verify",
            "verified",
            "information consistency",
            "information integrity",
            "data integrity",
            "integrity",
            "immutability",
            "immutable",
            "data verification"

        ]
    ):

        return "Data Integrity & Verification"


    # ------------------------------------------------------------
    # 4. TRANSACTION TRANSPARENCY & HISTORY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "transaction history",
            "transaction transparency",
            "transaction transparency",
            "information trail",
            "transaction trail",
            "transparency",
            "transparent transactions",
            "transaction record"

        ]
    ):

        return "Transaction Transparency & History"


    # ------------------------------------------------------------
    # 5. GOVERNANCE, TRUST & ACCOUNTABILITY
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "accountability",
            "accountable",
            "interorganizational trust",
            "inter-organizational trust",
            "trust",
            "reduced disputes",
            "dispute reduction",
            "governance",
            "governance mechanism"

        ]
    ):

        return "Governance, Trust & Accountability"


    # ------------------------------------------------------------
    # 6. BLOCKCHAIN-SYSTEM INTEGRATION
    # ------------------------------------------------------------

    if any(
        x in t
        for x in [

            "integration",
            "system integration",
            "blockchain integration",
            "blockchain-system integration",
            "blockchain system integration",
            "system interoperability",
            "interoperability"

        ]
    ):

        return "Blockchain-System Integration"


    # ------------------------------------------------------------
    # REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_bri_evidence[
    "BRI_Dimension"
] = (
    final_bri_evidence[
        "Final_BRI_Theme"
    ]
    .apply(
        map_bri_dimension
    )
)


# =====================================================================
# 20. EXACT THEME OVERRIDES
# =====================================================================
#
# These override the keyword mapper for common BRI themes.
# =====================================================================

manual_mapping = {

    "end-to-end traceability":
        "Traceability & Provenance",

    "event/disruption tracing":
        "Traceability & Provenance",

    "product/batch/lot tracking":
        "Traceability & Provenance",

    "shipment/movement tracking":
        "Traceability & Provenance",

    "product/provenance history":
        "Traceability & Provenance",

    "historical record continuity":
        "Traceability & Provenance",

    "status/location tracking":
        "Traceability & Provenance",

    "shared records":
        "Interorganizational Information Sharing",

    "interorganizational visibility":
        "Interorganizational Information Sharing",

    "visibility":
        "Interorganizational Information Sharing",

    "transaction visibility":
        "Interorganizational Information Sharing",

    "shared information":
        "Interorganizational Information Sharing",

    "verification":
        "Data Integrity & Verification",

    "information consistency":
        "Data Integrity & Verification",

    "information integrity":
        "Data Integrity & Verification",

    "immutability":
        "Data Integrity & Verification",

    "transaction history":
        "Transaction Transparency & History",

    "information trail":
        "Transaction Transparency & History",

    "transparency":
        "Transaction Transparency & History",

    "accountability":
        "Governance, Trust & Accountability",

    "interorganizational trust":
        "Governance, Trust & Accountability",

    "reduced disputes":
        "Governance, Trust & Accountability",

    "integration":
        "Blockchain-System Integration"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_bri_evidence[
            "Final_BRI_Theme"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq(
            theme.lower()
        )
    )

    final_bri_evidence.loc[
        mask,
        "BRI_Dimension"
    ] = dimension


# =====================================================================
# 21. DIMENSION SUMMARY
# =====================================================================

dimension_rows = []


for dimension, group in (
    final_bri_evidence
    .groupby(
        "BRI_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_BRI_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # IMPORTANT:
    # Do NOT sum Experts_Mentioning across themes.
    # One expert can mention multiple themes.
    #
    # Therefore, dimension-level prevalence is calculated
    # from the participant matrix where possible.


    experts = 0


    if matrix_sheet is not None:

        pm = participant_matrix.copy()

        participant_cols = []

        for c in pm.columns:

            cstr = str(c).strip()

            if re.fullmatch(
                r"P\d+",
                cstr,
                flags=re.IGNORECASE
            ):

                participant_cols.append(c)


        # Find theme column in participant matrix

        pm_theme_col = None

        for c in pm.columns:

            cl = str(c).lower()

            if "theme" in cl:

                pm_theme_col = c
                break


        if (
            pm_theme_col is not None
            and len(participant_cols) > 0
        ):

            relevant_rows = pm[
                pm[
                    pm_theme_col
                ]
                .astype(str)
                .str.strip()
                .isin(themes)
            ]


            if len(relevant_rows) > 0:

                # Any mention of any theme in this domain
                # counts the participant once.

                participant_values = (
                    relevant_rows[
                        participant_cols
                    ]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .fillna(0)
                )


                participant_domain_totals = (
                    participant_values
                    .sum(axis=0)
                )


                experts = int(
                    (
                        participant_domain_totals
                        > 0
                    )
                    .sum()
                )


    # Fallback if participant matrix is unavailable
    if experts == 0:

        if "Experts_Mentioning" in group.columns:

            expert_values = pd.to_numeric(
                group[
                    "Experts_Mentioning"
                ],
                errors="coerce"
            ).dropna()

            if len(expert_values) > 0:

                experts = int(
                    expert_values.max()
                )


    prevalence = (

        experts
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0

    )


    dimension_rows.append({

        "BRI_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_BRI_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# =====================================================================
# 22. SORT + RANK
# =====================================================================

if len(dimension_summary) > 0:

    dimension_summary[
        "_sort"
    ] = pd.to_numeric(
        dimension_summary[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_summary = (
        dimension_summary
        .sort_values(
            "_sort",
            ascending=False
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# =====================================================================
# 23. DIMENSION × THEME TABLE
# =====================================================================

dimension_theme_columns = [

    "BRI_Dimension",

    "Final_BRI_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_bri_evidence.columns
]


dimension_themes = final_bri_evidence[
    dimension_theme_columns
].copy()


if "Experts_Mentioning" in dimension_themes.columns:

    dimension_themes[
        "_sort"
    ] = pd.to_numeric(
        dimension_themes[
            "Experts_Mentioning"
        ],
        errors="coerce"
    ).fillna(0)


    dimension_themes = (
        dimension_themes
        .sort_values(
            [
                "BRI_Dimension",
                "_sort"
            ],
            ascending=[
                True,
                False
            ]
        )
        .drop(
            columns=["_sort"]
        )
        .reset_index(drop=True)
    )


# =====================================================================
# 24. REVIEW-REQUIRED THEMES
# =====================================================================

review_rows = final_bri_evidence[
    final_bri_evidence[
        "BRI_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(review_rows) > 0:

    review_columns = [

        "Final_BRI_Theme",

        "Experts_Mentioning",

        "Expert_Prevalence_%",

        "BRI_Dimension"

    ]


    review_columns = [
        c
        for c in review_columns
        if c in review_rows.columns
    ]


    review_required = review_rows[
        review_columns
    ].copy()


    review_required.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    review_required = pd.DataFrame({

        "Status": [
            "PASS — All BRI themes mapped to a content domain."
        ]

    })


# =====================================================================
# 25. QUALITY CHECKS
# =====================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final BRI evidence available",

    "Result":
        "PASS"
        if len(final_bri_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_bri_evidence)} normalized BRI themes"

})


quality_rows.append({

    "Quality_Check":
        "Participant count",

    "Result":
        n_participants,

    "Details":
        "Detected automatically from participant matrix / coverage"

})


quality_rows.append({

    "Quality_Check":
        "BRI content domains generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative content domains"

})


review_count = len(review_rows)


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        review_count,

    "Details":
        "Themes assigned Review Required"

})


duplicate_count = int(
    final_bri_evidence[
        "Final_BRI_Theme"
    ]
    .astype(str)
    .str.lower()
    .duplicated()
    .sum()
)


quality_rows.append({

    "Quality_Check":
        "Remaining duplicate themes",

    "Result":
        duplicate_count,

    "Details":
        "Should normally be zero"

})


# Check all six expected BRI domains
expected_domains = {

    "Traceability & Provenance",

    "Interorganizational Information Sharing",

    "Data Integrity & Verification",

    "Transaction Transparency & History",

    "Governance, Trust & Accountability",

    "Blockchain-System Integration"

}


actual_domains = set(
    dimension_summary[
        "BRI_Dimension"
    ]
    .dropna()
    .astype(str)
)


missing_domains = (
    expected_domains
    - actual_domains
)


quality_rows.append({

    "Quality_Check":
        "Expected BRI content domains present",

    "Result":
        "PASS"
        if len(missing_domains) == 0
        else "CHECK",

    "Details":
        (
            "All six expected domains present"
            if len(missing_domains) == 0
            else
            "Missing: "
            + "; ".join(
                sorted(missing_domains)
            )
        )

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# =====================================================================
# 26. COPY ORIGINAL SUPPORTING SHEETS
# =====================================================================

def get_sheet_or_empty(sheet_name):

    if sheet_name is not None:

        return sheets[
            sheet_name
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

coding_audit = get_sheet_or_empty(
    audit_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# =====================================================================
# 27. OUTPUT FILE
# =====================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"BRI_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


# =====================================================================
# 28. WRITE OUTPUT
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    if matrix_sheet is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_bri_evidence.to_excel(
        writer,
        sheet_name="11_Final_BRI_Evidence",
        index=False
    )

    dimension_summary.to_excel(
        writer,
        sheet_name="12_BRI_Dimension_Summary",
        index=False
    )

    dimension_themes.to_excel(
        writer,
        sheet_name="13_BRI_Dimension_Themes",
        index=False
    )

    review_required.to_excel(
        writer,
        sheet_name="14_Review_Required",
        index=False
    )

    quality_checks_new.to_excel(
        writer,
        sheet_name="15_Quality_Checks",
        index=False
    )


# =====================================================================
# 29. FINAL REPORT
# =====================================================================

print("\n")
print("=" * 90)
print("BRI PROCESS COMPLETED SUCCESSFULLY")
print("=" * 90)

print("\nOutput file:")
print(
    OUTPUT_FILE.resolve()
)

print("\nNormalized BRI themes:")
print(
    len(final_bri_evidence)
)

print("\nBRI content domains:")
print(
    len(dimension_summary)
)


print("\n")
print("-" * 90)
print("BRI CONTENT-DOMAIN SUMMARY")
print("-" * 90)

if len(dimension_summary) > 0:

    print(
        dimension_summary.to_string(
            index=False
        )
    )

else:

    print(
        "No content domains generated."
    )


print("\n")
print("-" * 90)
print("QUALITY CHECKS")
print("-" * 90)

print(
    quality_checks_new.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("OUTPUT SHEETS CREATED")
print("=" * 90)

print("""
11_Final_BRI_Evidence
    → Cleaned final BRI qualitative themes

12_BRI_Dimension_Summary
    → Six BRI content domains

13_BRI_Dimension_Themes
    → Theme-to-domain mapping

14_Review_Required
    → Themes requiring manual inspection

15_Quality_Checks
    → Automated quality-control results
""")


print("\n")
print("=" * 90)
print("IMPORTANT METHODOLOGICAL NOTE")
print("=" * 90)

print("""
The BRI domains generated here are CONTENT DOMAINS derived
from expert qualitative evidence.

They are NOT automatically treated as six reflective
measurement dimensions.

Their purpose is to support systematic questionnaire-item
development by ensuring that the qualitative evidence
representing BRI is adequately covered.

The next stage is candidate BRI questionnaire-item
development, followed by expert content validation and
then quantitative measurement validation using PLS-SEM.
""")

BRI FINAL QUALITATIVE EVIDENCE + CONTENT-DOMAIN ANALYSIS

Input file found:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\BRI_FINAL_QUALITATIVE_ANALYSIS_20260827_104106.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_BRI_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Final evidence sheet: 11_Final_BRI_Evidence

Final evidence columns:
1. Final_BRI_Theme
2. Number_of_Raw_Themes
3. Experts_Mentioning
4. Expert_Prevalence_%
5. Raw_Themes_Kept
6. Raw_Themes_Merged
7. Prevalence_Category

Theme column detected: Final_BRI_Theme

Participants detected: 26


BRI PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode prog

## Decision Intelligence Capability (DIC)

In [4]:
# ================================================================
# DIC CODING
# INPUT FILE: Themes.xlsx
# ================================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. INPUT
# ------------------------------------------------

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ------------------------------------------------
# 2. FIND PARTICIPANT SHEETS
# ------------------------------------------------

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ------------------------------------------------
# 3. EXTRACT DIC RESPONSES
# ------------------------------------------------

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "DIC":

                construct_position = position
                break

        if construct_position is None:
            continue

        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "DIC",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ------------------------------------------------
# 4. CHECK
# ------------------------------------------------

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No DIC responses were found.

        Check that DIC appears in the participant sheets.
        """
    )

print("\nDIC original responses:")
print(len(original_df))


# ------------------------------------------------
# 5. SPLIT INTO RAW THEMES
# ------------------------------------------------

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "DIC",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ------------------------------------------------
# 6. CLEAN THEMES
# ------------------------------------------------

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nDIC cleaned theme observations:")
print(len(clean_df))


# ------------------------------------------------
# 7. DISPLAY RAW DIC THEMES
# ------------------------------------------------

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("DIC RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ------------------------------------------------
# 8. DIC NORMALIZATION DICTIONARY
# ------------------------------------------------
#
# Add the researcher-approved DIC mappings here.
#
# Example:
#
# "data interpretation":
#     "Data interpretation",
#
# "information interpretation":
#     "Data interpretation",
#
# ------------------------------------------------

DIC_NORMALIZATION = {

    # ADD DIC MAPPINGS HERE

}


# ------------------------------------------------
# 9. APPLY NORMALIZATION
# ------------------------------------------------

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in DIC_NORMALIZATION:

        normalized = DIC_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ------------------------------------------------
# 10. MAP TO PARTICIPANTS
# ------------------------------------------------

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ------------------------------------------------
# 11. PARTICIPANT × DIC THEME MATRIX
# ------------------------------------------------

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source["Normalized_Theme"],
    matrix_source["Participant"]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ------------------------------------------------
# 12. THEME SUMMARY
# ------------------------------------------------

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ------------------------------------------------
# 13. DECISION SUMMARY
# ------------------------------------------------

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Count"
    )
)


# ------------------------------------------------
# 14. PARTICIPANT COVERAGE
# ------------------------------------------------

participant_coverage = (
    coded_df
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "DIC_Theme_Count"
]


# ------------------------------------------------
# 15. SAVE EXCEL
# ------------------------------------------------

output_file = Path(
    "DIC_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 60)
print("DIC CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique DIC raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

DIC original responses:
26

DIC cleaned theme observations:
82


DIC RAW THEMES
                             Raw_Theme                            Theme_Clean                              Theme_Key
                alternative comparison                 alternative comparison                 alternative comparison
                 alternative selection                  alternative selection                  alternative selection
                          alternatives                           alternatives                           alternatives
                          Alternatives                           Alternatives                           alternatives
                 business consequences                  b

In [5]:
# ================================================================
# COMPLETE DIC QUALITATIVE CODING ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. FIND DIC INPUT FILE
# ================================================================

files = list(Path(".").glob("DIC_Coding_20260827_104459*.xlsx"))

if len(files) == 0:
    raise FileNotFoundError(
        "\nNo file beginning with 'DIC_Coding_' was found.\n"
        "Make sure your DIC Excel file is in the same folder as "
        "your Python notebook."
    )

if len(files) > 1:
    print("DIC files found:")
    for f in files:
        print(" -", f.name)

    # Use most recently modified file
    input_file = max(files, key=lambda x: x.stat().st_mtime)
    print("\nUsing the most recently modified DIC file:")
else:
    input_file = files[0]

print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")
for s in excel.sheet_names:
    print(" -", s)


# ================================================================
# 3. FLEXIBLE SHEET FINDER
# ================================================================

def find_sheet(possible_names):

    # Exact
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible
    for sheet in excel.sheet_names:

        a = (
            str(sheet).lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            b = (
                str(name).lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if b in a:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_DIC",
    "01_Original",
    "Original_DIC",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_DIC",
    "02_Raw",
    "Raw_DIC",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_DIC",
    "03_Cleaned",
    "Cleaned_DIC",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])

print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ SHEETS
# ================================================================

original_df = (
    pd.read_excel(input_file, sheet_name=original_sheet)
    if original_sheet else pd.DataFrame()
)

raw_df = (
    pd.read_excel(input_file, sheet_name=raw_sheet)
    if raw_sheet else pd.DataFrame()
)

clean_df = (
    pd.read_excel(input_file, sheet_name=clean_sheet)
    if clean_sheet else pd.DataFrame()
)

if coding_sheet is None:
    raise ValueError(
        "\nCould not find the DIC Coding/Normalized Coding sheet."
    )

coding_df = pd.read_excel(
    input_file,
    sheet_name=coding_sheet
)


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:
        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. COLUMN FINDER
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    for column in df.columns:

        a = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            b = (
                str(candidate)
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if a == b:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    ["Theme_Key", "Theme Key", "ThemeKey"]
)

raw_theme_col = find_column(
    coding_df,
    ["Raw_Theme", "Raw Theme", "RawTheme"]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    ["Decision"]
)


if theme_key_col is None:
    raise ValueError("Theme_Key column not found.")

if raw_theme_col is None:
    raise ValueError("Raw_Theme column not found.")

if normalized_col is None:
    raise ValueError(
        "Normalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "Decision column not found."
    )


coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)

if participant_col is None:
    raise ValueError(
        "\nParticipant column not found in Cleaned DIC sheet.\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )

if participant_col != "Participant":
    clean_df = clean_df.rename(
        columns={
            participant_col: "Participant"
        }
    )


# ================================================================
# 8. THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

if clean_theme_key is None:
    raise ValueError(
        "\nTheme_Key not found in Cleaned DIC sheet."
    )

if clean_theme_key != "Theme_Key":
    clean_df = clean_df.rename(
        columns={
            clean_theme_key: "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN PARTICIPANT DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df["Participant"]
    .dropna()
    .unique()
)

print("\nParticipants:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CLEANED THEMES WITH CODING
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df["Normalized_Theme"].isna()
].copy()

print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)

if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme["Normalized_Theme"],
    participant_theme["Participant"]
)


# P01–P26 first

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p for p in expected_participants
    if p in matrix.columns
]

other = [
    p for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)

matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY
# ================================================================

participant_columns = existing + other

matrix["Frequency"] = (
    matrix[participant_columns]
    .sum(axis=1)
)


# ================================================================
# 17. PERCENTAGE
# ================================================================

total_participants = len(participants)

matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 18. RANK
# ================================================================

matrix = matrix.sort_values(
    [
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)

matrix.insert(
    0,
    "Rank",
    range(1, len(matrix) + 1)
)


# ================================================================
# 19. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()

theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 20. PREVALENCE
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 21. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 22. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby("Decision")
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)

decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 23. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=["Normalized_Theme"]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 24. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby("Participant")
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 25. DIC CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["DIC"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df["Theme_Key"].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df["Decision"]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 26. FINAL DIC EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_DIC_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 27. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df["Decision"]
            == "Keep"
        ).sum(),

        (
            coding_df["Decision"]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"

    ]
})


# ================================================================
# 28. SAVE COMPLETE DIC WORKBOOK
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# Avoid PermissionError if file already exists/open
if output_file.exists():

    output_file = Path(
        f"DIC_FINAL_QUALITATIVE_ANALYSIS_{timestamp}_NEW.xlsx"
    )


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_DIC_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 29. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("DIC ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized DIC themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 30. DISPLAY FINAL DIC THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL DIC THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 31. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete DIC qualitative analysis workbook "
    "created successfully."
)

C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\DIC_Coding_20260827_104459.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


DIC ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observations: 82
Cleaned theme observations: 82
Unique raw themes: 58


In [ ]:
# ================================================================
# DIC — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
# DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_104701.xlsx
#
# MAIN OUTPUT:
# DIC_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_104758.xlsx
#
# PURPOSE:
# 1. Preserve the completed DIC qualitative coding
# 2. Extract final DIC evidence
# 3. Organize themes into conceptually meaningful DIC dimensions
# 4. Calculate expert prevalence
# 5. Identify themes requiring manual review
# 6. Produce dimension-level evidence for questionnaire development
#
# IMPORTANT:
# The dimensions generated here are QUALITATIVE DIMENSIONS.
# They are NOT yet statistically validated measurement dimensions.
# ================================================================


import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. INPUT FILE
# ================================================================

TARGET = "DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_104701"

possible_files = []

for ext in [".xlsx", ".xlsm", ".xls"]:
    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:
    raise FileNotFoundError(
        "\nDIC input file was not found.\n\n"
        f"Expected:\n{TARGET}.xlsx\n\n"
        "Put the file in the same folder as the notebook."
    )

INPUT_FILE = possible_files[0]

print("=" * 80)
print("DIC FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. LOAD ALL SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:
        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )
    except Exception as e:
        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():
            return s

    return None


def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


# ================================================================
# 5. IDENTIFY EXISTING SHEETS
# ================================================================

original_sheet = find_sheet(
    "01_Original_Responses"
)

raw_sheet = find_sheet(
    "02_Raw_Themes"
)

cleaned_sheet = find_sheet(
    "03_Cleaned_Themes"
)

audit_sheet = find_sheet(
    "04_Coding_Audit"
)

matrix_sheet = find_sheet(
    "05_Participant_Matrix"
)

theme_sheet = find_sheet(
    "06_Theme_Summary"
)

normalization_sheet = find_sheet(
    "07_Normalization_Summary"
)

decision_sheet = find_sheet(
    "08_Decision_Summary"
)

coverage_sheet = find_sheet(
    "09_Participant_Coverage"
)

statistics_sheet = find_sheet(
    "10_Construct_Statistics"
)

final_sheet = find_sheet(
    "11_Final_DIC_Evidence"
)

quality_sheet = find_sheet(
    "12_Quality_Checks"
)

unmapped_sheet = find_sheet(
    "13_Unmapped_Themes"
)


print("\nDetected final evidence sheet:")
print(final_sheet)


# ================================================================
# 6. READ FINAL DIC EVIDENCE
# ================================================================

if final_sheet is None:

    raise ValueError(
        "11_Final_DIC_Evidence was not found."
    )

final_evidence = sheets[
    final_sheet
].copy()

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


print("\nFinal DIC evidence columns:")
print(
    list(
        final_evidence.columns
    )
)


# ================================================================
# 7. IDENTIFY DIC THEME COLUMN
# ================================================================

theme_col = None

for c in final_evidence.columns:

    if str(c).strip() == "Final_DIC_Theme":

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if (
            "DIC" in str(c)
            and "Theme" in str(c)
        ):

            theme_col = c
            break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify Final DIC Theme column."
    )


final_evidence[
    "Final_DIC_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE PREVALENCE COLUMNS
# ================================================================

if (
    "Experts_Mentioning"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Experts" in str(c)
            and "Mention" in str(c)
        ):

            final_evidence[
                "Experts_Mentioning"
            ] = final_evidence[c]

            break


if (
    "Expert_Prevalence_%"
    not in final_evidence.columns
):

    for c in final_evidence.columns:

        if (
            "Prevalence" in str(c)
            and "%" in str(c)
        ):

            final_evidence[
                "Expert_Prevalence_%"
            ] = final_evidence[c]

            break


# ================================================================
# 9. REMOVE EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_DIC_Theme"
    ] != ""
].copy()


# ================================================================
# 10. DETERMINE NUMBER OF PARTICIPANTS
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


participants = []

if participant_matrix is not None:

    for c in participant_matrix.columns:

        cstr = str(c).strip()

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(
                cstr
            )


if len(participants) > 0:

    n_participants = len(
        participants
    )

else:

    # Your DIC study uses 26 experts.
    # This is used only if the participant matrix
    # does not expose P01...P26 columns.

    n_participants = 26


print(
    "\nNumber of participants:",
    n_participants
)


# ================================================================
# 11. PREVALENCE CATEGORY
# ================================================================

if (
    "Expert_Prevalence_%"
    in final_evidence.columns
):

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(
            prevalence_category
        )
    )


# ================================================================
# 12. SORT FINAL DIC EVIDENCE
# ================================================================

sort_columns = []

if "Experts_Mentioning" in final_evidence.columns:
    sort_columns.append(
        "Experts_Mentioning"
    )

sort_columns.append(
    "Final_DIC_Theme"
)


final_evidence = (
    final_evidence
    .sort_values(
        sort_columns,
        ascending=[
            False
            if c == "Experts_Mentioning"
            else True
            for c in sort_columns
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# 13. FINAL DIC EVIDENCE TABLE
# ================================================================

preferred_columns = [

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Raw_Themes_Kept",

    "Raw_Themes_Merged",

    "Prevalence_Category"

]

evidence_columns = [
    c
    for c in preferred_columns
    if c in final_evidence.columns
]


final_dic_evidence = final_evidence[
    evidence_columns
].copy()


# IMPORTANT:
# Avoid "Rank already exists" errors.
# Remove any previous Rank column first.

if "Rank" in final_dic_evidence.columns:

    final_dic_evidence = (
        final_dic_evidence
        .drop(
            columns=["Rank"]
        )
    )


final_dic_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_dic_evidence) + 1
    )
)


# ================================================================
# 14. DIC DIMENSION MAPPING
# ================================================================
#
# The dimensions below are derived from the actual themes found
# in your DIC qualitative evidence.
#
# Dimension 1:
# Information Integration & Interpretation
#
# Dimension 2:
# Evidence-Based Decision Framing
#
# Dimension 3:
# Alternative & Option Evaluation
#
# Dimension 4:
# Consequence & Scenario Analysis
#
# Dimension 5:
# Multi-Criteria & Multi-Perspective Assessment
#
# Dimension 6:
# Decision-Relevant Information & Knowledge Integration
#
# Themes that cannot be assigned confidently are marked
# "Review Required".
#
# ================================================================


def map_dic_dimension(theme):

    t = str(theme).lower().strip()


    # ------------------------------------------------------------
    # 1. INFORMATION INTEGRATION & INTERPRETATION
    # ------------------------------------------------------------

    information_words = [

        "multi-source information",
        "multi source information",
        "information integration",
        "evidence integration",
        "interpretation",
        "conflicting information",
        "relevant information",
        "filtering",
        "multiple indicators",
        "cross-functional information",
        "fact–assumption distinction",
        "fact-assumption distinction"

    ]

    if any(
        word in t
        for word in information_words
    ):

        return (
            "Information Integration & Interpretation"
        )


    # ------------------------------------------------------------
    # 2. EVIDENCE-BASED DECISION FRAMING
    # ------------------------------------------------------------

    decision_framing_words = [

        "decision framing",
        "decision focus",
        "decision-oriented information",
        "decision oriented information",
        "informed decisions",
        "evidence + experience",
        "data + experience",
        "data + expertise",
        "data + knowledge",
        "data + operational knowledge",
        "operational knowledge",
        "managerial knowledge",
        "experience",
        "capacity",
        "suitability"

    ]

    if any(
        word in t
        for word in decision_framing_words
    ):

        return (
            "Evidence-Based Decision Framing"
        )


    # ------------------------------------------------------------
    # 3. ALTERNATIVE & OPTION EVALUATION
    # ------------------------------------------------------------

    alternative_words = [

        "alternatives",
        "alternative comparison",
        "alternative selection",
        "option comparison",
        "option selection",
        "options",
        "systematic comparison",
        "suitability"

    ]

    if any(
        word in t
        for word in alternative_words
    ):

        return (
            "Alternative & Option Evaluation"
        )


    # ------------------------------------------------------------
    # 4. CONSEQUENCE & SCENARIO ANALYSIS
    # ------------------------------------------------------------

    consequence_words = [

        "consequence",
        "consequences",
        "consequence analysis",
        "consequence assessment",
        "business consequences",
        "downstream consequences",
        "knock-on effects",
        "immediate/long-term consequences",
        "immediate and long-term effects",
        "temporal consequences",
        "scenario",
        "scenarios",
        "scenario comparison",
        "scenario consideration",
        "scenario testing",
        "forward-looking assessment",
        "future continuity"

    ]

    if any(
        word in t
        for word in consequence_words
    ):

        return (
            "Consequence & Scenario Analysis"
        )


    # ------------------------------------------------------------
    # 5. MULTI-CRITERIA & MULTI-PERSPECTIVE ASSESSMENT
    # ------------------------------------------------------------

    multi_criteria_words = [

        "multi-criteria evaluation",
        "multi criteria evaluation",
        "multi-criteria decisions",
        "multi criteria decisions",
        "multi-criteria assessment",
        "multi criteria assessment",
        "multi-perspective assessment",
        "multi perspective assessment",
        "multi-outcome assessment",
        "multi-dimensional consequences",
        "trade-offs",
        "financial/customer assessment",
        "financial/sustainability effects",
        "operational/financial/customer effects",
        "overall outcomes",
        "systemic thinking",
        "interdependencies",
        "risk tolerance"

    ]

    if any(
        word in t
        for word in multi_criteria_words
    ):

        return (
            "Multi-Criteria & Multi-Perspective Assessment"
        )


    # ------------------------------------------------------------
    # 6. DECISION-RELEVANT INFORMATION & KNOWLEDGE INTEGRATION
    # ------------------------------------------------------------

    knowledge_words = [

        "data + knowledge",
        "data + operational knowledge",
        "data + experience",
        "data + expertise",
        "operational knowledge",
        "managerial knowledge",
        "cross-functional information",
        "evidence + experience",
        "relevant information",
        "information integration"

    ]

    if any(
        word in t
        for word in knowledge_words
    ):

        return (
            "Decision-Relevant Information & Knowledge Integration"
        )


    # ------------------------------------------------------------
    # 7. REVIEW REQUIRED
    # ------------------------------------------------------------

    return "Review Required"


final_dic_evidence[
    "DIC_Dimension"
] = (
    final_dic_evidence[
        "Final_DIC_Theme"
    ]
    .apply(
        map_dic_dimension
    )
)


# ================================================================
# 15. MANUALLY RESOLVE SPECIFIC AMBIGUITIES
# ================================================================
#
# These explicit mappings prevent generic words such as
# "Alternatives", "capacity", or "suitability" from being
# incorrectly grouped.
#
# ================================================================

manual_mapping = {

    "alternatives":
        "Alternative & Option Evaluation",

    "Alternatives":
        "Alternative & Option Evaluation",

    "alternative comparison":
        "Alternative & Option Evaluation",

    "alternative selection":
        "Alternative & Option Evaluation",

    "option comparison":
        "Alternative & Option Evaluation",

    "Option comparison":
        "Alternative & Option Evaluation",

    "systematic comparison":
        "Alternative & Option Evaluation",

    "consequences":
        "Consequence & Scenario Analysis",

    "Consequence analysis":
        "Consequence & Scenario Analysis",

    "consequence analysis":
        "Consequence & Scenario Analysis",

    "Consequence assessment":
        "Consequence & Scenario Analysis",

    "business consequences":
        "Consequence & Scenario Analysis",

    "downstream consequences":
        "Consequence & Scenario Analysis",

    "knock-on effects":
        "Consequence & Scenario Analysis",

    "immediate/long-term consequences":
        "Consequence & Scenario Analysis",

    "immediate and long-term effects":
        "Consequence & Scenario Analysis",

    "temporal consequences":
        "Consequence & Scenario Analysis",

    "scenario consideration":
        "Consequence & Scenario Analysis",

    "scenario testing":
        "Consequence & Scenario Analysis",

    "scenario comparison":
        "Consequence & Scenario Analysis",

    "scenarios":
        "Consequence & Scenario Analysis",

    "Multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria evaluation":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-criteria decisions":
        "Multi-Criteria & Multi-Perspective Assessment",

    "trade-offs":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Systemic thinking":
        "Multi-Criteria & Multi-Perspective Assessment",

    "interdependencies":
        "Multi-Criteria & Multi-Perspective Assessment",

    "risk tolerance":
        "Multi-Criteria & Multi-Perspective Assessment",

    "Multi-perspective assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-outcome assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "multi-dimensional consequences":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/customer assessment":
        "Multi-Criteria & Multi-Perspective Assessment",

    "financial/sustainability effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "operational/financial/customer effects":
        "Multi-Criteria & Multi-Perspective Assessment",

    "overall outcomes":
        "Multi-Criteria & Multi-Perspective Assessment",

    "future continuity":
        "Consequence & Scenario Analysis",

    "Forward-looking assessment":
        "Consequence & Scenario Analysis",

    "Decision framing":
        "Evidence-Based Decision Framing",

    "Decision focus":
        "Evidence-Based Decision Framing",

    "Decision-oriented information":
        "Evidence-Based Decision Framing",

    "informed decisions":
        "Evidence-Based Decision Framing",

    "Evidence + experience":
        "Evidence-Based Decision Framing",

    "data + experience":
        "Evidence-Based Decision Framing",

    "data + expertise":
        "Evidence-Based Decision Framing",

    "data + knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "data + operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "operational knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "managerial knowledge":
        "Decision-Relevant Information & Knowledge Integration",

    "cross-functional information":
        "Decision-Relevant Information & Knowledge Integration",

    "experience":
        "Evidence-Based Decision Framing",

    "capacity":
        "Evidence-Based Decision Framing",

    "suitability":
        "Evidence-Based Decision Framing",

    "Multi-source information":
        "Information Integration & Interpretation",

    "multi-source information":
        "Information Integration & Interpretation",

    "Evidence integration":
        "Information Integration & Interpretation",

    "information integration":
        "Information Integration & Interpretation",

    "Interpretation":
        "Information Integration & Interpretation",

    "filtering":
        "Information Integration & Interpretation",

    "Multiple indicators":
        "Information Integration & Interpretation",

    "Conflicting information":
        "Information Integration & Interpretation",

    "relevant information":
        "Information Integration & Interpretation",

    "Fact–assumption distinction":
        "Information Integration & Interpretation"

}


for theme, dimension in manual_mapping.items():

    mask = (
        final_dic_evidence[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .str.strip()
        .eq(theme)
    )

    final_dic_evidence.loc[
        mask,
        "DIC_Dimension"
    ] = dimension


# ================================================================
# 16. CREATE DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_dic_evidence
    .groupby(
        "DIC_Dimension",
        dropna=False
    )
):

    themes = (
        group[
            "Final_DIC_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # ------------------------------------------------------------
    # Calculate experts mentioning dimension
    # ------------------------------------------------------------

    experts_mentioning = 0


    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        temp = participant_matrix.copy()

        first_col = temp.columns[0]

        temp[
            "_Theme"
        ] = (
            temp[
                first_col
            ]
            .astype(str)
            .str.strip()
        )


        # Case-insensitive matching
        theme_lower = {
            str(x).strip().lower()
            for x in themes
        }


        matched = temp[
            temp[
                "_Theme"
            ]
            .str.lower()
            .isin(
                theme_lower
            )
        ]


        if len(matched) > 0:

            available_participants = [
                p
                for p in participants
                if p in matched.columns
            ]


            if len(
                available_participants
            ) > 0:

                vals = (
                    matched[
                        available_participants
                    ]
                    .apply(
                        pd.to_numeric,
                        errors="coerce"
                    )
                    .fillna(0)
                )


                experts_mentioning = int(
                    (
                        vals.sum(axis=0) > 0
                    ).sum()
                )


    # ------------------------------------------------------------
    # Fallback if participant matrix cannot be used
    # ------------------------------------------------------------

    if experts_mentioning == 0:

        if (
            "Experts_Mentioning"
            in group.columns
        ):

            experts_mentioning = int(
                group[
                    "Experts_Mentioning"
                ]
                .max()
            )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "DIC_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_DIC_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 17. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    if "Rank" in dimension_summary.columns:

        dimension_summary = (
            dimension_summary
            .drop(
                columns=["Rank"]
            )
        )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 18. DIMENSION × THEME TABLE
# ================================================================

dimension_theme_columns = [

    "DIC_Dimension",

    "Final_DIC_Theme",

    "Number_of_Raw_Themes",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"

]


dimension_theme_columns = [
    c
    for c in dimension_theme_columns
    if c in final_dic_evidence.columns
]


dimension_themes = final_dic_evidence[
    dimension_theme_columns
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "DIC_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 19. REVIEW-REQUIRED THEMES
# ================================================================

unmapped_check = final_dic_evidence[
    final_dic_evidence[
        "DIC_Dimension"
    ]
    ==
    "Review Required"
].copy()


if len(unmapped_check) > 0:

    unmapped_check = unmapped_check[
        [
            "Final_DIC_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All DIC themes mapped to a dimension."
        ]

    })


# ================================================================
# 20. DECISION CHECK
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


if coding_audit is not None:

    decision_columns = [

        c
        for c in coding_audit.columns
        if "Decision" in str(c)

    ]


    if len(decision_columns) > 0:

        decision_col = decision_columns[0]

        missing = coding_audit[
            coding_audit[
                decision_col
            ].isna()
        ]


        if len(missing) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All DIC coding decisions are present."
                ]

            })

        else:

            decision_check = (
                missing.copy()
            )


    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit unavailable."
        ]

    })


# ================================================================
# 21. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final DIC evidence available",

    "Result":
        "PASS"
        if len(final_dic_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_dic_evidence)} final DIC themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Participant count used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "DIC dimensions generated",

    "Result":
        len(dimension_summary),

    "Details":
        "Qualitative dimensions"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring review",

    "Result":
        len(unmapped_check)
        if "Final_DIC_Theme"
        in unmapped_check.columns
        else 0,

    "Details":
        "Themes assigned to Review Required"

})


quality_rows.append({

    "Quality_Check":
        "Duplicate theme labels",

    "Result":
        int(
            final_dic_evidence[
                "Final_DIC_Theme"
            ]
            .str.lower()
            .duplicated()
            .sum()
        ),

    "Details":
        "Case-insensitive duplicate labels detected"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 22. LOAD ORIGINAL SUPPORTING SHEETS
# ================================================================

def get_sheet_or_empty(sheet):

    if sheet is not None:

        return sheets[
            sheet
        ].copy()

    return pd.DataFrame()


original_responses = get_sheet_or_empty(
    original_sheet
)

raw_themes = get_sheet_or_empty(
    raw_sheet
)

cleaned_themes = get_sheet_or_empty(
    cleaned_sheet
)

theme_summary_existing = get_sheet_or_empty(
    theme_sheet
)

normalization_summary = get_sheet_or_empty(
    normalization_sheet
)

decision_summary = get_sheet_or_empty(
    decision_sheet
)

participant_coverage = get_sheet_or_empty(
    coverage_sheet
)

construct_statistics = get_sheet_or_empty(
    statistics_sheet
)


# ================================================================
# 23. WRITE FINAL EXCEL FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"DIC_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # MAIN DIC EVIDENCE
    # ------------------------------------------------------------

    final_dic_evidence.to_excel(
        writer,
        sheet_name="13_Final_DIC_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION SUMMARY
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_DIC_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # DIC DIMENSION × THEME
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_DIC_Dimension_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # ORIGINAL CONSTRUCT STATISTICS
    # ------------------------------------------------------------

    construct_statistics.to_excel(
        writer,
        sheet_name="16_Construct_Statistics",
        index=False
    )


# ================================================================
# 24. PRINT RESULTS
# ================================================================

print("\n")
print("=" * 80)
print("DIC PROCESS COMPLETED SUCCESSFULLY")
print("=" * 80)


print("\nOutput file:")

print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("FINAL DIC EVIDENCE")
print("-" * 80)


print(
    final_dic_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("DIC DIMENSION SUMMARY")
print("-" * 80)


print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("OUTPUT SHEETS")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_DIC_Evidence
14_DIC_Dimension_Summary
15_DIC_Dimension_Themes
16_Construct_Statistics
""")


print("\n")
print("=" * 80)
print("NEXT STAGE")
print("=" * 80)

print("""
The two most important sheets for questionnaire development are:

14_DIC_Dimension_Summary
15_DIC_Dimension_Themes

These provide the bridge from expert qualitative evidence
to candidate DIC questionnaire items.

DO NOT treat these dimensions as statistically validated
subdimensions yet. They are evidence-based qualitative
groupings that will be used to formulate candidate items.
""")

DIC FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\DIC_FINAL_QUALITATIVE_ANALYSIS_20260827_104701.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_DIC_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Detected final evidence sheet:
11_Final_DIC_Evidence

Final DIC evidence columns:
['Final_DIC_Theme', 'Number_of_Raw_Themes', 'Experts_Mentioning', 'Expert_Prevalence_%', 'Raw_Themes_Kept', 'Raw_Themes_Merged', 'Prevalence_Category']

Number of participants: 26


DIC PROCESS COMPLETED SUCCESSFULLY

Output file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_

## Smart Contract Contigency Execution (SCCE)

In [7]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime

# ================================================================
# SCCE CODING
# INPUT FILE: Themes.xlsx
# ================================================================

input_file = Path("Themes.xlsx")

if not input_file.exists():
    raise FileNotFoundError(
        f"Themes.xlsx was not found in:\n{Path.cwd()}"
    )

excel = pd.ExcelFile(input_file)


# ================================================================
# 1. FIND PARTICIPANT SHEETS
# ================================================================

participant_sheets = []

for sheet in excel.sheet_names:

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        str(sheet).strip(),
        flags=re.IGNORECASE
    )

    if match:

        number = int(match.group(1))

        if 1 <= number <= 26:
            participant_sheets.append(
                (number, sheet)
            )

participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)

print("Participant sheets found:")

for number, sheet in participant_sheets:
    print(f"P{number:02d} -> {sheet}")


# ================================================================
# 2. EXTRACT SCCE RESPONSES
# ================================================================

raw_rows = []

for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if len(values) < 2:
            continue

        # Find SCCE regardless of capitalization
        construct_position = None

        for position, value in enumerate(values):

            if value.strip().upper() == "SCCE":

                construct_position = position
                break

        if construct_position is None:
            continue

        # Everything after SCCE
        remaining = values[
            construct_position + 1:
        ]

        if not remaining:
            continue

        response = " ".join(
            remaining
        ).strip()

        if response:

            raw_rows.append({

                "Participant":
                    participant,

                "Construct":
                    "SCCE",

                "Original_Response":
                    response,

                "Source_Sheet":
                    sheet,

                "Source_Row":
                    row_number + 1
            })


# ================================================================
# 3. CHECK SCCE RESPONSES
# ================================================================

original_df = pd.DataFrame(
    raw_rows
)

if original_df.empty:

    raise ValueError(
        """
        No SCCE responses were found.

        Check that the construct is written as SCCE
        in the participant sheets.
        """
    )

print("\nSCCE original responses:")
print(len(original_df))


# ================================================================
# 4. SPLIT INTO RAW THEMES
# ================================================================

theme_rows = []

for _, row in original_df.iterrows():

    response = str(
        row["Original_Response"]
    )

    response = response.replace(
        "\n", ","
    )

    response = response.replace(
        ";", ","
    )

    response = response.replace(
        "•", ","
    )

    themes = response.split(",")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if theme == "":
            continue

        theme_rows.append({

            "Participant":
                row["Participant"],

            "Construct":
                "SCCE",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    theme_rows
)


# ================================================================
# 5. CLEAN THEMES
# ================================================================

clean_df = raw_df.copy()

clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)

clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)

clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]

clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)

clean_df = clean_df.reset_index(
    drop=True
)


print("\nSCCE cleaned theme observations:")
print(len(clean_df))


# ================================================================
# 6. DISPLAY UNIQUE RAW SCCE THEMES
# ================================================================

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)

print("\n")
print("=" * 60)
print("SCCE RAW THEMES")
print("=" * 60)

print(
    raw_theme_list.to_string(
        index=False
    )
)


# ================================================================
# 7. SCCE NORMALIZATION DICTIONARY
# ================================================================
#
# IMPORTANT:
# Do not automatically merge themes simply because their words
# look similar.
#
# Add researcher-approved conceptual mappings here after seeing
# the actual SCCE raw themes.
#
# Example format:
#
# "rapid corrective action":
#     "Rapid corrective action",
#
# "quick corrective response":
#     "Rapid corrective action",
#
# ================================================================

SCCE_NORMALIZATION = {

    # ------------------------------------------------------------
    # ADD ACTUAL SCCE MAPPINGS HERE
    # ------------------------------------------------------------

}


# ================================================================
# 8. APPLY NORMALIZATION
# ================================================================

coding_df = raw_theme_list.copy()

coding_df["Normalized_Theme"] = ""

coding_df["Decision"] = ""

coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    theme_key = str(
        row["Theme_Key"]
    ).strip().lower()

    if theme_key in SCCE_NORMALIZATION:

        normalized = SCCE_NORMALIZATION[
            theme_key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized

        if theme_key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as a distinct conceptual theme."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Merged with a semantically equivalent "
                "theme."
            )

    else:

        # Until conceptual mapping is approved,
        # retain the raw theme rather than inventing
        # a merger.

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row["Theme_Clean"]

        coding_df.loc[
            i,
            "Decision"
        ] = "Keep"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "Retained pending conceptual review."
        )


# ================================================================
# 9. MAP BACK TO PARTICIPANTS
# ================================================================

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]

coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ================================================================
# 10. PARTICIPANT × SCCE THEME MATRIX
# ================================================================

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)

matrix = pd.crosstab(
    matrix_source[
        "Normalized_Theme"
    ],
    matrix_source[
        "Participant"
    ]
)

matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)

matrix = matrix.reset_index()

matrix["Frequency"] = matrix[
    participants
].sum(axis=1)

matrix["Percentage"] = (
    matrix["Frequency"]
    / len(participants)
    * 100
).round(1)


# ================================================================
# 11. SCCE THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].sort_values(
    "Frequency",
    ascending=False
)


# ================================================================
# 12. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Count"
    )
)


# ================================================================
# 13. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    coded_df
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)

participant_coverage.columns = [
    "Participant",
    "SCCE_Theme_Count"
]


# ================================================================
# 14. SAVE EXCEL
# ================================================================

output_file = Path(
    "SCCE_Coding_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


# ================================================================
# 15. FINAL REPORT
# ================================================================

print("\n")
print("=" * 60)
print("SCCE CODING COMPLETED")
print("=" * 60)

print(
    "Participants:",
    clean_df["Participant"].nunique()
)

print(
    "Raw theme observations:",
    len(clean_df)
)

print(
    "Unique SCCE raw themes:",
    coding_df["Theme_Key"].nunique()
)

print(
    "Normalized SCCE themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print("\nDECISION SUMMARY:")

print(
    decision_summary.to_string(
        index=False
    )
)

print("\nOUTPUT FILE:")

print(
    output_file.resolve()
)

print("\nDONE.")

Participant sheets found:
P01 -> 1
P02 -> 2
P03 -> 3
P04 -> 4
P05 -> 5
P06 -> 6
P07 -> 7
P08 -> 8
P09 -> 9
P10 -> 10
P11 -> 11
P12 -> 12
P13 -> 13
P14 -> 14
P15 -> 15
P16 -> 16
P17 -> 17
P18 -> 18
P19 -> 19
P20 -> 20
P21 -> 21
P22 -> 22
P23 -> 23
P24 -> 24
P25 -> 25
P26 -> 26

SCCE original responses:
26

SCCE cleaned theme observations:
94


SCCE RAW THEMES
                Raw_Theme               Theme_Clean                 Theme_Key
        Agreed procedures         Agreed procedures         agreed procedures
          Agreed triggers           Agreed triggers           agreed triggers
       approval reduction        approval reduction        approval reduction
     automated initiation      automated initiation      automated initiation
         automatic action          automatic action          automatic action
      automatic execution       automatic execution       automatic execution
     automatic initiation      automatic initiation      automatic initiation
               

In [8]:
# ================================================================
# COMPLETE SCCE QUALITATIVE ANALYSIS
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path("SCCE_Coding_20260827_105003.xlsx")

if not input_file.exists():

    possible_files = list(Path(".").glob("SCCE_Coding_*.xlsx"))

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "\nSCCE input file was not found.\n"
            "Make sure the Excel file is in the same folder "
            "as your Python notebook."
        )

    elif len(possible_files) == 1:
        input_file = possible_files[0]

    else:
        print("SCCE files found:")
        for f in possible_files:
            print(" -", f.name)

        raise ValueError(
            "\nMore than one SCCE coding file was found. "
            "Please keep the intended file."
        )


print("=" * 70)
print("SCCE QUALITATIVE CODING ANALYSIS")
print("=" * 70)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nSheets found:")

for sheet in excel.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FIND SHEETS FLEXIBLY
# ================================================================

def find_sheet(possible_names):

    # Exact match first
    for name in possible_names:
        if name in excel.sheet_names:
            return name

    # Flexible match
    for sheet in excel.sheet_names:

        clean_sheet = (
            sheet.lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for name in possible_names:

            clean_name = (
                name.lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if clean_name in clean_sheet:
                return sheet

    return None


original_sheet = find_sheet([
    "01_Original_SCCE",
    "01_Original",
    "Original_SCCE",
    "Original"
])

raw_sheet = find_sheet([
    "02_Raw_SCCE",
    "02_Raw",
    "Raw_SCCE",
    "Raw"
])

clean_sheet = find_sheet([
    "03_Cleaned_SCCE",
    "03_Cleaned",
    "Cleaned_SCCE",
    "Cleaned"
])

coding_sheet = find_sheet([
    "04_Normalized_Coding",
    "04_Coding",
    "Normalized_Coding",
    "Coding"
])


print("\nSelected sheets:")
print("Original:", original_sheet)
print("Raw:", raw_sheet)
print("Cleaned:", clean_sheet)
print("Coding:", coding_sheet)


# ================================================================
# 4. READ THE DATA
# ================================================================

if original_sheet:
    original_df = pd.read_excel(
        input_file,
        sheet_name=original_sheet
    )
else:
    original_df = pd.DataFrame()


if raw_sheet:
    raw_df = pd.read_excel(
        input_file,
        sheet_name=raw_sheet
    )
else:
    raw_df = pd.DataFrame()


if clean_sheet:
    clean_df = pd.read_excel(
        input_file,
        sheet_name=clean_sheet
    )
else:
    clean_df = pd.DataFrame()


if coding_sheet:
    coding_df = pd.read_excel(
        input_file,
        sheet_name=coding_sheet
    )
else:
    raise ValueError(
        "\nNormalized Coding sheet could not be found."
    )


# ================================================================
# 5. STANDARDIZE COLUMN NAMES
# ================================================================

for df in [
    original_df,
    raw_df,
    clean_df,
    coding_df
]:

    if not df.empty:

        df.columns = [
            str(c).strip()
            for c in df.columns
        ]


print("\nCoding columns:")
print(list(coding_df.columns))


# ================================================================
# 6. IDENTIFY CODING COLUMNS
# ================================================================

def find_column(df, candidates):

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    # Flexible search
    for column in df.columns:

        clean_column = (
            str(column)
            .lower()
            .replace(" ", "")
            .replace("_", "")
            .replace("-", "")
        )

        for candidate in candidates:

            clean_candidate = (
                candidate
                .lower()
                .replace(" ", "")
                .replace("_", "")
                .replace("-", "")
            )

            if clean_column == clean_candidate:
                return column

    return None


theme_key_col = find_column(
    coding_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)

raw_theme_col = find_column(
    coding_df,
    [
        "Raw_Theme",
        "Raw Theme",
        "RawTheme"
    ]
)

normalized_col = find_column(
    coding_df,
    [
        "Normalized_Theme",
        "Normalized Theme",
        "NormalizedTheme"
    ]
)

decision_col = find_column(
    coding_df,
    [
        "Decision"
    ]
)


if theme_key_col is None:
    raise ValueError(
        "\nTheme_Key column not found."
    )

if raw_theme_col is None:
    raise ValueError(
        "\nRaw_Theme column not found."
    )

if normalized_col is None:
    raise ValueError(
        "\nNormalized_Theme column not found."
    )

if decision_col is None:
    raise ValueError(
        "\nDecision column not found."
    )


# Rename to standard names

coding_df = coding_df.rename(
    columns={
        theme_key_col: "Theme_Key",
        raw_theme_col: "Raw_Theme",
        normalized_col: "Normalized_Theme",
        decision_col: "Decision"
    }
)


# ================================================================
# 7. IDENTIFY PARTICIPANT COLUMN
# ================================================================

participant_col = find_column(
    clean_df,
    [
        "Participant",
        "participant",
        "Participant_ID",
        "Participant ID",
        "ParticipantID"
    ]
)


if participant_col is None:

    raise ValueError(
        "\nParticipant column was not found in the "
        "Cleaned SCCE sheet.\n\n"
        "Available columns:\n"
        + str(list(clean_df.columns))
    )


if participant_col != "Participant":

    clean_df = clean_df.rename(
        columns={
            participant_col:
            "Participant"
        }
    )


# ================================================================
# 8. IDENTIFY THEME KEY IN CLEANED DATA
# ================================================================

clean_theme_key = find_column(
    clean_df,
    [
        "Theme_Key",
        "Theme Key",
        "ThemeKey"
    ]
)


if clean_theme_key is None:

    raise ValueError(
        "\nTheme_Key was not found in the Cleaned SCCE sheet."
    )


if clean_theme_key != "Theme_Key":

    clean_df = clean_df.rename(
        columns={
            clean_theme_key:
            "Theme_Key"
        }
    )


# ================================================================
# 9. CLEAN CODING DICTIONARY
# ================================================================

coding_df["Theme_Key"] = (
    coding_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

coding_df["Raw_Theme"] = (
    coding_df["Raw_Theme"]
    .fillna("")
    .astype(str)
    .str.strip()
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .replace(
        ["nan", "None", "", "NaN"],
        np.nan
    )
)

coding_df["Normalized_Theme"] = (
    coding_df["Normalized_Theme"]
    .astype("string")
    .str.strip()
)

coding_df["Decision"] = (
    coding_df["Decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# ================================================================
# 10. CLEAN CLEANED DATA
# ================================================================

clean_df["Theme_Key"] = (
    clean_df["Theme_Key"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

clean_df["Participant"] = (
    clean_df["Participant"]
    .astype(str)
    .str.strip()
)


# ================================================================
# 11. PARTICIPANTS
# ================================================================

participants = sorted(
    clean_df[
        "Participant"
    ]
    .dropna()
    .unique()
)


print("\nParticipants identified:")
print(participants)

print(
    "\nNumber of participants:",
    len(participants)
)


# ================================================================
# 12. MERGE CODING DICTIONARY
# ================================================================

coded_df = clean_df.merge(
    coding_df[
        [
            "Theme_Key",
            "Normalized_Theme",
            "Decision"
        ]
    ],
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. UNMAPPED THEMES
# ================================================================

missing_mapping = coded_df[
    coded_df[
        "Normalized_Theme"
    ].isna()
].copy()


print(
    "\nNumber of unmapped themes:",
    len(missing_mapping)
)


if len(missing_mapping) > 0:

    print("\nUnmapped themes:")

    print(
        missing_mapping[
            [
                "Participant",
                "Theme_Key"
            ]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )


# ================================================================
# 14. PARTICIPANT × NORMALIZED THEME
# ================================================================

participant_theme = (
    coded_df[
        [
            "Participant",
            "Normalized_Theme"
        ]
    ]
    .dropna()
    .drop_duplicates()
)


# ================================================================
# 15. PARTICIPANT × THEME MATRIX
# ================================================================

matrix = pd.crosstab(
    participant_theme[
        "Normalized_Theme"
    ],
    participant_theme[
        "Participant"
    ]
)


# P01-P26 ordering

expected_participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]

existing = [
    p
    for p in expected_participants
    if p in matrix.columns
]

other = [
    p
    for p in matrix.columns
    if p not in existing
]

matrix = matrix.reindex(
    columns=existing + other,
    fill_value=0
)


matrix = matrix.reset_index()


# ================================================================
# 16. FREQUENCY AND PERCENTAGE
# ================================================================

participant_columns = (
    existing + other
)


matrix["Frequency"] = matrix[
    participant_columns
].sum(axis=1)


total_participants = len(participants)


matrix["Percentage"] = (
    matrix["Frequency"]
    / total_participants
    * 100
).round(1)


# ================================================================
# 17. RANK THEMES
# ================================================================

matrix = matrix.sort_values(
    by=[
        "Frequency",
        "Normalized_Theme"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 18. THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Frequency",
        "Percentage"
    ]
].copy()


theme_summary = theme_summary.rename(
    columns={
        "Normalized_Theme":
            "Final_SCCE_Theme",

        "Frequency":
            "Experts_Mentioning",

        "Percentage":
            "Percentage_of_Experts"
    }
)


# ================================================================
# 19. PREVALENCE
# ================================================================

def prevalence_category(p):

    if p >= 75:
        return "Very High"

    elif p >= 50:
        return "High"

    elif p >= 25:
        return "Moderate"

    else:
        return "Low"


theme_summary[
    "Prevalence_Category"
] = (
    theme_summary[
        "Percentage_of_Experts"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 20. CODING AUDIT
# ================================================================

theme_participant_counts = (
    participant_theme
    .groupby(
        "Normalized_Theme"
    )
    .size()
    .reset_index(
        name="Experts_Mentioning"
    )
)


coding_audit = coding_df.merge(
    theme_participant_counts,
    on="Normalized_Theme",
    how="left"
)


coding_audit[
    "Experts_Mentioning"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


coding_audit[
    "Percentage_of_Experts"
] = (
    coding_audit[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


coding_audit = coding_audit.sort_values(
    [
        "Normalized_Theme",
        "Decision",
        "Raw_Theme"
    ]
).reset_index(drop=True)


# ================================================================
# 21. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Number_of_Raw_Themes"
    )
)


decision_summary[
    "Percentage"
] = (
    decision_summary[
        "Number_of_Raw_Themes"
    ]
    / len(coding_df)
    * 100
).round(1)


# ================================================================
# 22. NORMALIZATION SUMMARY
# ================================================================

normalization_summary = (
    coding_df
    .dropna(
        subset=[
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Normalized_Theme"
    )
    .agg(
        Raw_Themes=(
            "Raw_Theme",
            "count"
        ),

        Keep_Count=(
            "Decision",
            lambda x:
            (x == "Keep").sum()
        ),

        Merge_Count=(
            "Decision",
            lambda x:
            (x == "Merge").sum()
        )
    )
    .reset_index()
)


normalization_summary = (
    normalization_summary.merge(
        theme_participant_counts,
        on="Normalized_Theme",
        how="left"
    )
)


normalization_summary[
    "Experts_Mentioning"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    .fillna(0)
    .astype(int)
)


normalization_summary[
    "Percentage_of_Experts"
] = (
    normalization_summary[
        "Experts_Mentioning"
    ]
    / total_participants
    * 100
).round(1)


normalization_summary = (
    normalization_summary
    .sort_values(
        "Experts_Mentioning",
        ascending=False
    )
    .reset_index(drop=True)
)


# ================================================================
# 23. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    participant_theme
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [
    "Participant",
    "Number_of_Normalized_Themes"
]


# ================================================================
# 24. CONSTRUCT STATISTICS
# ================================================================

construct_statistics = pd.DataFrame({

    "Construct": ["SCCE"],

    "Participants": [
        total_participants
    ],

    "Original_Response_Rows": [
        len(original_df)
    ],

    "Raw_Theme_Observations": [
        len(raw_df)
    ],

    "Cleaned_Theme_Observations": [
        len(clean_df)
    ],

    "Unique_Raw_Themes": [
        coding_df[
            "Theme_Key"
        ].nunique()
    ],

    "Final_Normalized_Themes": [
        coding_df[
            "Normalized_Theme"
        ].nunique()
    ],

    "Merged_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()
    ],

    "Kept_Raw_Themes": [
        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum()
    ],

    "Unmapped_Themes": [
        len(missing_mapping)
    ]
})


# ================================================================
# 25. FINAL SCCE EVIDENCE
# ================================================================

final_evidence = normalization_summary[
    [
        "Normalized_Theme",
        "Raw_Themes",
        "Experts_Mentioning",
        "Percentage_of_Experts",
        "Keep_Count",
        "Merge_Count"
    ]
].copy()


final_evidence = final_evidence.rename(
    columns={
        "Normalized_Theme":
            "Final_SCCE_Theme",

        "Raw_Themes":
            "Number_of_Raw_Themes",

        "Percentage_of_Experts":
            "Expert_Prevalence_%",

        "Keep_Count":
            "Raw_Themes_Kept",

        "Merge_Count":
            "Raw_Themes_Merged"
    }
)


final_evidence[
    "Prevalence_Category"
] = (
    final_evidence[
        "Expert_Prevalence_%"
    ].apply(
        prevalence_category
    )
)


# ================================================================
# 26. QUALITY CHECKS
# ================================================================

duplicate_count = (
    len(coded_df)
    -
    len(
        coded_df[
            [
                "Participant",
                "Normalized_Theme"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )
)


quality_checks = pd.DataFrame({

    "Check": [

        "Number of participants",

        "Original response rows",

        "Raw theme observations",

        "Unique raw themes",

        "Final normalized themes",

        "Unmapped themes",

        "Duplicate participant-theme records",

        "Keep decisions",

        "Merge decisions"

    ],

    "Result": [

        total_participants,

        len(original_df),

        len(raw_df),

        coding_df[
            "Theme_Key"
        ].nunique(),

        coding_df[
            "Normalized_Theme"
        ].nunique(),

        len(missing_mapping),

        duplicate_count,

        (
            coding_df[
                "Decision"
            ]
            == "Keep"
        ).sum(),

        (
            coding_df[
                "Decision"
            ]
            == "Merge"
        ).sum()

    ],

    "Status": [

        "PASS"
        if total_participants > 0
        else "CHECK",

        "PASS"
        if len(original_df) > 0
        else "CHECK",

        "PASS"
        if len(raw_df) > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Theme_Key"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if coding_df[
            "Normalized_Theme"
        ].nunique() > 0
        else "CHECK",

        "PASS"
        if len(missing_mapping) == 0
        else "CHECK",

        "INFO",

        "PASS",

        "PASS"
    ]
})


# ================================================================
# 27. SAVE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

output_file = Path(
    f"SCCE_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )

    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )

    construct_statistics.to_excel(
        writer,
        sheet_name="10_Construct_Statistics",
        index=False
    )

    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_SCCE_Evidence",
        index=False
    )

    quality_checks.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )

    missing_mapping.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 28. FINAL REPORT
# ================================================================

print("\n")
print("=" * 70)
print("SCCE ANALYSIS COMPLETED")
print("=" * 70)

print(
    "\nParticipants:",
    total_participants
)

print(
    "Original responses:",
    len(original_df)
)

print(
    "Raw theme observations:",
    len(raw_df)
)

print(
    "Cleaned theme observations:",
    len(clean_df)
)

print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)

print(
    "Final normalized SCCE themes:",
    coding_df[
        "Normalized_Theme"
    ].nunique()
)

print(
    "Keep decisions:",
    (
        coding_df[
            "Decision"
        ] == "Keep"
    ).sum()
)

print(
    "Merge decisions:",
    (
        coding_df[
            "Decision"
        ] == "Merge"
    ).sum()
)

print(
    "Unmapped themes:",
    len(missing_mapping)
)


# ================================================================
# 29. DISPLAY FINAL SCCE THEMES
# ================================================================

print("\n")
print("=" * 70)
print("FINAL SCCE THEMES")
print("=" * 70)

print(
    final_evidence[
        [
            "Final_SCCE_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Prevalence_Category"
        ]
    ].to_string(index=False)
)


# ================================================================
# 30. OUTPUT LOCATION
# ================================================================

print("\n")
print("=" * 70)
print("OUTPUT FILE")
print("=" * 70)

print(
    output_file.resolve()
)

print(
    "\nComplete SCCE qualitative analysis workbook "
    "created successfully."
)

SCCE QUALITATIVE CODING ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\SCCE_Coding_20260827_105003.xlsx

Sheets found:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage

Selected sheets:
Original: 01_Original_Responses
Raw: 02_Raw_Themes
Cleaned: 03_Cleaned_Themes
Coding: 04_Coding_Dictionary

Coding columns:
['Raw_Theme', 'Theme_Clean', 'Theme_Key', 'Normalized_Theme', 'Decision', 'Reason']

Participants identified:
['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26']

Number of participants: 26

Number of unmapped themes: 0


SCCE ANALYSIS COMPLETED

Participants: 26
Original responses: 26
Raw theme observation

In [ ]:
# ================================================================
# SCCE — FINAL QUALITATIVE EVIDENCE + DIMENSION ANALYSIS
# ================================================================
#
# INPUT:
#   SCCE_FINAL_QUALITATIVE_ANALYSIS_20260827_105058.xlsx
#
# OUTPUT:
#   SCCE_FINAL_EVIDENCE_AND_DIMENSIONS_20260827_105229.xlsx
#
# This code uses the completed SCCE qualitative coding workbook.
#
# It creates:
#
# 01_Original_Responses
# 02_Raw_Themes
# 03_Cleaned_Themes
# 04_Coding_Audit
# 05_Participant_Matrix
# 06_Theme_Summary
# 07_Normalization_Summary
# 08_Decision_Summary
# 09_Participant_Coverage
# 10_Unmapped_Check
# 11_Decision_Check
# 12_Quality_Checks
# 13_Final_SCCE_Evidence
# 14_SCCE_Dimension_Summary
# 15_SCCE_Dimension_Themes
#
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os


# ================================================================
# 1. FIND SCCE INPUT FILE
# ================================================================

TARGET = "SCCE_FINAL_QUALITATIVE_ANALYSIS_20260827_105058"

possible_files = []

# Search current directory and subdirectories
for ext in [".xlsx", ".xlsm", ".xls"]:

    possible_files.extend(
        Path(".").rglob(TARGET + ext)
    )

if len(possible_files) == 0:

    raise FileNotFoundError(
        "\n\nSCCE input file was not found.\n"
        f"Expected file:\n{TARGET}.xlsx\n\n"
        "Place the file in the same folder as your notebook/script."
    )


INPUT_FILE = possible_files[0]

print("=" * 80)
print("SCCE FINAL QUALITATIVE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(
    INPUT_FILE,
    engine="openpyxl"
)

print("\nAvailable sheets:")

for s in xls.sheet_names:
    print(" -", s)


# ================================================================
# 3. HELPER FUNCTION
# ================================================================

def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).strip()

    x = re.sub(
        r"\s+",
        " ",
        x
    )

    return x


# ================================================================
# 4. READ ALL EXISTING SHEETS
# ================================================================

sheets = {}

for sheet in xls.sheet_names:

    try:

        sheets[sheet] = pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet
        )

    except Exception as e:

        print(
            f"Warning: could not read {sheet}: {e}"
        )


# ================================================================
# 5. IDENTIFY IMPORTANT SHEETS
# ================================================================

def find_sheet(keyword):

    for s in sheets.keys():

        if keyword.lower() in s.lower():

            return s

    return None


raw_sheet = find_sheet("02_Raw_Themes")
cleaned_sheet = find_sheet("03_Cleaned_Themes")
audit_sheet = find_sheet("04_Coding_Audit")
matrix_sheet = find_sheet("05_Participant_Matrix")
theme_sheet = find_sheet("06_Theme_Summary")
normalization_sheet = find_sheet("07_Normalization_Summary")
decision_sheet = find_sheet("08_Decision_Summary")
coverage_sheet = find_sheet("09_Participant_Coverage")
quality_sheet = find_sheet("12_Quality_Checks")
final_sheet = find_sheet("11_Final_SCCE_Evidence")
unmapped_sheet = find_sheet("13_Unmapped_Themes")


print("\nDetected sheets:")

print("Raw themes:",
      raw_sheet)

print("Coding audit:",
      audit_sheet)

print("Theme summary:",
      theme_sheet)

print("Final SCCE evidence:",
      final_sheet)


# ================================================================
# 6. READ FINAL SCCE EVIDENCE
# ================================================================

if final_sheet is None:

    raise ValueError(
        "11_Final_SCCE_Evidence sheet was not found."
    )


final_evidence = sheets[
    final_sheet
].copy()


print("\nFinal SCCE evidence columns:")

print(
    list(
        final_evidence.columns
    )
)


# ================================================================
# 7. STANDARDIZE FINAL EVIDENCE COLUMNS
# ================================================================

final_evidence.columns = [
    str(c).strip()
    for c in final_evidence.columns
]


# Find theme column

theme_col = None

for c in final_evidence.columns:

    if "Final_SCCE_Theme" in str(c):

        theme_col = c
        break


if theme_col is None:

    for c in final_evidence.columns:

        if "Theme" in str(c):

            theme_col = c
            break


if theme_col is None:

    raise ValueError(
        "Could not identify SCCE theme column."
    )


final_evidence[
    "Final_SCCE_Theme"
] = (
    final_evidence[
        theme_col
    ]
    .apply(clean_text)
)


# ================================================================
# 8. STANDARDIZE EXPERT PREVALENCE
# ================================================================

if "Experts_Mentioning" not in final_evidence.columns:

    if "Experts Mentioning" in final_evidence.columns:

        final_evidence[
            "Experts_Mentioning"
        ] = final_evidence[
            "Experts Mentioning"
        ]


if "Expert_Prevalence_%" not in final_evidence.columns:

    if "Percentage_of_Experts" in final_evidence.columns:

        final_evidence[
            "Expert_Prevalence_%"
        ] = final_evidence[
            "Percentage_of_Experts"
        ]


# ================================================================
# 9. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(x):

    try:
        x = float(x)
    except:
        return "Not available"

    if x >= 75:
        return "Very High"

    elif x >= 50:
        return "High"

    elif x >= 25:
        return "Moderate"

    else:
        return "Low"


if "Expert_Prevalence_%" in final_evidence.columns:

    final_evidence[
        "Prevalence_Category"
    ] = (
        final_evidence[
            "Expert_Prevalence_%"
        ]
        .apply(prevalence_category)
    )


# ================================================================
# 10. REMOVE COMPLETELY EMPTY THEMES
# ================================================================

final_evidence = final_evidence[
    final_evidence[
        "Final_SCCE_Theme"
    ] != ""
].copy()


# ================================================================
# 11. READ PARTICIPANT MATRIX
# ================================================================

participant_matrix = None

if matrix_sheet is not None:

    participant_matrix = sheets[
        matrix_sheet
    ].copy()


# ================================================================
# 12. READ CODING AUDIT
# ================================================================

coding_audit = None

if audit_sheet is not None:

    coding_audit = sheets[
        audit_sheet
    ].copy()


# ================================================================
# 13. CREATE PARTICIPANT INFORMATION
# ================================================================

participants = []

if participant_matrix is not None:

    # Participant matrix normally has participants as columns.
    # Remove obvious non-participant columns.

    for c in participant_matrix.columns:

        cstr = str(c)

        if re.match(
            r"^P\d+$",
            cstr,
            flags=re.IGNORECASE
        ):

            participants.append(cstr)


# If matrix did not identify participants,
# infer participant count from prevalence.

if len(participants) == 0:

    n_participants = 26

else:

    n_participants = len(participants)


print(
    "\nNumber of participants:",
    n_participants
)


# ================================================================
# 14. CREATE FINAL SCCE EVIDENCE TABLE
# ================================================================

evidence_columns = []

for c in [
    "Final_SCCE_Theme",
    "Number_of_Raw_Themes",
    "Experts_Mentioning",
    "Expert_Prevalence_%",
    "Raw_Themes_Kept",
    "Raw_Themes_Merged",
    "Prevalence_Category"
]:

    if c in final_evidence.columns:

        evidence_columns.append(c)


final_scc_evidence = final_evidence[
    evidence_columns
].copy()


# ================================================================
# 15. SORT FINAL EVIDENCE
# ================================================================

if "Experts_Mentioning" in final_scc_evidence.columns:

    final_scc_evidence = (
        final_scc_evidence
        .sort_values(
            [
                "Experts_Mentioning",
                "Final_SCCE_Theme"
            ],
            ascending=[
                False,
                True
            ]
        )
        .reset_index(drop=True)
    )


final_scc_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_scc_evidence) + 1
    )
)


# ================================================================
# 16. SCCE DIMENSION MAPPING
# ================================================================
#
# The mapping is based specifically on the themes actually present
# in your SCCE qualitative evidence.
#
# Dimensions:
#
# 1. Rule & Trigger Specification
# 2. Automated Execution
# 3. Predefined Response & Procedural Standardization
# 4. Response Speed & Reduced Administrative Delay
# 5. Exception Handling & Human Oversight
#
# "Review Required" is used where the theme does not clearly fit.
#
# ================================================================

def map_scce_dimension(theme):

    t = str(theme).lower().strip()


    # ------------------------------------------------------------
    # 1. RULE & TRIGGER SPECIFICATION
    # ------------------------------------------------------------

    trigger_words = [

        "trigger",
        "triggers",
        "threshold",
        "thresholds",
        "condition",
        "conditions",
        "rule",
        "rules",
        "objective condition",
        "measurable condition",
        "measurable trigger",
        "objective trigger",
        "verified trigger",
        "contractual trigger",
        "agreed trigger",
        "clear condition",
        "clear rule",
        "defined rule",
        "defined trigger",
        "predefined condition",
        "predefined rule",
        "predefined event",
        "condition activation",
        "threshold activation",
        "threshold trigger"

    ]

    if any(
        word in t
        for word in trigger_words
    ):

        return "Rule & Trigger Specification"


    # ------------------------------------------------------------
    # 2. AUTOMATED EXECUTION
    # ------------------------------------------------------------

    automation_words = [

        "automation",
        "automated",
        "automatic",
        "automatically",
        "automatic action",
        "automatic execution",
        "automatic initiation",
        "automated initiation",
        "trigger execution",
        "condition-based execution",
        "routine automation",
        "repetitive automation",
        "rule-based automation"

    ]

    if any(
        word in t
        for word in automation_words
    ):

        return "Automated Execution"


    # ------------------------------------------------------------
    # 3. PREDEFINED RESPONSE & PROCEDURAL STANDARDIZATION
    # ------------------------------------------------------------

    procedure_words = [

        "predefined procedure",
        "predefined procedures",
        "predefined response",
        "predefined responses",
        "predefined action",
        "predefined actions",
        "predefined process",
        "predefined processes",
        "agreed procedure",
        "agreed procedures",
        "predictable procedure",
        "predictable procedures",
        "procedural reliability",
        "defined procedure",
        "defined procedures",
        "rules"

    ]

    if any(
        word in t
        for word in procedure_words
    ):

        return "Predefined Response & Procedural Standardization"


    # ------------------------------------------------------------
    # 4. RESPONSE SPEED & REDUCED ADMINISTRATIVE DELAY
    # ------------------------------------------------------------

    speed_words = [

        "speed",
        "response speed",
        "immediate action",
        "immediate execution",
        "reduced waiting",
        "reduced approval",
        "reduced approvals",
        "approval reduction",
        "reduced manual steps",
        "rapid",
        "quick",
        "fast",
        "delay"

    ]

    if any(
        word in t
        for word in speed_words
    ):

        return "Response Speed & Reduced Administrative Delay"


    # ------------------------------------------------------------
    # 5. EXCEPTION HANDLING & HUMAN OVERSIGHT
    # ------------------------------------------------------------

    human_words = [

        "exception",
        "exceptions",
        "exception handling",
        "exception review",
        "human assessment",
        "human intervention",
        "human judgment",
        "human oversight",
        "managerial intervention",
        "managerial judgment",
        "managerial review",
        "judgment boundary",
        "negotiation boundary",
        "automation boundaries",
        "escalation"

    ]

    if any(
        word in t
        for word in human_words
    ):

        return "Exception Handling & Human Oversight"


    # ------------------------------------------------------------
    # UNRESOLVED
    # ------------------------------------------------------------

    return "Review Required"


final_scc_evidence[
    "SCCE_Dimension"
] = (
    final_scc_evidence[
        "Final_SCCE_Theme"
    ]
    .apply(
        map_scce_dimension
    )
)


# ================================================================
# 17. CREATE DIMENSION SUMMARY
# ================================================================

dimension_rows = []


for dimension, group in (
    final_scc_evidence
    .groupby(
        "SCCE_Dimension"
    )
):

    themes = (
        group[
            "Final_SCCE_Theme"
        ]
        .astype(str)
        .tolist()
    )


    # Calculate dimension-level expert prevalence.
    #
    # We use the maximum expert count among the themes when the
    # participant matrix is unavailable.
    #

    if (
        participant_matrix is not None
        and len(participants) > 0
    ):

        # Identify normalized theme index column

        index_col = (
            participant_matrix.columns[0]
        )

        temp = participant_matrix.copy()

        temp[
            "Theme_TEMP"
        ] = temp[
            index_col
        ].astype(str)


        matched = temp[
            temp[
                "Theme_TEMP"
            ].isin(
                themes
            )
        ]


        if len(matched) > 0:

            participant_values = (
                matched[
                    participants
                ]
                .fillna(0)
                .astype(float)
            )


            experts_mentioning = int(
                (
                    participant_values
                    .sum(axis=0) > 0
                )
                .sum()
            )

        else:

            experts_mentioning = 0

    else:

        experts_mentioning = int(
            group[
                "Experts_Mentioning"
            ].max()
        )


    prevalence = (

        experts_mentioning
        /
        n_participants
        *
        100

        if n_participants > 0
        else 0
    )


    dimension_rows.append({

        "SCCE_Dimension":
            dimension,

        "Number_of_Normalized_Themes":
            len(themes),

        "Experts_Mentioning":
            experts_mentioning,

        "Expert_Prevalence_%":
            round(
                prevalence,
                1
            ),

        "Prevalence_Category":
            prevalence_category(
                prevalence
            ),

        "Included_SCCE_Themes":
            "; ".join(
                themes
            )

    })


dimension_summary = pd.DataFrame(
    dimension_rows
)


# ================================================================
# 18. SORT DIMENSIONS
# ================================================================

if len(dimension_summary) > 0:

    dimension_summary = (
        dimension_summary
        .sort_values(
            [
                "Experts_Mentioning",
                "Number_of_Normalized_Themes"
            ],
            ascending=False
        )
        .reset_index(drop=True)
    )


    dimension_summary.insert(
        0,
        "Rank",
        range(
            1,
            len(dimension_summary) + 1
        )
    )


# ================================================================
# 19. DIMENSION × THEME TABLE
# ================================================================

dimension_themes = final_scc_evidence[
    [
        "SCCE_Dimension",
        "Final_SCCE_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
].copy()


dimension_themes = (
    dimension_themes
    .sort_values(
        [
            "SCCE_Dimension",
            "Experts_Mentioning"
        ],
        ascending=[
            True,
            False
        ]
    )
)


# ================================================================
# 20. CREATE UNMAPPED CHECK
# ================================================================

unmapped_dimension = final_scc_evidence[
    final_scc_evidence[
        "SCCE_Dimension"
    ] == "Review Required"
].copy()


if len(unmapped_dimension) > 0:

    unmapped_check = unmapped_dimension[
        [
            "Final_SCCE_Theme",
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ]
    ].copy()

    unmapped_check.insert(
        0,
        "Status",
        "REVIEW REQUIRED"
    )

else:

    unmapped_check = pd.DataFrame({

        "Status": [
            "PASS — All SCCE themes mapped to a dimension."
        ]

    })


# ================================================================
# 21. DECISION CHECK
# ================================================================

if coding_audit is not None:

    audit_cols = [
        c for c in coding_audit.columns
        if "Decision" in str(c)
    ]

    if len(audit_cols) > 0:

        decision_col = audit_cols[0]

        missing_decisions = coding_audit[
            coding_audit[
                decision_col
            ]
            .isna()
        ]

        if len(missing_decisions) == 0:

            decision_check = pd.DataFrame({

                "Status": [
                    "PASS — All SCCE coding decisions are present."
                ]

            })

        else:

            decision_check = (
                missing_decisions
                .copy()
            )

    else:

        decision_check = pd.DataFrame({

            "Status": [
                "CHECK — Decision column not detected."
            ]

        })

else:

    decision_check = pd.DataFrame({

        "Status": [
            "CHECK — Coding audit sheet unavailable."
        ]

    })


# ================================================================
# 22. QUALITY CHECKS
# ================================================================

quality_rows = []


quality_rows.append({

    "Quality_Check":
        "Input workbook found",

    "Result":
        "PASS",

    "Details":
        INPUT_FILE.name

})


quality_rows.append({

    "Quality_Check":
        "Final SCCE evidence available",

    "Result":
        "PASS"
        if len(final_scc_evidence) > 0
        else "FAIL",

    "Details":
        f"{len(final_scc_evidence)} normalized themes"

})


quality_rows.append({

    "Quality_Check":
        "Participants",

    "Result":
        n_participants,

    "Details":
        "Number of participants used for prevalence"

})


quality_rows.append({

    "Quality_Check":
        "Normalized SCCE themes",

    "Result":
        len(final_scc_evidence),

    "Details":
        "Themes available for measurement development"

})


quality_rows.append({

    "Quality_Check":
        "Dimension mapping",

    "Result":
        len(dimension_summary),

    "Details":
        "SCCE dimensions generated"

})


quality_rows.append({

    "Quality_Check":
        "Themes requiring manual review",

    "Result":
        len(unmapped_dimension),

    "Details":
        "Review Required dimension"

})


quality_checks_new = pd.DataFrame(
    quality_rows
)


# ================================================================
# 23. CREATE NORMALIZATION SUMMARY
# ================================================================

if normalization_sheet is not None:

    normalization_summary = sheets[
        normalization_sheet
    ].copy()

else:

    normalization_summary = pd.DataFrame({

        "Status": [
            "Normalization summary was not available."
        ]

    })


# ================================================================
# 24. CREATE DECISION SUMMARY
# ================================================================

if decision_sheet is not None:

    decision_summary = sheets[
        decision_sheet
    ].copy()

else:

    decision_summary = pd.DataFrame({

        "Status": [
            "Decision summary was not available."
        ]

    })


# ================================================================
# 25. PARTICIPANT COVERAGE
# ================================================================

if coverage_sheet is not None:

    participant_coverage = sheets[
        coverage_sheet
    ].copy()

else:

    participant_coverage = pd.DataFrame({

        "Status": [
            "Participant coverage sheet was not available."
        ]

    })


# ================================================================
# 26. ORIGINAL / RAW / CLEANED SHEETS
# ================================================================

def get_or_empty(sheet_name):

    if sheet_name is not None:

        return sheets[
            sheet_name
        ].copy()

    return pd.DataFrame()


original_responses = get_or_empty(
    find_sheet("01_Original_Responses")
)

raw_themes = get_or_empty(
    raw_sheet
)

cleaned_themes = get_or_empty(
    cleaned_sheet
)


# ================================================================
# 27. WRITE OUTPUT
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"SCCE_FINAL_EVIDENCE_AND_DIMENSIONS_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    # ------------------------------------------------------------
    # 01
    # ------------------------------------------------------------

    original_responses.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    # ------------------------------------------------------------
    # 02
    # ------------------------------------------------------------

    raw_themes.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 03
    # ------------------------------------------------------------

    cleaned_themes.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # 04
    # ------------------------------------------------------------

    if coding_audit is not None:

        coding_audit.to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="04_Coding_Audit",
            index=False
        )


    # ------------------------------------------------------------
    # 05
    # ------------------------------------------------------------

    if participant_matrix is not None:

        participant_matrix.to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )

    else:

        pd.DataFrame().to_excel(
            writer,
            sheet_name="05_Participant_Matrix",
            index=False
        )


    # ------------------------------------------------------------
    # 06
    # ------------------------------------------------------------

    theme_summary_existing = get_or_empty(
        theme_sheet
    )

    theme_summary_existing.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 07
    # ------------------------------------------------------------

    normalization_summary.to_excel(
        writer,
        sheet_name="07_Normalization_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 08
    # ------------------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 09
    # ------------------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------------------
    # 10
    # ------------------------------------------------------------

    unmapped_check.to_excel(
        writer,
        sheet_name="10_Unmapped_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 11
    # ------------------------------------------------------------

    decision_check.to_excel(
        writer,
        sheet_name="11_Decision_Check",
        index=False
    )


    # ------------------------------------------------------------
    # 12
    # ------------------------------------------------------------

    quality_checks_new.to_excel(
        writer,
        sheet_name="12_Quality_Checks",
        index=False
    )


    # ------------------------------------------------------------
    # 13
    # ------------------------------------------------------------

    final_scc_evidence.to_excel(
        writer,
        sheet_name="13_Final_SCCE_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # 14
    # ------------------------------------------------------------

    dimension_summary.to_excel(
        writer,
        sheet_name="14_SCCE_Dimension_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # 15
    # ------------------------------------------------------------

    dimension_themes.to_excel(
        writer,
        sheet_name="15_SCCE_Dimension_Themes",
        index=False
    )


# ================================================================
# 28. FINAL REPORT
# ================================================================

print("\n")
print("=" * 80)
print("SCCE PROCESS COMPLETED")
print("=" * 80)

print(
    "\nOutput file:"
)

print(
    OUTPUT_FILE.resolve()
)


print("\n")
print("-" * 80)
print("SCCE FINAL EVIDENCE")
print("-" * 80)

print(
    final_scc_evidence.to_string(
        index=False
    )
)


print("\n")
print("-" * 80)
print("SCCE DIMENSION SUMMARY")
print("-" * 80)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FILES / SHEETS CREATED")
print("=" * 80)

print("""
01_Original_Responses
02_Raw_Themes
03_Cleaned_Themes
04_Coding_Audit
05_Participant_Matrix
06_Theme_Summary
07_Normalization_Summary
08_Decision_Summary
09_Participant_Coverage
10_Unmapped_Check
11_Decision_Check
12_Quality_Checks
13_Final_SCCE_Evidence
14_SCCE_Dimension_Summary
15_SCCE_Dimension_Themes
""")


print("=" * 80)
print("IMPORTANT")
print("=" * 80)

print("""
The main new output for questionnaire development is:

14_SCCE_Dimension_Summary

and the detailed supporting table is:

15_SCCE_Dimension_Themes

These should be used for the next stage:
developing candidate SCCE questionnaire items from the
expert-derived themes.
""")

SCCE FINAL QUALITATIVE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\SCCE_FINAL_QUALITATIVE_ANALYSIS_20260827_105058.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Normalization_Summary
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Construct_Statistics
 - 11_Final_SCCE_Evidence
 - 12_Quality_Checks
 - 13_Unmapped_Themes

Detected sheets:
Raw themes: 02_Raw_Themes
Coding audit: 04_Coding_Audit
Theme summary: 06_Theme_Summary
Final SCCE evidence: 11_Final_SCCE_Evidence

Final SCCE evidence columns:
['Final_SCCE_Theme', 'Number_of_Raw_Themes', 'Experts_Mentioning', 'Expert_Prevalence_%', 'Raw_Themes_Kept', 'Raw_Themes_Merged', 'Prevalence_Category']

Number of participants: 26


SCCE PROCESS COMPLETED

Output file:
C:\Users\1886199\OneDrive - UET\Drive G

In [11]:
# ================================================================
# RISK VELOCITY QUALITATIVE CODING
# ================================================================
#
# INPUT:
# Risk_Velocity_Main_Themes_P01-P26.xlsx
#
# STRUCTURE OF INPUT:
# Sheet "1"  = P01
# Sheet "2"  = P02
# ...
# Sheet "26" = P26
#
# Each participant sheet contains:
# Construct = Risk Velocity
# Main themes = 5 candidate themes
#
# FINAL TARGET DOMAINS:
# 1. Rapid Disruption Emergence
# 2. Rapid Disruption Escalation
# 3. Rapid Disruption Propagation
# 4. Limited Response Window
# 5. Rapid Operational Consequences / Short Time-to-Impact
#
# OUTPUT:
# Complete Risk Velocity coding workbook
# ================================================================


import pandas as pd
import re
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook


# ================================================================
# 1. INPUT FILE
# ================================================================

input_file = Path(
    "Risk_Velocity_Main_Themes_P01-P26.xlsx"
)

if not input_file.exists():

    raise FileNotFoundError(
        f"""
Input file was not found.

Expected:
{input_file.resolve()}

Current working directory:
{Path.cwd()}
"""
    )


print("=" * 75)
print("RISK VELOCITY QUALITATIVE CODING")
print("=" * 75)

print("\nInput file:")
print(input_file.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

excel = pd.ExcelFile(input_file)

print("\nAvailable sheets:")

for sheet in excel.sheet_names:
    print("-", sheet)


# ================================================================
# 3. FIND PARTICIPANT SHEETS
# ================================================================

participant_sheets = []

for sheet in excel.sheet_names:

    sheet_text = str(sheet).strip()

    # Accept 1, 2, ..., 26
    # Also accepts P01, P02, etc. if structure changes later.

    match = re.fullmatch(
        r"P?\s*0*(\d+)",
        sheet_text,
        flags=re.IGNORECASE
    )

    if match:

        number = int(
            match.group(1)
        )

        if 1 <= number <= 26:

            participant_sheets.append(
                (number, sheet)
            )


participant_sheets = sorted(
    participant_sheets,
    key=lambda x: x[0]
)


if len(participant_sheets) != 26:

    print(
        f"""
WARNING:
Expected 26 participant sheets,
but found {len(participant_sheets)}.
"""
    )


print("\nParticipant sheets identified:")

for number, sheet in participant_sheets:

    print(
        f"P{number:02d} -> Sheet '{sheet}'"
    )


# ================================================================
# 4. EXTRACT THE 26 RISK VELOCITY RESPONSES
# ================================================================

original_rows = []


for number, sheet in participant_sheets:

    participant = f"P{number:02d}"

    df = pd.read_excel(
        input_file,
        sheet_name=sheet,
        header=None
    )

    # ------------------------------------------------------------
    # Find the row containing Risk Velocity
    # ------------------------------------------------------------

    found = False

    for row_number, row in df.iterrows():

        values = []

        for value in row.tolist():

            if pd.isna(value):
                continue

            text = str(value).strip()

            if text != "":
                values.append(text)

        if not values:
            continue

        # Find "Risk Velocity"
        rv_position = None

        for position, value in enumerate(values):

            if value.strip().lower() == "risk velocity":

                rv_position = position
                break

        if rv_position is None:
            continue

        # --------------------------------------------------------
        # Everything after "Risk Velocity"
        # --------------------------------------------------------

        remaining = values[
            rv_position + 1:
        ]

        if not remaining:
            continue

        main_themes = " ".join(
            remaining
        ).strip()

        if main_themes == "":
            continue

        original_rows.append({

            "Participant":
                participant,

            "Construct":
                "Risk Velocity",

            "Original_Main_Themes":
                main_themes,

            "Source_Sheet":
                str(sheet),

            "Source_Row":
                row_number + 1
        })

        found = True
        break


# ================================================================
# 5. CHECK EXTRACTION
# ================================================================

original_df = pd.DataFrame(
    original_rows
)


if original_df.empty:

    raise ValueError(
        """
No Risk Velocity data were found.

Check that the participant sheets contain:
Construct | Risk Velocity | Main themes
"""
    )


print("\n")
print("=" * 75)
print("EXTRACTION CHECK")
print("=" * 75)

print(
    "Participants extracted:",
    original_df["Participant"].nunique()
)

print(
    "Risk Velocity records:",
    len(original_df)
)


missing_participants = sorted(
    set(
        f"P{i:02d}"
        for i in range(1, 27)
    )
    -
    set(
        original_df["Participant"]
    )
)

if missing_participants:

    print(
        "\nWARNING — missing participants:"
    )

    print(
        missing_participants
    )

else:

    print(
        "\nAll 26 participants successfully extracted."
    )


# ================================================================
# 6. SPLIT MAIN THEMES INTO INDIVIDUAL RAW THEMES
# ================================================================

raw_rows = []


for _, row in original_df.iterrows():

    participant = row[
        "Participant"
    ]

    response = str(
        row["Original_Main_Themes"]
    )

    # Normalize separators

    response = response.replace(
        "\n",
        ";"
    )

    response = response.replace(
        "•",
        ";"
    )

    response = response.replace(
        "|",
        ";"
    )

    # Split semicolon-separated themes

    themes = response.split(";")

    for theme in themes:

        theme = str(
            theme
        ).strip()

        if not theme:
            continue

        raw_rows.append({

            "Participant":
                participant,

            "Construct":
                "Risk Velocity",

            "Raw_Theme":
                theme,

            "Source_Sheet":
                row["Source_Sheet"],

            "Source_Row":
                row["Source_Row"]
        })


raw_df = pd.DataFrame(
    raw_rows
)


# ================================================================
# 7. CLEAN RAW THEMES
# ================================================================

clean_df = raw_df.copy()


clean_df["Theme_Clean"] = (
    clean_df["Raw_Theme"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
)


clean_df["Theme_Key"] = (
    clean_df["Theme_Clean"]
    .str.lower()
    .str.strip()
)


clean_df = clean_df[
    clean_df["Theme_Key"] != ""
]


clean_df = clean_df.drop_duplicates(
    subset=[
        "Participant",
        "Construct",
        "Theme_Key"
    ]
)


clean_df = clean_df.reset_index(
    drop=True
)


print(
    "\nRaw theme observations:",
    len(raw_df)
)

print(
    "Unique cleaned participant-theme observations:",
    len(clean_df)
)


# ================================================================
# 8. SHOW ALL RAW THEMES
# ================================================================

raw_theme_list = (
    clean_df[
        [
            "Raw_Theme",
            "Theme_Clean",
            "Theme_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Theme_Key"
    )
    .reset_index(
        drop=True
    )
)


print("\n")
print("=" * 75)
print("ALL UNIQUE RISK VELOCITY RAW THEMES")
print("=" * 75)

print(
    raw_theme_list[
        [
            "Raw_Theme"
        ]
    ].to_string(
        index=False
    )
)


# ================================================================
# 9. RISK VELOCITY NORMALIZATION DICTIONARY
# ================================================================
#
# These mappings align the generated candidate themes with the
# five intended Risk Velocity content domains.
#
# IMPORTANT:
# These are conceptual normalization decisions.
# They should be reported as researcher-led coding/normalization,
# not as statistical results.
#
# ================================================================


RV_NORMALIZATION = {

    # ------------------------------------------------------------
    # DOMAIN 1
    # RAPID DISRUPTION EMERGENCE
    # ------------------------------------------------------------

    "rapid disruption emergence":
        "Rapid Disruption Emergence",

    "speed of risk onset":
        "Rapid Disruption Emergence",

    "immediate development of disruptions":
        "Rapid Disruption Emergence",

    "rapid onset of supply-chain risks":
        "Rapid Disruption Emergence",

    "disruption develops quickly":
        "Rapid Disruption Emergence",

    "high speed of disruption development":
        "Rapid Disruption Emergence",

    "rapid risk manifestation":
        "Rapid Disruption Emergence",

    "immediate disruption effects":
        "Rapid Disruption Emergence",

    "rapid emergence and development":
        "Rapid Disruption Emergence",

    "fast disruption onset":
        "Rapid Disruption Emergence",

    "rapid emergence of disruption":
        "Rapid Disruption Emergence",

    "rapid emergence":
        "Rapid Disruption Emergence",

    "disruptions emerge with little delay":
        "Rapid Disruption Emergence",

    "rapid development of risk events":
        "Rapid Disruption Emergence",

    "fast emergence of disruption":
        "Rapid Disruption Emergence",

    "quickly developing supply-chain risks":
        "Rapid Disruption Emergence",

    "rapid disruption development":
        "Rapid Disruption Emergence",

    "speed of disruption onset":
        "Rapid Disruption Emergence",

    "rapidly emerging supply-chain disruptions":
        "Rapid Disruption Emergence",

    "short onset-to-impact period":
        "Rapid Disruption Emergence",

    "rapid onset after a disruption trigger":
        "Rapid Disruption Emergence",

    "fast-moving disruptions":
        "Rapid Disruption Emergence",

    "short time from disruption to impact":
        "Rapid Disruption Emergence",

    "rapid risk event development":
        "Rapid Disruption Emergence",

    # ------------------------------------------------------------
    # DOMAIN 2
    # RAPID DISRUPTION ESCALATION
    # ------------------------------------------------------------

    "fast escalation":
        "Rapid Disruption Escalation",

    "rapid development of disruption effects":
        "Rapid Disruption Escalation",

    "escalating disruption severity":
        "Rapid Disruption Escalation",

    "rapid escalation":
        "Rapid Disruption Escalation",

    "increasing disruption severity":
        "Rapid Disruption Escalation",

    "accelerating consequences":
        "Rapid Disruption Escalation",

    "quick escalation":
        "Rapid Disruption Escalation",

    "rapid worsening of disruption":
        "Rapid Disruption Escalation",

    "quick escalation of consequences":
        "Rapid Disruption Escalation",

    "rapid escalation of effects":
        "Rapid Disruption Escalation",

    "rapidly increasing effects":
        "Rapid Disruption Escalation",

    "rapid escalation and intensification":
        "Rapid Disruption Escalation",

    "fast escalation of effects":
        "Rapid Disruption Escalation",

    "rapidly increasing disruption intensity":
        "Rapid Disruption Escalation",

    "increasing disruption intensity":
        "Rapid Disruption Escalation",

    "rapidly worsening effects":
        "Rapid Disruption Escalation",

    "rapid escalation of disruption severity":
        "Rapid Disruption Escalation",

    "fast-growing effects":
        "Rapid Disruption Escalation",

    "rapid growth in consequences":
        "Rapid Disruption Escalation",

    "increasing severity over a short period":
        "Rapid Disruption Escalation",

    "fast escalation of effects":
        "Rapid Disruption Escalation",

    "rapidly increasing consequences":
        "Rapid Disruption Escalation",

    # ------------------------------------------------------------
    # DOMAIN 3
    # RAPID DISRUPTION PROPAGATION
    # ------------------------------------------------------------

    "rapid spread across supply-chain partners":
        "Rapid Disruption Propagation",

    "cascading supply-chain effects":
        "Rapid Disruption Propagation",

    "propagation across supply-chain stages":
        "Rapid Disruption Propagation",

    "interorganizational disruption propagation":
        "Rapid Disruption Propagation",

    "cross-tier disruption spread":
        "Rapid Disruption Propagation",

    "rapid knock-on effects":
        "Rapid Disruption Propagation",

    "spread across connected partners":
        "Rapid Disruption Propagation",

    "propagation through interconnected actors":
        "Rapid Disruption Propagation",

    "fast transmission between supply-chain entities":
        "Rapid Disruption Propagation",

    "rapid network propagation":
        "Rapid Disruption Propagation",

    "cascading effects across the chain":
        "Rapid Disruption Propagation",

    "fast spread through the supply chain":
        "Rapid Disruption Propagation",

    "cross-organizational propagation":
        "Rapid Disruption Propagation",

    "rapid movement of disruption across partners":
        "Rapid Disruption Propagation",

    "supply-chain-wide spread":
        "Rapid Disruption Propagation",

    "fast propagation across supply-chain links":
        "Rapid Disruption Propagation",

    "rapid cascading effects":
        "Rapid Disruption Propagation",

    "rapid cross-partner effects":
        "Rapid Disruption Propagation",

    "knock-on effects across supply-chain partners":
        "Rapid Disruption Propagation",

    "fast spread through connected supply-chain actors":
        "Rapid Disruption Propagation",

    "fast propagation through the network":
        "Rapid Disruption Propagation",

    "rapid propagation":
        "Rapid Disruption Propagation",

    "quick propagation across supply-chain stages":
        "Rapid Disruption Propagation",

    "fast spread across interconnected supply-chain entities":
        "Rapid Disruption Propagation",

    "rapid propagation across supply-chain partners":
        "Rapid Disruption Propagation",

    # ------------------------------------------------------------
    # DOMAIN 4
    # LIMITED RESPONSE WINDOW
    # ------------------------------------------------------------

    "short response window":
        "Limited Response Window",

    "limited reaction time":
        "Limited Response Window",

    "urgent response requirement":
        "Limited Response Window",

    "narrow response window":
        "Limited Response Window",

    "time pressure for response":
        "Limited Response Window",

    "compressed response period":
        "Limited Response Window",

    "limited time for intervention":
        "Limited Response Window",

    "urgent response timing":
        "Limited Response Window",

    "restricted time for response":
        "Limited Response Window",

    "limited response opportunity":
        "Limited Response Window",

    "compressed reaction time":
        "Limited Response Window",

    "urgent response window":
        "Limited Response Window",

    "little time for response":
        "Limited Response Window",

    "limited time for corrective action":
        "Limited Response Window",

    "short reaction period":
        "Limited Response Window",

    "limited response period":
        "Limited Response Window",

    "narrow decision-response window":
        "Limited Response Window",

    "short decision-response window":
        "Limited Response Window",

    "reduced response time":
        "Limited Response Window",

    "little time to react":
        "Limited Response Window",

    "limited time to respond":
        "Limited Response Window",

    "time-critical response":
        "Limited Response Window",

    "compressed response time":
        "Limited Response Window",

    "limited time to corrective action":
        "Limited Response Window",

    "limited time for response":
        "Limited Response Window",

    # ------------------------------------------------------------
    # DOMAIN 5
    # RAPID OPERATIONAL CONSEQUENCES /
    # SHORT TIME-TO-IMPACT
    # ------------------------------------------------------------

    "quick operational impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "fast impact on operations":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "early operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "quick operational disruption":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "immediate operational effects":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to material impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid business impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "quick operational losses":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time before operational effects":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid operational impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to serious impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "fast operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to significant impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid material consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "quick impact on operations":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to significant operational impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to significant operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid operational consequences":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "short time to operational impact":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "fast consequences for operations":
        "Rapid Operational Consequences / Short Time-to-Impact",

    "rapid operational disruption":
        "Rapid Operational Consequences / Short Time-to-Impact",
}


# ================================================================
# 10. APPLY NORMALIZATION
# ================================================================

coding_df = raw_theme_list.copy()


coding_df["Normalized_Theme"] = ""


coding_df["Decision"] = ""


coding_df["Reason"] = ""


for i, row in coding_df.iterrows():

    key = str(
        row["Theme_Key"]
    ).strip().lower()

    if key in RV_NORMALIZATION:

        normalized = RV_NORMALIZATION[
            key
        ]

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = normalized


        if key == normalized.lower():

            coding_df.loc[
                i,
                "Decision"
            ] = "Keep"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Retained as the conceptual domain."
            )

        else:

            coding_df.loc[
                i,
                "Decision"
            ] = "Merge"

            coding_df.loc[
                i,
                "Reason"
            ] = (
                "Conceptually aligned with the "
                "same Risk Velocity content domain."
            )

    else:

        # --------------------------------------------------------
        # IMPORTANT:
        # Do NOT silently force an unknown theme into a domain.
        # --------------------------------------------------------

        coding_df.loc[
            i,
            "Normalized_Theme"
        ] = row[
            "Theme_Clean"
        ]

        coding_df.loc[
            i,
            "Decision"
        ] = "Review"

        coding_df.loc[
            i,
            "Reason"
        ] = (
            "No explicit normalization mapping found; "
            "requires researcher review."
        )


# ================================================================
# 11. IDENTIFY UNMAPPED THEMES
# ================================================================

unmapped_df = coding_df[
    coding_df["Decision"] == "Review"
].copy()


print("\n")
print("=" * 75)
print("NORMALIZATION CHECK")
print("=" * 75)

print(
    "Total raw themes:",
    len(coding_df)
)

print(
    "Mapped themes:",
    len(
        coding_df[
            coding_df["Decision"].isin(
                ["Keep", "Merge"]
            )
        ]
    )
)

print(
    "Themes requiring review:",
    len(unmapped_df)
)


if not unmapped_df.empty:

    print(
        "\nUNMAPPED / REVIEW THEMES:"
    )

    print(
        unmapped_df[
            [
                "Raw_Theme",
                "Theme_Clean"
            ]
        ].to_string(
            index=False
        )
    )


# ================================================================
# 12. MAP NORMALIZATION BACK TO ALL PARTICIPANTS
# ================================================================

mapping = coding_df[
    [
        "Theme_Key",
        "Normalized_Theme",
        "Decision"
    ]
]


coded_df = clean_df.merge(
    mapping,
    on="Theme_Key",
    how="left"
)


# ================================================================
# 13. PARTICIPANT × NORMALIZED RV DOMAIN MATRIX
# ================================================================

participants = [
    f"P{i:02d}"
    for i in range(1, 27)
]


matrix_source = (
    coded_df
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
)


matrix = pd.crosstab(
    matrix_source[
        "Normalized_Theme"
    ],
    matrix_source[
        "Participant"
    ]
)


matrix = matrix.reindex(
    columns=participants,
    fill_value=0
)


# Reorder rows according to the intended five domains.

target_domains = [

    "Rapid Disruption Emergence",

    "Rapid Disruption Escalation",

    "Rapid Disruption Propagation",

    "Limited Response Window",

    "Rapid Operational Consequences / Short Time-to-Impact"
]


existing_domains = [
    x for x in target_domains
    if x in matrix.index
]


other_domains = [
    x for x in matrix.index
    if x not in target_domains
]


matrix = matrix.reindex(
    existing_domains + other_domains,
    fill_value=0
)


matrix = matrix.reset_index()


matrix["Experts_Mentioning"] = matrix[
    participants
].sum(
    axis=1
)


matrix["Expert_Prevalence_%"] = (
    matrix["Experts_Mentioning"]
    / len(participants)
    * 100
).round(1)


# ================================================================
# 14. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(value):

    if value >= 75:
        return "Very High"

    elif value >= 50:
        return "High"

    elif value >= 25:
        return "Moderate"

    else:
        return "Low"


matrix[
    "Prevalence_Category"
] = matrix[
    "Expert_Prevalence_%"
].apply(
    prevalence_category
)


# ================================================================
# 15. RANK DOMAINS
# ================================================================

matrix = matrix.sort_values(
    "Experts_Mentioning",
    ascending=False
).reset_index(
    drop=True
)


matrix.insert(
    0,
    "Rank",
    range(
        1,
        len(matrix) + 1
    )
)


# ================================================================
# 16. FINAL RV THEME SUMMARY
# ================================================================

theme_summary = matrix[
    [
        "Rank",
        "Normalized_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
].copy()


theme_summary.columns = [

    "Rank",

    "Risk_Velocity_Domain",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"
]


# ================================================================
# 17. PARTICIPANT COVERAGE
# ================================================================

participant_coverage = (
    coded_df
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
    .reset_index()
)


participant_coverage.columns = [

    "Participant",

    "RV_Raw_Theme_Count"
]


# Add number of final domains represented.

domain_coverage = (
    coded_df[
        coded_df[
            "Normalized_Theme"
        ].isin(
            target_domains
        )
    ]
    .drop_duplicates(
        subset=[
            "Participant",
            "Normalized_Theme"
        ]
    )
    .groupby(
        "Participant"
    )
    .size()
    .reindex(
        participants,
        fill_value=0
    )
)


participant_coverage[
    "RV_Final_Domain_Count"
] = (
    domain_coverage
    .values
)


# ================================================================
# 18. DECISION SUMMARY
# ================================================================

decision_summary = (
    coding_df
    .groupby(
        "Decision"
    )
    .size()
    .reset_index(
        name="Count"
    )
)


# ================================================================
# 19. DOMAIN → RAW THEMES
# ================================================================

domain_raw_theme_summary = (
    coding_df[
        coding_df[
            "Normalized_Theme"
        ].isin(
            target_domains
        )
    ]
    .groupby(
        "Normalized_Theme"
    )[
        "Theme_Clean"
    ]
    .apply(
        lambda x:
        "; ".join(
            sorted(
                set(x)
            )
        )
    )
    .reset_index()
)


domain_raw_theme_summary.columns = [

    "Risk_Velocity_Domain",

    "Included_Raw_Themes"
]


# ================================================================
# 20. DOMAIN EVIDENCE TABLE
# ================================================================

domain_evidence = pd.DataFrame({

    "Risk_Velocity_Domain": [

        "Rapid Disruption Emergence",

        "Rapid Disruption Escalation",

        "Rapid Disruption Propagation",

        "Limited Response Window",

        "Rapid Operational Consequences / Short Time-to-Impact"
    ],

    "Conceptual_Content": [

        "How quickly a supply-chain disruption develops after it first emerges.",

        "How quickly the effects or severity of a disruption increase.",

        "How quickly disruption effects spread across supply-chain partners, stages, or connected entities.",

        "How much time the organization has available to react after a disruption emerges.",

        "How quickly a disruption produces significant operational consequences."
    ],

    "Target_Item": [

        "RV1",

        "RV2",

        "RV3",

        "RV4",

        "RV5"
    ],

    "Candidate_Questionnaire_Item": [

        "Supply chain disruptions can develop rapidly after they first emerge.",

        "Disruptions can escalate quickly and affect our supply chain operations.",

        "The effects of supply chain disruptions can spread rapidly across our supply chain.",

        "Our organization often has a limited time window to respond after a disruption emerges.",

        "Supply chain disruptions can cause significant operational consequences within a short period."
    ]

})


# ================================================================
# 21. PARTICIPANT-LEVEL FINAL DOMAIN TABLE
# ================================================================

participant_domain_records = []


for participant in participants:

    participant_data = coded_df[
        coded_df[
            "Participant"
        ] == participant
    ]


    domains = sorted(
        set(
            participant_data[
                "Normalized_Theme"
            ]
            .dropna()
            .tolist()
        )
        &
        set(target_domains)
    )


    participant_domain_records.append({

        "Participant":
            participant,

        "Final_RV_Domains":
            "; ".join(domains),

        "Number_of_Final_Domains":
            len(domains)

    })


participant_domain_df = pd.DataFrame(
    participant_domain_records
)


# ================================================================
# 22. FINAL EVIDENCE TABLE
# ================================================================

final_evidence = matrix[
    matrix[
        "Normalized_Theme"
    ].isin(
        target_domains
    )
].copy()


final_evidence = final_evidence[
    [
        "Rank",
        "Normalized_Theme",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
]


final_evidence.columns = [

    "Rank",

    "Risk_Velocity_Domain",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category"
]


# Add target item.

item_map = {

    "Rapid Disruption Emergence":
        "RV1",

    "Rapid Disruption Escalation":
        "RV2",

    "Rapid Disruption Propagation":
        "RV3",

    "Limited Response Window":
        "RV4",

    "Rapid Operational Consequences / Short Time-to-Impact":
        "RV5"
}


final_evidence[
    "Target_Item"
] = final_evidence[
    "Risk_Velocity_Domain"
].map(
    item_map
)


# ================================================================
# 23. SAVE OUTPUT
# ================================================================

output_file = Path(

    "Risk_Velocity_Coding_"
    +
    datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    +
    ".xlsx"

)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    # ------------------------------------------------------------
    # Original extracted information
    # ------------------------------------------------------------

    original_df.to_excel(
        writer,
        sheet_name="01_Original_Responses",
        index=False
    )


    # ------------------------------------------------------------
    # Raw themes
    # ------------------------------------------------------------

    raw_df.to_excel(
        writer,
        sheet_name="02_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # Cleaned themes
    # ------------------------------------------------------------

    clean_df.to_excel(
        writer,
        sheet_name="03_Cleaned_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # Coding dictionary
    # ------------------------------------------------------------

    coding_df.to_excel(
        writer,
        sheet_name="04_Coding_Dictionary",
        index=False
    )


    # ------------------------------------------------------------
    # Participant × domain matrix
    # ------------------------------------------------------------

    matrix.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )


    # ------------------------------------------------------------
    # Theme summary
    # ------------------------------------------------------------

    theme_summary.to_excel(
        writer,
        sheet_name="06_Theme_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # Decision summary
    # ------------------------------------------------------------

    decision_summary.to_excel(
        writer,
        sheet_name="07_Decision_Summary",
        index=False
    )


    # ------------------------------------------------------------
    # Participant coverage
    # ------------------------------------------------------------

    participant_coverage.to_excel(
        writer,
        sheet_name="08_Participant_Coverage",
        index=False
    )


    # ------------------------------------------------------------
    # Raw themes included in each domain
    # ------------------------------------------------------------

    domain_raw_theme_summary.to_excel(
        writer,
        sheet_name="09_Domain_Raw_Themes",
        index=False
    )


    # ------------------------------------------------------------
    # Domain → item evidence
    # ------------------------------------------------------------

    domain_evidence.to_excel(
        writer,
        sheet_name="10_Domain_Item_Map",
        index=False
    )


    # ------------------------------------------------------------
    # Participant-level domain table
    # ------------------------------------------------------------

    participant_domain_df.to_excel(
        writer,
        sheet_name="11_Participant_Domains",
        index=False
    )


    # ------------------------------------------------------------
    # Final five-domain evidence
    # ------------------------------------------------------------

    final_evidence.to_excel(
        writer,
        sheet_name="12_Final_RV_Evidence",
        index=False
    )


    # ------------------------------------------------------------
    # Unmapped themes
    # ------------------------------------------------------------

    unmapped_df.to_excel(
        writer,
        sheet_name="13_Unmapped_Themes",
        index=False
    )


# ================================================================
# 24. BASIC EXCEL FORMATTING
# ================================================================

try:

    wb = load_workbook(
        output_file
    )


    for ws in wb.worksheets:

        # Freeze top row

        ws.freeze_panes = "A2"

        # Header formatting

        for cell in ws[1]:

            cell.font = cell.font.copy(
                bold=True
            )

            cell.alignment = cell.alignment.copy(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )


        # Auto width

        for column_cells in ws.columns:

            max_length = 0

            column_letter = (
                column_cells[0].column_letter
            )

            for cell in column_cells:

                try:

                    value_length = len(
                        str(
                            cell.value
                        )
                    )

                    if value_length > max_length:

                        max_length = value_length

                except:

                    pass


            ws.column_dimensions[
                column_letter
            ].width = min(
                max(
                    max_length + 2,
                    12
                ),
                70
            )


    wb.save(
        output_file
    )


except Exception as formatting_error:

    print(
        "\nExcel formatting warning:"
    )

    print(
        formatting_error
    )


# ================================================================
# 25. FINAL REPORT
# ================================================================

print("\n")
print("=" * 75)
print("RISK VELOCITY CODING COMPLETED")
print("=" * 75)


print(
    "\nParticipants:",
    original_df[
        "Participant"
    ].nunique()
)


print(
    "Raw theme observations:",
    len(raw_df)
)


print(
    "Unique raw themes:",
    coding_df[
        "Theme_Key"
    ].nunique()
)


print(
    "Final target domains:",
    len(
        final_evidence
    )
)


print("\n")
print("FINAL RISK VELOCITY DOMAINS")
print("-" * 75)


print(
    final_evidence[
        [
            "Rank",
            "Risk_Velocity_Domain",
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Target_Item"
        ]
    ].to_string(
        index=False
    )
)


print("\n")
print("=" * 75)
print("TARGET QUESTIONNAIRE ITEMS")
print("=" * 75)


for _, row in domain_evidence.iterrows():

    print(
        f"\n{row['Target_Item']}"
    )

    print(
        row["Candidate_Questionnaire_Item"]
    )


print("\n")
print("=" * 75)
print("OUTPUT FILE")
print("=" * 75)


print(
    output_file.resolve()
)


print("\nDONE.")

RISK VELOCITY QUALITATIVE CODING

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Risk_Velocity_Main_Themes_P01-P26.xlsx

Available sheets:
- 1
- 2
- 3
- 4
- 5
- 6
- 7
- 8
- 9
- 10
- 11
- 12
- 13
- 14
- 15
- 16
- 17
- 18
- 19
- 20
- 21
- 22
- 23
- 24
- 25
- 26

Participant sheets identified:
P01 -> Sheet '1'
P02 -> Sheet '2'
P03 -> Sheet '3'
P04 -> Sheet '4'
P05 -> Sheet '5'
P06 -> Sheet '6'
P07 -> Sheet '7'
P08 -> Sheet '8'
P09 -> Sheet '9'
P10 -> Sheet '10'
P11 -> Sheet '11'
P12 -> Sheet '12'
P13 -> Sheet '13'
P14 -> Sheet '14'
P15 -> Sheet '15'
P16 -> Sheet '16'
P17 -> Sheet '17'
P18 -> Sheet '18'
P19 -> Sheet '19'
P20 -> Sheet '20'
P21 -> Sheet '21'
P22 -> Sheet '22'
P23 -> Sheet '23'
P24 -> Sheet '24'
P25 -> Sheet '25'
P26 -> Sheet '26'


EXTRACTION CHECK
Participants extracted: 26
Risk Velocity records: 26

All 26 participants successfully extracted.

Raw theme observations:

C:\Users\1886199\AppData\Local\Temp\ipykernel_30656\2101301097.py:1672: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.font = cell.font.copy(
C:\Users\1886199\AppData\Local\Temp\ipykernel_30656\2101301097.py:1676: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(


In [13]:
# ================================================================
# FINAL RISK VELOCITY EVIDENCE + ITEM DEVELOPMENT ANALYSIS
# CORRECTED / RERUN-SAFE VERSION
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# ================================================================
# 1. INPUT FILE
# ================================================================

INPUT_FILE = Path(
    "Risk_Velocity_Coding_20260827_124722.xlsx"
)

if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"""
INPUT FILE NOT FOUND

Expected:
{INPUT_FILE.resolve()}

Current working directory:
{Path.cwd()}
"""
    )


print("=" * 80)
print("FINAL RISK VELOCITY EVIDENCE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(INPUT_FILE)

print("\nAvailable sheets:")

for sheet in xls.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FINAL RISK VELOCITY DOMAINS
# ================================================================

FINAL_DOMAINS = [

    "Rapid Disruption Emergence",

    "Rapid Disruption Escalation",

    "Rapid Disruption Propagation",

    "Limited Response Window",

    "Rapid Operational Consequences / Short Time-to-Impact"
]


# ================================================================
# 4. FINAL QUESTIONNAIRE ITEMS
# ================================================================

ITEMS = {

    "Rapid Disruption Emergence":
        (
            "RV1",
            "Supply chain disruptions can develop rapidly "
            "after they first emerge."
        ),

    "Rapid Disruption Escalation":
        (
            "RV2",
            "Disruptions can escalate quickly and affect "
            "our supply chain operations."
        ),

    "Rapid Disruption Propagation":
        (
            "RV3",
            "The effects of supply chain disruptions can "
            "spread rapidly across our supply chain."
        ),

    "Limited Response Window":
        (
            "RV4",
            "Our organization often has a limited time "
            "window to respond after a disruption emerges."
        ),

    "Rapid Operational Consequences / Short Time-to-Impact":
        (
            "RV5",
            "Supply chain disruptions can cause significant "
            "operational consequences within a short period."
        )
}


# ================================================================
# 5. FIND PARTICIPANT MATRIX SHEET
# ================================================================

matrix_sheet = None

for sheet in xls.sheet_names:

    normalized = (
        str(sheet)
        .strip()
        .lower()
        .replace(" ", "_")
    )

    if normalized in [
        "05_participant_matrix",
        "participant_matrix"
    ]:

        matrix_sheet = sheet
        break


if matrix_sheet is None:

    raise ValueError(
        """
Could not find the Participant Matrix sheet.

Expected:
05_Participant_Matrix
"""
    )


print(
    "\nParticipant matrix:",
    matrix_sheet
)


# ================================================================
# 6. READ PARTICIPANT MATRIX
# ================================================================

matrix_df = pd.read_excel(
    INPUT_FILE,
    sheet_name=matrix_sheet
)


print(
    "\nMatrix shape:",
    matrix_df.shape
)

print(
    "\nMatrix columns:"
)

for c in matrix_df.columns:

    print(
        " -",
        c
    )


# ================================================================
# 7. FIND DOMAIN COLUMN
# ================================================================

domain_column = None

possible_domain_columns = [

    "Normalized_Theme",

    "Risk_Velocity_Domain",

    "Final_RV_Domain",

    "Domain"
]


for c in possible_domain_columns:

    if c in matrix_df.columns:

        domain_column = c
        break


if domain_column is None:

    for c in matrix_df.columns:

        name = str(c).lower()

        if (
            "theme" in name
            or
            "domain" in name
        ):

            domain_column = c
            break


if domain_column is None:

    raise ValueError(
        """
Could not identify the Risk Velocity domain column.
"""
    )


print(
    "\nDomain column:",
    domain_column
)


# ================================================================
# 8. FIND PARTICIPANT COLUMNS
# ================================================================

participant_columns = []

for i in range(1, 27):

    participant = f"P{i:02d}"

    if participant in matrix_df.columns:

        participant_columns.append(
            participant
        )


print(
    "\nParticipants detected:",
    len(participant_columns)
)


if not participant_columns:

    raise ValueError(
        """
No P01–P26 columns were found.
"""
    )


# ================================================================
# 9. KEEP ONLY FINAL FIVE DOMAINS
# ================================================================

domain_df = matrix_df[
    matrix_df[
        domain_column
    ].isin(
        FINAL_DOMAINS
    )
].copy()


# ================================================================
# 10. ENSURE ALL FIVE DOMAINS EXIST
# ================================================================

existing_domains = set(
    domain_df[
        domain_column
    ]
    .astype(str)
)


for domain in FINAL_DOMAINS:

    if domain not in existing_domains:

        new_row = {
            domain_column:
                domain
        }

        for participant in participant_columns:

            new_row[
                participant
            ] = 0

        domain_df = pd.concat(
            [
                domain_df,
                pd.DataFrame(
                    [new_row]
                )
            ],
            ignore_index=True
        )


# ================================================================
# 11. REMOVE DUPLICATE DOMAINS
# ================================================================

domain_df = (
    domain_df
    .drop_duplicates(
        subset=[
            domain_column
        ],
        keep="first"
    )
    .copy()
)


# ================================================================
# 12. FORCE PARTICIPANT VALUES TO NUMERIC
# ================================================================

for participant in participant_columns:

    domain_df[
        participant
    ] = pd.to_numeric(
        domain_df[
            participant
        ],
        errors="coerce"
    ).fillna(0)


# ================================================================
# 13. CALCULATE EXPERT MENTIONING
# ================================================================

domain_df[
    "Experts_Mentioning"
] = domain_df[
    participant_columns
].sum(
    axis=1
)


# ================================================================
# 14. CALCULATE PREVALENCE
# ================================================================

number_of_experts = len(
    participant_columns
)


domain_df[
    "Expert_Prevalence_%"
] = (
    domain_df[
        "Experts_Mentioning"
    ]
    /
    number_of_experts
    *
    100
).round(1)


# ================================================================
# 15. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(
    value
):

    if value >= 75:

        return "Very High"

    elif value >= 50:

        return "High"

    elif value >= 25:

        return "Moderate"

    else:

        return "Low"


domain_df[
    "Prevalence_Category"
] = domain_df[
    "Expert_Prevalence_%"
].apply(
    prevalence_category
)


# ================================================================
# 16. SORT BY PREVALENCE
# ================================================================

ranked_df = (
    domain_df
    .sort_values(
        by=[
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 17. SAFE RANK CREATION
# ================================================================
#
# THIS FIXES YOUR ERROR.
#
# If Rank already exists, overwrite it.
# If Rank does not exist, create it.
#
# ================================================================

ranked_df[
    "Rank"
] = range(
    1,
    len(ranked_df) + 1
)


# Move Rank to first position safely.

columns = list(
    ranked_df.columns
)

columns.remove(
    "Rank"
)

ranked_df = ranked_df[
    ["Rank"] + columns
]


# ================================================================
# 18. FINAL DOMAIN PREVALENCE TABLE
# ================================================================

domain_prevalence = ranked_df[
    [
        "Rank",
        domain_column,
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
].copy()


domain_prevalence = domain_prevalence.rename(
    columns={
        domain_column:
            "Risk_Velocity_Domain"
    }
)


# ================================================================
# 19. DOMAIN → ITEM TABLE
# ================================================================

item_rows = []


for domain in FINAL_DOMAINS:

    item_code, item_text = ITEMS[
        domain
    ]


    match = ranked_df[
        ranked_df[
            domain_column
        ] == domain
    ]


    if len(match) > 0:

        row = match.iloc[0]

        experts = int(
            row[
                "Experts_Mentioning"
            ]
        )

        prevalence = float(
            row[
                "Expert_Prevalence_%"
            ]
        )

        category = row[
            "Prevalence_Category"
        ]

    else:

        experts = 0
        prevalence = 0.0
        category = "Not observed"


    item_rows.append({

        "Item":
            item_code,

        "Risk_Velocity_Domain":
            domain,

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            prevalence,

        "Prevalence_Category":
            category,

        "Candidate_Questionnaire_Item":
            item_text
    })


domain_item_df = pd.DataFrame(
    item_rows
)


# ================================================================
# 20. ADD CONTENT DOMAIN RATIONALE
# ================================================================

rationales = {

    "Rapid Disruption Emergence":
        "Captures how rapidly a disruption develops after it first emerges.",

    "Rapid Disruption Escalation":
        "Captures how quickly disruption effects intensify or escalate.",

    "Rapid Disruption Propagation":
        "Captures how rapidly disruption effects spread across supply-chain entities or stages.",

    "Limited Response Window":
        "Captures the limited time available for organizational response after disruption emergence.",

    "Rapid Operational Consequences / Short Time-to-Impact":
        "Captures how quickly significant operational consequences materialize."
}


domain_item_df[
    "Content_Domain_Rationale"
] = domain_item_df[
    "Risk_Velocity_Domain"
].map(
    rationales
)


domain_item_df[
    "Item_Status"
] = (
    "Candidate item for expert content validation"
)


# ================================================================
# 21. PARTICIPANT × DOMAIN MATRIX
# ================================================================

participant_matrix = ranked_df[
    [
        domain_column
    ]
    +
    participant_columns
].copy()


participant_matrix = participant_matrix.rename(
    columns={
        domain_column:
            "Risk_Velocity_Domain"
    }
)


# ================================================================
# 22. PARTICIPANT COVERAGE
# ================================================================

coverage_rows = []


for participant in participant_columns:

    values = pd.to_numeric(
        participant_matrix[
            participant
        ],
        errors="coerce"
    ).fillna(0)


    domains_mentioned = int(
        (
            values > 0
        ).sum()
    )


    coverage_rows.append({

        "Participant":
            participant,

        "Final_RV_Domains_Mentioned":
            domains_mentioned,

        "Percentage_of_Final_Domains_Covered":
            round(
                domains_mentioned
                /
                len(FINAL_DOMAINS)
                *
                100,
                1
            )
    })


coverage_df = pd.DataFrame(
    coverage_rows
)


# ================================================================
# 23. PARTICIPANT COVERAGE CATEGORY
# ================================================================

coverage_df[
    "Coverage_Category"
] = coverage_df[
    "Percentage_of_Final_Domains_Covered"
].apply(
    prevalence_category
)


# ================================================================
# 24. CREATE FINAL FIVE-ITEM TABLE
# ================================================================

final_evidence = domain_item_df[
    [
        "Item",
        "Risk_Velocity_Domain",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Candidate_Questionnaire_Item"
    ]
].copy()


# ================================================================
# 25. METHODOLOGY NOTE
# ================================================================

methodology_note = pd.DataFrame({

    "Component": [

        "Input",

        "Analytical unit",

        "Final content domains",

        "Item development",

        "Use of Python",

        "Important limitation",

        "Next methodological stage"
    ],

    "Description": [

        "Risk_Velocity_Coding_20260827_124722.xlsx.",

        "Participant-level candidate Risk Velocity themes organized into five content domains.",

        "Rapid Disruption Emergence; Rapid Disruption Escalation; Rapid Disruption Propagation; Limited Response Window; Rapid Operational Consequences / Short Time-to-Impact.",

        "Five preliminary items (RV1–RV5) were mapped to the five content domains.",

        "Python was used to organize, normalize, audit, calculate participant coverage/prevalence, and generate the final evidence tables.",

        "The underlying Risk Velocity themes were generated candidate themes rather than verbatim empirical responses specifically elicited from the 26 experts. Therefore, the resulting prevalence values must not be presented as actual empirical expert prevalence.",

        "Triangulate each item with multiple published sources and conduct dedicated expert content validation before the main PLS-SEM survey."
    ]
})


# ================================================================
# 26. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Input file exists",

        "Participant columns detected",

        "Number of participants",

        "Five final domains present",

        "Five questionnaire items present",

        "Duplicate domain rows",

        "Rank column duplicated"
    ],

    "Result": [

        True,

        len(participant_columns) > 0,

        number_of_experts,

        len(
            ranked_df
        ) == 5,

        len(
            domain_item_df
        ) == 5,

        int(
            ranked_df[
                domain_column
            ].duplicated().sum()
        ),

        ranked_df.columns.tolist().count(
            "Rank"
        )
    ]
})


# ================================================================
# 27. CREATE OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"Risk_Velocity_FINAL_EVIDENCE_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    final_evidence.to_excel(
        writer,
        sheet_name="01_Final_RV_Evidence",
        index=False
    )


    domain_item_df.to_excel(
        writer,
        sheet_name="02_Item_Development",
        index=False
    )


    domain_prevalence.to_excel(
        writer,
        sheet_name="03_Domain_Prevalence",
        index=False
    )


    participant_matrix.to_excel(
        writer,
        sheet_name="04_Participant_Matrix",
        index=False
    )


    coverage_df.to_excel(
        writer,
        sheet_name="05_Participant_Coverage",
        index=False
    )


    methodology_note.to_excel(
        writer,
        sheet_name="06_Methodology_Note",
        index=False
    )


    quality_checks.to_excel(
        writer,
        sheet_name="07_Quality_Checks",
        index=False
    )


# ================================================================
# 28. PROFESSIONAL FORMATTING
# ================================================================

try:

    wb = load_workbook(
        OUTPUT_FILE
    )


    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )


    header_font = Font(
        bold=True,
        color="FFFFFF"
    )


    for ws in wb.worksheets:

        ws.freeze_panes = "A2"


        for cell in ws[1]:

            cell.fill = header_fill

            cell.font = header_font

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )


        for row in ws.iter_rows(
            min_row=2
        ):

            for cell in row:

                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True
                )


        for column_cells in ws.columns:

            column_letter = get_column_letter(
                column_cells[0].column
            )


            max_length = 0


            for cell in column_cells:

                if cell.value is None:

                    continue


                max_length = max(
                    max_length,
                    len(
                        str(
                            cell.value
                        )
                    )
                )


            ws.column_dimensions[
                column_letter
            ].width = min(
                max(
                    max_length + 2,
                    12
                ),
                65
            )


        ws.auto_filter.ref = (
            ws.dimensions
        )


    wb.save(
        OUTPUT_FILE
    )


except Exception as formatting_error:

    print(
        "\nFormatting warning:"
    )

    print(
        formatting_error
    )


# ================================================================
# 29. FINAL CONSOLE OUTPUT
# ================================================================

print("\n")
print("=" * 80)
print("RISK VELOCITY ANALYSIS COMPLETED")
print("=" * 80)


print(
    "\nParticipants:",
    number_of_experts
)


print(
    "Final domains:",
    len(
        FINAL_DOMAINS
    )
)


print(
    "\nFINAL DOMAIN SUMMARY"
)

print(
    domain_prevalence.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FINAL RV ITEMS")
print("=" * 80)


for _, row in domain_item_df.iterrows():

    print(
        f"\n{row['Item']}"
    )

    print(
        row[
            "Candidate_Questionnaire_Item"
        ]
    )


print("\n")
print("=" * 80)
print("OUTPUT FILE")
print("=" * 80)

print(
    OUTPUT_FILE.resolve()
)

print("\nCompleted successfully.")

FINAL RISK VELOCITY EVIDENCE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Risk_Velocity_Coding_20260827_124722.xlsx

Available sheets:
 - 01_Original_Responses
 - 02_Raw_Themes
 - 03_Cleaned_Themes
 - 04_Coding_Dictionary
 - 05_Participant_Matrix
 - 06_Theme_Summary
 - 07_Decision_Summary
 - 08_Participant_Coverage
 - 09_Domain_Raw_Themes
 - 10_Domain_Item_Map
 - 11_Participant_Domains
 - 12_Final_RV_Evidence
 - 13_Unmapped_Themes

Participant matrix: 05_Participant_Matrix

Matrix shape: (26, 31)

Matrix columns:
 - Rank
 - Normalized_Theme
 - P01
 - P02
 - P03
 - P04
 - P05
 - P06
 - P07
 - P08
 - P09
 - P10
 - P11
 - P12
 - P13
 - P14
 - P15
 - P16
 - P17
 - P18
 - P19
 - P20
 - P21
 - P22
 - P23
 - P24
 - P25
 - P26
 - Experts_Mentioning
 - Expert_Prevalence_%
 - Prevalence_Category

Domain column: Normalized_Theme

Participants detected: 26


RISK VELOCITY ANALYSI

In [1]:
# =====================================================================
# RISK VELOCITY — FINAL QUALITATIVE EVIDENCE WORKBOOK
# =====================================================================
#
# INPUT FILE:
#     Risk_Velocity_Coding_20260827_124722.xlsx
#
# PURPOSE:
#     Convert the existing Risk Velocity evidence workbook into the
#     same structured final-evidence format used for the other constructs.
#
# OUTPUT:
#     Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_YYYYMMDD_HHMMSS.xlsx
#
# OUTPUT SHEETS:
#
# 01_Original_RV_Evidence
# 02_Item_Development
# 03_Cleaned_RV_Evidence
# 04_Coding_Audit
# 05_Participant_Matrix
# 06_Domain_Summary
# 07_Domain_Prevalence
# 08_Decision_Summary
# 09_Participant_Coverage
# 10_Unmapped_Themes
# 11_Final_RV_Evidence
# 12_RV_Dimension_Summary
# 13_RV_Dimension_Themes
# 14_Construct_Statistics
# 15_Quality_Checks
# 16_Methodology_Note
#
# =====================================================================


import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# =====================================================================
# 1. INPUT FILE
# =====================================================================

INPUT_FILE = Path(
    "Risk_Velocity_FINAL_EVIDENCE_20260827_125104.xlsx"
)


if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"""
Input file was not found.

Expected:
{INPUT_FILE.resolve()}

Current working directory:
{Path.cwd()}
"""
    )


print("=" * 90)
print("RISK VELOCITY — FINAL QUALITATIVE EVIDENCE ANALYSIS")
print("=" * 90)

print("\nInput file:")
print(INPUT_FILE.resolve())


# =====================================================================
# 2. READ WORKBOOK
# =====================================================================

xls = pd.ExcelFile(
    INPUT_FILE
)


print("\nAvailable sheets:")

for sheet in xls.sheet_names:

    print(" -", sheet)


# =====================================================================
# 3. READ THE EXISTING SHEETS
# =====================================================================

def read_if_exists(sheet_name):

    if sheet_name in xls.sheet_names:

        return pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet_name
        )

    return pd.DataFrame()


original_df = read_if_exists(
    "01_Final_RV_Evidence"
)

item_development_df = read_if_exists(
    "02_Item_Development"
)

domain_prevalence_df = read_if_exists(
    "03_Domain_Prevalence"
)

participant_matrix_df = read_if_exists(
    "04_Participant_Matrix"
)

participant_coverage_df = read_if_exists(
    "05_Participant_Coverage"
)

methodology_original_df = read_if_exists(
    "06_Methodology_Note"
)

quality_original_df = read_if_exists(
    "07_Quality_Checks"
)


# =====================================================================
# 4. VERIFY THE MAIN INPUT
# =====================================================================

required_columns = [

    "Item",

    "Risk_Velocity_Domain",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category",

    "Candidate_Questionnaire_Item"
]


missing_columns = [

    column
    for column in required_columns
    if column not in original_df.columns
]


if missing_columns:

    raise ValueError(
        f"""
The following columns are missing from
01_Final_RV_Evidence:

{missing_columns}

Available columns:
{list(original_df.columns)}
"""
    )


# =====================================================================
# 5. CLEAN MAIN EVIDENCE TABLE
# =====================================================================

rv = original_df[
    required_columns
].copy()


# Remove blank rows

rv = rv.dropna(
    subset=[
        "Item",
        "Risk_Velocity_Domain"
    ]
)


# Convert text columns to strings

for column in [
    "Item",
    "Risk_Velocity_Domain",
    "Prevalence_Category",
    "Candidate_Questionnaire_Item"
]:

    rv[column] = (
        rv[column]
        .astype(str)
        .str.strip()
    )


# Convert numeric fields

rv[
    "Experts_Mentioning"
] = pd.to_numeric(
    rv[
        "Experts_Mentioning"
    ],
    errors="coerce"
).fillna(0).astype(int)


rv[
    "Expert_Prevalence_%"
] = pd.to_numeric(
    rv[
        "Expert_Prevalence_%"
    ],
    errors="coerce"
)


# =====================================================================
# 6. REMOVE DUPLICATES
# =====================================================================

rv = (
    rv
    .drop_duplicates(
        subset=[
            "Item",
            "Risk_Velocity_Domain"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =====================================================================
# 7. ITEM ORDER
# =====================================================================

item_order = {

    "RV1": 1,

    "RV2": 2,

    "RV3": 3,

    "RV4": 4,

    "RV5": 5
}


rv[
    "_Item_Order"
] = rv[
    "Item"
].map(
    item_order
)


rv[
    "_Item_Order"
] = rv[
    "_Item_Order"
].fillna(999)


rv = (
    rv
    .sort_values(
        "_Item_Order"
    )
    .drop(
        columns="_Item_Order"
    )
    .reset_index(
        drop=True
    )
)


# =====================================================================
# 8. PARTICIPANT COLUMNS
# =====================================================================

participant_columns = [

    column
    for column in participant_matrix_df.columns
    if str(column).upper().startswith("P")
]


participant_columns = [

    column
    for column in participant_columns
    if str(column)[1:].isdigit()
]


participant_columns = sorted(
    participant_columns,
    key=lambda x: int(
        str(x)[1:]
    )
)


number_of_participants = len(
    participant_columns
)


print(
    "\nNumber of participants:",
    number_of_participants
)


# =====================================================================
# 9. REBUILD PARTICIPANT PREVALENCE DIRECTLY
# =====================================================================

matrix = participant_matrix_df.copy()


matrix_theme_column = (
    "Risk_Velocity_Domain"
)


# Make sure participant values are numeric

for column in participant_columns:

    matrix[column] = pd.to_numeric(
        matrix[column],
        errors="coerce"
    ).fillna(0)


# Calculate domain prevalence directly from
# the actual 26-participant matrix.

prevalence_records = []


for _, row in matrix.iterrows():

    domain = row[
        matrix_theme_column
    ]


    experts = int(
        (
            row[
                participant_columns
            ] > 0
        ).sum()
    )


    prevalence = round(
        experts /
        number_of_participants *
        100,
        1
    )


    if prevalence >= 75:

        category = "Very High"

    elif prevalence >= 50:

        category = "High"

    elif prevalence >= 25:

        category = "Moderate"

    else:

        category = "Low"


    prevalence_records.append({

        "Risk_Velocity_Domain":
            domain,

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            prevalence,

        "Prevalence_Category":
            category
    })


rebuilt_prevalence = pd.DataFrame(
    prevalence_records
)


# =====================================================================
# 10. MERGE PREVALENCE WITH RV ITEMS
# =====================================================================

rv = rv.drop(
    columns=[
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
)


rv = rv.merge(
    rebuilt_prevalence,
    on="Risk_Velocity_Domain",
    how="left"
)


# =====================================================================
# 11. ITEM → DOMAIN RATIONALE
# =====================================================================

rationales = {

    "RV1":
        "Captures the speed with which a supply-chain disruption develops after it first emerges.",

    "RV2":
        "Captures the speed with which disruption effects intensify and increasingly affect supply-chain operations.",

    "RV3":
        "Captures the speed with which disruption effects spread across the supply chain.",

    "RV4":
        "Captures the limited amount of time available for organizational response following disruption emergence.",

    "RV5":
        "Captures how quickly significant operational consequences materialize following a disruption."
}


rv[
    "Content_Domain_Rationale"
] = rv[
    "Item"
].map(
    rationales
)


rv[
    "Item_Status"
] = (
    "Candidate item — "
    "requires dedicated expert content validation"
)


# =====================================================================
# 12. RAW / NORMALIZED THEME COUNTS
# =====================================================================
#
# Because this Risk Velocity file contains five candidate domains
# rather than the original verbatim qualitative theme-level dataset,
# we explicitly distinguish:
#
#     candidate domain
#     normalized candidate theme
#
# We do NOT fabricate raw-theme counts.
#
# =====================================================================

rv[
    "Number_of_Raw_Themes"
] = np.nan


rv[
    "Raw_Themes_Kept"
] = np.nan


rv[
    "Raw_Themes_Merged"
] = np.nan


# =====================================================================
# 13. CREATE FINAL EVIDENCE TABLE
# =====================================================================

final_evidence = rv[
    [
        "Item",
        "Risk_Velocity_Domain",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Candidate_Questionnaire_Item",
        "Content_Domain_Rationale",
        "Item_Status",
        "Number_of_Raw_Themes",
        "Raw_Themes_Kept",
        "Raw_Themes_Merged"
    ]
].copy()


# =====================================================================
# 14. CREATE SAFE RANK
# =====================================================================
#
# IMPORTANT:
# We explicitly remove any existing Rank column before creating it.
# This eliminates the previous:
#
# ValueError: cannot insert Rank, already exists
#
# =====================================================================

if "Rank" in final_evidence.columns:

    final_evidence = final_evidence.drop(
        columns=["Rank"]
    )


final_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_evidence) + 1
    )
)


# =====================================================================
# 15. DIMENSION SUMMARY
# =====================================================================

dimension_summary = (
    rebuilt_prevalence
    .copy()
)


dimension_summary[
    "Number_of_Normalized_Themes"
] = 1


# Number of items mapped to each domain

domain_item_counts = (
    rv
    .groupby(
        "Risk_Velocity_Domain"
    )
    .size()
    .rename(
        "Number_of_Questionnaire_Items"
    )
    .reset_index()
)


dimension_summary = dimension_summary.merge(
    domain_item_counts,
    on="Risk_Velocity_Domain",
    how="left"
)


# =====================================================================
# 16. INCLUDED RV ITEMS
# =====================================================================

included_items = (
    rv
    .groupby(
        "Risk_Velocity_Domain"
    )[
        "Item"
    ]
    .apply(
        lambda x:
        "; ".join(
            x.astype(str)
        )
    )
    .reset_index()
)


dimension_summary = dimension_summary.merge(
    included_items,
    on="Risk_Velocity_Domain",
    how="left"
)


dimension_summary = dimension_summary.rename(
    columns={
        "Item":
            "Included_RV_Items"
    }
)


# =====================================================================
# 17. DIMENSION RANKING
# =====================================================================

dimension_summary = (
    dimension_summary
    .sort_values(
        [
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Number_of_Normalized_Themes"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


if "Rank" in dimension_summary.columns:

    dimension_summary = dimension_summary.drop(
        columns=["Rank"]
    )


dimension_summary.insert(
    0,
    "Rank",
    range(
        1,
        len(dimension_summary) + 1
    )
)


# =====================================================================
# 18. DIMENSION THEMES
# =====================================================================

dimension_themes = rv[
    [
        "Risk_Velocity_Domain",
        "Item",
        "Candidate_Questionnaire_Item",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Content_Domain_Rationale"
    ]
].copy()


# =====================================================================
# 19. PARTICIPANT MATRIX
# =====================================================================

participant_matrix_final = matrix[
    [
        "Risk_Velocity_Domain"
    ]
    +
    participant_columns
].copy()


# =====================================================================
# 20. PARTICIPANT COVERAGE
# =====================================================================

coverage_records = []


for _, row in participant_matrix_final.iterrows():

    pass


for participant in participant_columns:

    number_domains = int(
        (
            participant_matrix_final[
                participant
            ] > 0
        ).sum()
    )


    percentage = round(
        number_domains /
        max(
            len(
                participant_matrix_final
            ),
            1
        )
        *
        100,
        1
    )


    if percentage >= 75:

        category = "Very High"

    elif percentage >= 50:

        category = "High"

    elif percentage >= 25:

        category = "Moderate"

    else:

        category = "Low"


    coverage_records.append({

        "Participant":
            participant,

        "Final_RV_Domains_Mentioned":
            number_domains,

        "Percentage_of_Final_Domains_Covered":
            percentage,

        "Coverage_Category":
            category
    })


participant_coverage_final = pd.DataFrame(
    coverage_records
)


# =====================================================================
# 21. UNMAPPED THEMES
# =====================================================================
#
# In this particular input, all five candidate domains are already
# mapped. Therefore the expected result is zero unmapped domains.
#
# =====================================================================

unmapped = rv[
    rv[
        "Risk_Velocity_Domain"
    ].isna()
    |
    (
        rv[
            "Risk_Velocity_Domain"
        ]
        .astype(str)
        .str.strip()
        == ""
    )
].copy()


if len(unmapped) == 0:

    unmapped_output = pd.DataFrame({

        "Status": [
            "PASS"
        ],

        "Unmapped_RV_Domains": [
            0
        ]
    })

else:

    unmapped_output = unmapped[
        [
            "Item",
            "Risk_Velocity_Domain"
        ]
    ].copy()


# =====================================================================
# 22. DECISION SUMMARY
# =====================================================================

decision_summary = pd.DataFrame({

    "Decision_Component": [

        "Candidate Risk Velocity items",

        "Risk Velocity content domains",

        "Highest-prevalence domain",

        "Lowest-prevalence domain",

        "All candidate domains mapped",

        "All five items have wording",

        "Next methodological stage"
    ],

    "Result": [

        len(rv),

        rv[
            "Risk_Velocity_Domain"
        ].nunique(),

        dimension_summary.iloc[0][
            "Risk_Velocity_Domain"
        ],

        dimension_summary.iloc[-1][
            "Risk_Velocity_Domain"
        ],

        len(unmapped) == 0,

        rv[
            "Candidate_Questionnaire_Item"
        ].notna().all(),

        "Literature triangulation + dedicated expert content validation"
    ]
})


# =====================================================================
# 23. CODING AUDIT
# =====================================================================

coding_audit = pd.DataFrame({

    "Audit_Item": [

        "Number of candidate items",

        "Number of candidate domains",

        "Participants",

        "Duplicate items",

        "Duplicate domains",

        "Items without questionnaire wording",

        "Domains without item mapping"
    ],

    "Result": [

        len(rv),

        rv[
            "Risk_Velocity_Domain"
        ].nunique(),

        number_of_participants,

        int(
            rv[
                "Item"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Risk_Velocity_Domain"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Candidate_Questionnaire_Item"
            ]
            .isna()
            .sum()
        ),

        int(
            dimension_summary[
                "Number_of_Questionnaire_Items"
            ]
            .eq(0)
            .sum()
        )
    ]
})


# =====================================================================
# 24. CONSTRUCT STATISTICS
# =====================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "Risk Velocity"
    ],

    "Participants": [
        number_of_participants
    ],

    "Candidate_Items": [
        len(rv)
    ],

    "Candidate_Content_Domains": [
        rv[
            "Risk_Velocity_Domain"
        ].nunique()
    ],

    "Highest_Domain_Prevalence_%": [
        dimension_summary[
            "Expert_Prevalence_%"
        ].max()
    ],

    "Lowest_Domain_Prevalence_%": [
        dimension_summary[
            "Expert_Prevalence_%"
        ].min()
    ],

    "Mean_Domain_Prevalence_%": [
        round(
            dimension_summary[
                "Expert_Prevalence_%"
            ].mean(),
            1
        )
    ],

    "All_Domains_Mapped": [
        len(unmapped) == 0
    ],

    "All_Items_Worded": [
        rv[
            "Candidate_Questionnaire_Item"
        ].notna().all()
    ]
})


# =====================================================================
# 25. QUALITY CHECKS
# =====================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Input file exists",

        "Participant columns detected",

        "Number of participants",

        "Five candidate RV items present",

        "Five candidate RV domains present",

        "Duplicate item rows",

        "Duplicate domain rows",

        "Unmapped domains",

        "Missing questionnaire items",

        "Rank column duplicated",

        "Prevalence values available"
    ],

    "Result": [

        True,

        len(participant_columns) > 0,

        number_of_participants,

        len(rv) == 5,

        rv[
            "Risk_Velocity_Domain"
        ].nunique() == 5,

        int(
            rv[
                "Item"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Risk_Velocity_Domain"
            ]
            .duplicated()
            .sum()
        ),

        len(unmapped),

        int(
            rv[
                "Candidate_Questionnaire_Item"
            ]
            .isna()
            .sum()
        ),

        int(
            final_evidence.columns
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Expert_Prevalence_%"
            ]
            .notna()
            .sum()
        )
    ]
})


# =====================================================================
# 26. METHODOLOGY NOTE
# =====================================================================

methodology_note = pd.DataFrame({

    "Component": [

        "Input",

        "Analytical unit",

        "Participants",

        "Risk Velocity domains",

        "Candidate items",

        "Participant prevalence",

        "Use of Python",

        "Important limitation",

        "Next stage"
    ],

    "Description": [

        "Risk_Velocity_Coding_20260827_124722.xlsx.",

        "Participant-level candidate Risk Velocity domains and their corresponding preliminary questionnaire items.",

        f"{number_of_participants} experts/participants represented in the participant matrix.",

        "Rapid Disruption Emergence; Rapid Disruption Escalation; Rapid Disruption Propagation; Limited Response Window; Rapid Operational Consequences / Short Time-to-Impact.",

        "Five preliminary items RV1–RV5 were mapped one-to-one to the five candidate content domains.",

        "Domain prevalence was recalculated directly from the 26-participant matrix rather than relying on manually entered prevalence values.",

        "Python was used for data organization, prevalence calculation, ranking, dimensional structuring, duplicate checks, quality assurance and reproducible Excel-file generation.",

        "The Risk Velocity material represents candidate content-development evidence. It should not be presented as equivalent to verbatim qualitative responses specifically elicited for a pre-established Risk Velocity construct.",

        "The next stage is literature triangulation of RV1–RV5 followed by dedicated expert content validation before the main quantitative PLS-SEM survey."
    ]
})


# =====================================================================
# 27. CREATE OUTPUT FILE
# =====================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# =====================================================================
# 28. WRITE ALL SHEETS
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    final_evidence.to_excel(
        writer,
        sheet_name="01_Original_RV_Evidence",
        index=False
    )


    item_development_df.to_excel(
        writer,
        sheet_name="02_Item_Development",
        index=False
    )


    final_evidence.to_excel(
        writer,
        sheet_name="03_Cleaned_RV_Evidence",
        index=False
    )


    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    participant_matrix_final.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="06_Domain_Summary",
        index=False
    )


    rebuilt_prevalence.to_excel(
        writer,
        sheet_name="07_Domain_Prevalence",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage_final.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_output.to_excel(
        writer,
        sheet_name="10_Unmapped_Themes",
        index=False
    )


    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RV_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="12_RV_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="13_RV_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="14_Construct_Statistics",
        index=False
    )


    quality_checks.to_excel(
        writer,
        sheet_name="15_Quality_Checks",
        index=False
    )


    methodology_note.to_excel(
        writer,
        sheet_name="16_Methodology_Note",
        index=False
    )


# =====================================================================
# 29. PROFESSIONAL EXCEL FORMATTING
# =====================================================================

wb = load_workbook(
    OUTPUT_FILE
)


header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)


header_font = Font(
    bold=True,
    color="FFFFFF"
)


for ws in wb.worksheets:


    ws.freeze_panes = "A2"


    # Header formatting

    for cell in ws[1]:

        cell.fill = header_fill

        cell.font = header_font

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    # Body formatting

    for row in ws.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )


    # Column widths

    for column_cells in ws.columns:

        column_letter = get_column_letter(
            column_cells[0].column
        )

        max_length = 0


        for cell in column_cells:

            if cell.value is None:
                continue


            max_length = max(
                max_length,
                len(
                    str(
                        cell.value
                    )
                )
            )


        ws.column_dimensions[
            column_letter
        ].width = min(
            max(
                max_length + 2,
                12
            ),
            65
        )


    # Autofilter

    if ws.max_row > 1:

        ws.auto_filter.ref = (
            ws.dimensions
        )


# Save

wb.save(
    OUTPUT_FILE
)


# =====================================================================
# 30. FINAL CONSOLE REPORT
# =====================================================================

print("\n")
print("=" * 90)
print("RISK VELOCITY FINAL FILE CREATED")
print("=" * 90)

print(
    "\nParticipants:",
    number_of_participants
)

print(
    "Candidate items:",
    len(rv)
)

print(
    "Candidate domains:",
    rv[
        "Risk_Velocity_Domain"
    ].nunique()
)

print(
    "Unmapped domains:",
    len(unmapped)
)

print(
    "Duplicate item rows:",
    int(
        rv[
            "Item"
        ]
        .duplicated()
        .sum()
    )
)

print(
    "Duplicate domain rows:",
    int(
        rv[
            "Risk_Velocity_Domain"
        ]
        .duplicated()
        .sum()
    )
)


print("\n")
print("=" * 90)
print("RISK VELOCITY DOMAIN SUMMARY")
print("=" * 90)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("FINAL OUTPUT")
print("=" * 90)

print(
    OUTPUT_FILE.resolve()
)

print("\nDone.")

RISK VELOCITY — FINAL QUALITATIVE EVIDENCE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Risk_Velocity_FINAL_EVIDENCE_20260827_125104.xlsx

Available sheets:
 - 01_Final_RV_Evidence
 - 02_Item_Development
 - 03_Domain_Prevalence
 - 04_Participant_Matrix
 - 05_Participant_Coverage
 - 06_Methodology_Note
 - 07_Quality_Checks

Number of participants: 26


RISK VELOCITY FINAL FILE CREATED

Participants: 26
Candidate items: 5
Candidate domains: 5
Unmapped domains: 0
Duplicate item rows: 0
Duplicate domain rows: 0


RISK VELOCITY DOMAIN SUMMARY
 Rank                                  Risk_Velocity_Domain  Experts_Mentioning  Expert_Prevalence_% Prevalence_Category  Number_of_Normalized_Themes  Number_of_Questionnaire_Items Included_RV_Items
    1                          Rapid Disruption Propagation                  24                 92.3           Very High                 

In [2]:
# ================================================================
# FINAL RISK VELOCITY EVIDENCE + ITEM DEVELOPMENT ANALYSIS
# CORRECTED / RERUN-SAFE VERSION
# ================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# ================================================================
# 1. INPUT FILE
# ================================================================

INPUT_FILE = Path(
    "Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_20260828_132525.xlsx"
)

if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"""
INPUT FILE NOT FOUND

Expected:
{INPUT_FILE.resolve()}

Current working directory:
{Path.cwd()}
"""
    )


print("=" * 80)
print("FINAL RISK VELOCITY EVIDENCE ANALYSIS")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())


# ================================================================
# 2. READ WORKBOOK
# ================================================================

xls = pd.ExcelFile(INPUT_FILE)

print("\nAvailable sheets:")

for sheet in xls.sheet_names:
    print(" -", sheet)


# ================================================================
# 3. FINAL RISK VELOCITY DOMAINS
# ================================================================

FINAL_DOMAINS = [

    "Rapid Disruption Emergence",

    "Rapid Disruption Escalation",

    "Rapid Disruption Propagation",

    "Limited Response Window",

    "Rapid Operational Consequences / Short Time-to-Impact"
]


# ================================================================
# 4. FINAL QUESTIONNAIRE ITEMS
# ================================================================

ITEMS = {

    "Rapid Disruption Emergence":
        (
            "RV1",
            "Supply chain disruptions can develop rapidly "
            "after they first emerge."
        ),

    "Rapid Disruption Escalation":
        (
            "RV2",
            "Disruptions can escalate quickly and affect "
            "our supply chain operations."
        ),

    "Rapid Disruption Propagation":
        (
            "RV3",
            "The effects of supply chain disruptions can "
            "spread rapidly across our supply chain."
        ),

    "Limited Response Window":
        (
            "RV4",
            "Our organization often has a limited time "
            "window to respond after a disruption emerges."
        ),

    "Rapid Operational Consequences / Short Time-to-Impact":
        (
            "RV5",
            "Supply chain disruptions can cause significant "
            "operational consequences within a short period."
        )
}


# ================================================================
# 5. FIND PARTICIPANT MATRIX SHEET
# ================================================================

matrix_sheet = None

for sheet in xls.sheet_names:

    normalized = (
        str(sheet)
        .strip()
        .lower()
        .replace(" ", "_")
    )

    if normalized in [
        "05_participant_matrix",
        "participant_matrix"
    ]:

        matrix_sheet = sheet
        break


if matrix_sheet is None:

    raise ValueError(
        """
Could not find the Participant Matrix sheet.

Expected:
05_Participant_Matrix
"""
    )


print(
    "\nParticipant matrix:",
    matrix_sheet
)


# ================================================================
# 6. READ PARTICIPANT MATRIX
# ================================================================

matrix_df = pd.read_excel(
    INPUT_FILE,
    sheet_name=matrix_sheet
)


print(
    "\nMatrix shape:",
    matrix_df.shape
)

print(
    "\nMatrix columns:"
)

for c in matrix_df.columns:

    print(
        " -",
        c
    )


# ================================================================
# 7. FIND DOMAIN COLUMN
# ================================================================

domain_column = None

possible_domain_columns = [

    "Normalized_Theme",

    "Risk_Velocity_Domain",

    "Final_RV_Domain",

    "Domain"
]


for c in possible_domain_columns:

    if c in matrix_df.columns:

        domain_column = c
        break


if domain_column is None:

    for c in matrix_df.columns:

        name = str(c).lower()

        if (
            "theme" in name
            or
            "domain" in name
        ):

            domain_column = c
            break


if domain_column is None:

    raise ValueError(
        """
Could not identify the Risk Velocity domain column.
"""
    )


print(
    "\nDomain column:",
    domain_column
)


# ================================================================
# 8. FIND PARTICIPANT COLUMNS
# ================================================================

participant_columns = []

for i in range(1, 27):

    participant = f"P{i:02d}"

    if participant in matrix_df.columns:

        participant_columns.append(
            participant
        )


print(
    "\nParticipants detected:",
    len(participant_columns)
)


if not participant_columns:

    raise ValueError(
        """
No P01–P26 columns were found.
"""
    )


# ================================================================
# 9. KEEP ONLY FINAL FIVE DOMAINS
# ================================================================

domain_df = matrix_df[
    matrix_df[
        domain_column
    ].isin(
        FINAL_DOMAINS
    )
].copy()


# ================================================================
# 10. ENSURE ALL FIVE DOMAINS EXIST
# ================================================================

existing_domains = set(
    domain_df[
        domain_column
    ]
    .astype(str)
)


for domain in FINAL_DOMAINS:

    if domain not in existing_domains:

        new_row = {
            domain_column:
                domain
        }

        for participant in participant_columns:

            new_row[
                participant
            ] = 0

        domain_df = pd.concat(
            [
                domain_df,
                pd.DataFrame(
                    [new_row]
                )
            ],
            ignore_index=True
        )


# ================================================================
# 11. REMOVE DUPLICATE DOMAINS
# ================================================================

domain_df = (
    domain_df
    .drop_duplicates(
        subset=[
            domain_column
        ],
        keep="first"
    )
    .copy()
)


# ================================================================
# 12. FORCE PARTICIPANT VALUES TO NUMERIC
# ================================================================

for participant in participant_columns:

    domain_df[
        participant
    ] = pd.to_numeric(
        domain_df[
            participant
        ],
        errors="coerce"
    ).fillna(0)


# ================================================================
# 13. CALCULATE EXPERT MENTIONING
# ================================================================

domain_df[
    "Experts_Mentioning"
] = domain_df[
    participant_columns
].sum(
    axis=1
)


# ================================================================
# 14. CALCULATE PREVALENCE
# ================================================================

number_of_experts = len(
    participant_columns
)


domain_df[
    "Expert_Prevalence_%"
] = (
    domain_df[
        "Experts_Mentioning"
    ]
    /
    number_of_experts
    *
    100
).round(1)


# ================================================================
# 15. PREVALENCE CATEGORY
# ================================================================

def prevalence_category(
    value
):

    if value >= 75:

        return "Very High"

    elif value >= 50:

        return "High"

    elif value >= 25:

        return "Moderate"

    else:

        return "Low"


domain_df[
    "Prevalence_Category"
] = domain_df[
    "Expert_Prevalence_%"
].apply(
    prevalence_category
)


# ================================================================
# 16. SORT BY PREVALENCE
# ================================================================

ranked_df = (
    domain_df
    .sort_values(
        by=[
            "Experts_Mentioning",
            "Expert_Prevalence_%"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 17. SAFE RANK CREATION
# ================================================================
#
# THIS FIXES YOUR ERROR.
#
# If Rank already exists, overwrite it.
# If Rank does not exist, create it.
#
# ================================================================

ranked_df[
    "Rank"
] = range(
    1,
    len(ranked_df) + 1
)


# Move Rank to first position safely.

columns = list(
    ranked_df.columns
)

columns.remove(
    "Rank"
)

ranked_df = ranked_df[
    ["Rank"] + columns
]


# ================================================================
# 18. FINAL DOMAIN PREVALENCE TABLE
# ================================================================

domain_prevalence = ranked_df[
    [
        "Rank",
        domain_column,
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
].copy()


domain_prevalence = domain_prevalence.rename(
    columns={
        domain_column:
            "Risk_Velocity_Domain"
    }
)


# ================================================================
# 19. DOMAIN → ITEM TABLE
# ================================================================

item_rows = []


for domain in FINAL_DOMAINS:

    item_code, item_text = ITEMS[
        domain
    ]


    match = ranked_df[
        ranked_df[
            domain_column
        ] == domain
    ]


    if len(match) > 0:

        row = match.iloc[0]

        experts = int(
            row[
                "Experts_Mentioning"
            ]
        )

        prevalence = float(
            row[
                "Expert_Prevalence_%"
            ]
        )

        category = row[
            "Prevalence_Category"
        ]

    else:

        experts = 0
        prevalence = 0.0
        category = "Not observed"


    item_rows.append({

        "Item":
            item_code,

        "Risk_Velocity_Domain":
            domain,

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            prevalence,

        "Prevalence_Category":
            category,

        "Candidate_Questionnaire_Item":
            item_text
    })


domain_item_df = pd.DataFrame(
    item_rows
)


# ================================================================
# 20. ADD CONTENT DOMAIN RATIONALE
# ================================================================

rationales = {

    "Rapid Disruption Emergence":
        "Captures how rapidly a disruption develops after it first emerges.",

    "Rapid Disruption Escalation":
        "Captures how quickly disruption effects intensify or escalate.",

    "Rapid Disruption Propagation":
        "Captures how rapidly disruption effects spread across supply-chain entities or stages.",

    "Limited Response Window":
        "Captures the limited time available for organizational response after disruption emergence.",

    "Rapid Operational Consequences / Short Time-to-Impact":
        "Captures how quickly significant operational consequences materialize."
}


domain_item_df[
    "Content_Domain_Rationale"
] = domain_item_df[
    "Risk_Velocity_Domain"
].map(
    rationales
)


domain_item_df[
    "Item_Status"
] = (
    "Candidate item for expert content validation"
)


# ================================================================
# 21. PARTICIPANT × DOMAIN MATRIX
# ================================================================

participant_matrix = ranked_df[
    [
        domain_column
    ]
    +
    participant_columns
].copy()


participant_matrix = participant_matrix.rename(
    columns={
        domain_column:
            "Risk_Velocity_Domain"
    }
)


# ================================================================
# 22. PARTICIPANT COVERAGE
# ================================================================

coverage_rows = []


for participant in participant_columns:

    values = pd.to_numeric(
        participant_matrix[
            participant
        ],
        errors="coerce"
    ).fillna(0)


    domains_mentioned = int(
        (
            values > 0
        ).sum()
    )


    coverage_rows.append({

        "Participant":
            participant,

        "Final_RV_Domains_Mentioned":
            domains_mentioned,

        "Percentage_of_Final_Domains_Covered":
            round(
                domains_mentioned
                /
                len(FINAL_DOMAINS)
                *
                100,
                1
            )
    })


coverage_df = pd.DataFrame(
    coverage_rows
)


# ================================================================
# 23. PARTICIPANT COVERAGE CATEGORY
# ================================================================

coverage_df[
    "Coverage_Category"
] = coverage_df[
    "Percentage_of_Final_Domains_Covered"
].apply(
    prevalence_category
)


# ================================================================
# 24. CREATE FINAL FIVE-ITEM TABLE
# ================================================================

final_evidence = domain_item_df[
    [
        "Item",
        "Risk_Velocity_Domain",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Candidate_Questionnaire_Item"
    ]
].copy()


# ================================================================
# 25. METHODOLOGY NOTE
# ================================================================

methodology_note = pd.DataFrame({

    "Component": [

        "Input",

        "Analytical unit",

        "Final content domains",

        "Item development",

        "Use of Python",

        "Important limitation",

        "Next methodological stage"
    ],

    "Description": [

        "Risk_Velocity_Coding_20260827_124722.xlsx.",

        "Participant-level candidate Risk Velocity themes organized into five content domains.",

        "Rapid Disruption Emergence; Rapid Disruption Escalation; Rapid Disruption Propagation; Limited Response Window; Rapid Operational Consequences / Short Time-to-Impact.",

        "Five preliminary items (RV1–RV5) were mapped to the five content domains.",

        "Python was used to organize, normalize, audit, calculate participant coverage/prevalence, and generate the final evidence tables.",

        "The underlying Risk Velocity themes were generated candidate themes rather than verbatim empirical responses specifically elicited from the 26 experts. Therefore, the resulting prevalence values must not be presented as actual empirical expert prevalence.",

        "Triangulate each item with multiple published sources and conduct dedicated expert content validation before the main PLS-SEM survey."
    ]
})


# ================================================================
# 26. QUALITY CHECKS
# ================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Input file exists",

        "Participant columns detected",

        "Number of participants",

        "Five final domains present",

        "Five questionnaire items present",

        "Duplicate domain rows",

        "Rank column duplicated"
    ],

    "Result": [

        True,

        len(participant_columns) > 0,

        number_of_experts,

        len(
            ranked_df
        ) == 5,

        len(
            domain_item_df
        ) == 5,

        int(
            ranked_df[
                domain_column
            ].duplicated().sum()
        ),

        ranked_df.columns.tolist().count(
            "Rank"
        )
    ]
})


# ================================================================
# 27. CREATE OUTPUT FILE
# ================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"Risk_Velocity_FINAL_EVIDENCE_{timestamp}.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    final_evidence.to_excel(
        writer,
        sheet_name="01_Final_RV_Evidence",
        index=False
    )


    domain_item_df.to_excel(
        writer,
        sheet_name="02_Item_Development",
        index=False
    )


    domain_prevalence.to_excel(
        writer,
        sheet_name="03_Domain_Prevalence",
        index=False
    )


    participant_matrix.to_excel(
        writer,
        sheet_name="04_Participant_Matrix",
        index=False
    )


    coverage_df.to_excel(
        writer,
        sheet_name="05_Participant_Coverage",
        index=False
    )


    methodology_note.to_excel(
        writer,
        sheet_name="06_Methodology_Note",
        index=False
    )


    quality_checks.to_excel(
        writer,
        sheet_name="07_Quality_Checks",
        index=False
    )


# ================================================================
# 28. PROFESSIONAL FORMATTING
# ================================================================

try:

    wb = load_workbook(
        OUTPUT_FILE
    )


    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )


    header_font = Font(
        bold=True,
        color="FFFFFF"
    )


    for ws in wb.worksheets:

        ws.freeze_panes = "A2"


        for cell in ws[1]:

            cell.fill = header_fill

            cell.font = header_font

            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )


        for row in ws.iter_rows(
            min_row=2
        ):

            for cell in row:

                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=True
                )


        for column_cells in ws.columns:

            column_letter = get_column_letter(
                column_cells[0].column
            )


            max_length = 0


            for cell in column_cells:

                if cell.value is None:

                    continue


                max_length = max(
                    max_length,
                    len(
                        str(
                            cell.value
                        )
                    )
                )


            ws.column_dimensions[
                column_letter
            ].width = min(
                max(
                    max_length + 2,
                    12
                ),
                65
            )


        ws.auto_filter.ref = (
            ws.dimensions
        )


    wb.save(
        OUTPUT_FILE
    )


except Exception as formatting_error:

    print(
        "\nFormatting warning:"
    )

    print(
        formatting_error
    )


# ================================================================
# 29. FINAL CONSOLE OUTPUT
# ================================================================

print("\n")
print("=" * 80)
print("RISK VELOCITY ANALYSIS COMPLETED")
print("=" * 80)


print(
    "\nParticipants:",
    number_of_experts
)


print(
    "Final domains:",
    len(
        FINAL_DOMAINS
    )
)


print(
    "\nFINAL DOMAIN SUMMARY"
)

print(
    domain_prevalence.to_string(
        index=False
    )
)


print("\n")
print("=" * 80)
print("FINAL RV ITEMS")
print("=" * 80)


for _, row in domain_item_df.iterrows():

    print(
        f"\n{row['Item']}"
    )

    print(
        row[
            "Candidate_Questionnaire_Item"
        ]
    )


print("\n")
print("=" * 80)
print("OUTPUT FILE")
print("=" * 80)

print(
    OUTPUT_FILE.resolve()
)

print("\nCompleted successfully.")

FINAL RISK VELOCITY EVIDENCE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_20260828_132525.xlsx

Available sheets:
 - 01_Original_RV_Evidence
 - 02_Item_Development
 - 03_Cleaned_RV_Evidence
 - 04_Coding_Audit
 - 05_Participant_Matrix
 - 06_Domain_Summary
 - 07_Domain_Prevalence
 - 08_Decision_Summary
 - 09_Participant_Coverage
 - 10_Unmapped_Themes
 - 11_Final_RV_Evidence
 - 12_RV_Dimension_Summary
 - 13_RV_Dimension_Themes
 - 14_Construct_Statistics
 - 15_Quality_Checks
 - 16_Methodology_Note

Participant matrix: 05_Participant_Matrix

Matrix shape: (5, 27)

Matrix columns:
 - Risk_Velocity_Domain
 - P01
 - P02
 - P03
 - P04
 - P05
 - P06
 - P07
 - P08
 - P09
 - P10
 - P11
 - P12
 - P13
 - P14
 - P15
 - P16
 - P17
 - P18
 - P19
 - P20
 - P21
 - P22
 - P23
 - P24
 - P25
 - P26

Domain column: Risk_Velocity_Domain

Participants 

In [ ]:
# =====================================================================
# RISK VELOCITY — FINAL QUALITATIVE EVIDENCE WORKBOOK
# =====================================================================
#
# INPUT FILE:
#     Risk_Velocity_FINAL_EVIDENCE_20260828_132957.xlsx
#
# PURPOSE:
#     Convert the existing Risk Velocity evidence workbook into the
#     same structured final-evidence format used for the other constructs.
#
# OUTPUT:
#     Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_20260828_133159.xlsx
#
# OUTPUT SHEETS:
#
# 01_Original_RV_Evidence
# 02_Item_Development
# 03_Cleaned_RV_Evidence
# 04_Coding_Audit
# 05_Participant_Matrix
# 06_Domain_Summary
# 07_Domain_Prevalence
# 08_Decision_Summary
# 09_Participant_Coverage
# 10_Unmapped_Themes
# 11_Final_RV_Evidence
# 12_RV_Dimension_Summary
# 13_RV_Dimension_Themes
# 14_Construct_Statistics
# 15_Quality_Checks
# 16_Methodology_Note
#
# =====================================================================


import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# =====================================================================
# 1. INPUT FILE
# =====================================================================

INPUT_FILE = Path(
    "Risk_Velocity_FINAL_EVIDENCE_20260828_132957.xlsx"
)


if not INPUT_FILE.exists():

    raise FileNotFoundError(
        f"""
Input file was not found.

Expected:
{INPUT_FILE.resolve()}

Current working directory:
{Path.cwd()}
"""
    )


print("=" * 90)
print("RISK VELOCITY — FINAL QUALITATIVE EVIDENCE ANALYSIS")
print("=" * 90)

print("\nInput file:")
print(INPUT_FILE.resolve())


# =====================================================================
# 2. READ WORKBOOK
# =====================================================================

xls = pd.ExcelFile(
    INPUT_FILE
)


print("\nAvailable sheets:")

for sheet in xls.sheet_names:

    print(" -", sheet)


# =====================================================================
# 3. READ THE EXISTING SHEETS
# =====================================================================

def read_if_exists(sheet_name):

    if sheet_name in xls.sheet_names:

        return pd.read_excel(
            INPUT_FILE,
            sheet_name=sheet_name
        )

    return pd.DataFrame()


original_df = read_if_exists(
    "01_Final_RV_Evidence"
)

item_development_df = read_if_exists(
    "02_Item_Development"
)

domain_prevalence_df = read_if_exists(
    "03_Domain_Prevalence"
)

participant_matrix_df = read_if_exists(
    "04_Participant_Matrix"
)

participant_coverage_df = read_if_exists(
    "05_Participant_Coverage"
)

methodology_original_df = read_if_exists(
    "06_Methodology_Note"
)

quality_original_df = read_if_exists(
    "07_Quality_Checks"
)


# =====================================================================
# 4. VERIFY THE MAIN INPUT
# =====================================================================

required_columns = [

    "Item",

    "Risk_Velocity_Domain",

    "Experts_Mentioning",

    "Expert_Prevalence_%",

    "Prevalence_Category",

    "Candidate_Questionnaire_Item"
]


missing_columns = [

    column
    for column in required_columns
    if column not in original_df.columns
]


if missing_columns:

    raise ValueError(
        f"""
The following columns are missing from
01_Final_RV_Evidence:

{missing_columns}

Available columns:
{list(original_df.columns)}
"""
    )


# =====================================================================
# 5. CLEAN MAIN EVIDENCE TABLE
# =====================================================================

rv = original_df[
    required_columns
].copy()


# Remove blank rows

rv = rv.dropna(
    subset=[
        "Item",
        "Risk_Velocity_Domain"
    ]
)


# Convert text columns to strings

for column in [
    "Item",
    "Risk_Velocity_Domain",
    "Prevalence_Category",
    "Candidate_Questionnaire_Item"
]:

    rv[column] = (
        rv[column]
        .astype(str)
        .str.strip()
    )


# Convert numeric fields

rv[
    "Experts_Mentioning"
] = pd.to_numeric(
    rv[
        "Experts_Mentioning"
    ],
    errors="coerce"
).fillna(0).astype(int)


rv[
    "Expert_Prevalence_%"
] = pd.to_numeric(
    rv[
        "Expert_Prevalence_%"
    ],
    errors="coerce"
)


# =====================================================================
# 6. REMOVE DUPLICATES
# =====================================================================

rv = (
    rv
    .drop_duplicates(
        subset=[
            "Item",
            "Risk_Velocity_Domain"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =====================================================================
# 7. ITEM ORDER
# =====================================================================

item_order = {

    "RV1": 1,

    "RV2": 2,

    "RV3": 3,

    "RV4": 4,

    "RV5": 5
}


rv[
    "_Item_Order"
] = rv[
    "Item"
].map(
    item_order
)


rv[
    "_Item_Order"
] = rv[
    "_Item_Order"
].fillna(999)


rv = (
    rv
    .sort_values(
        "_Item_Order"
    )
    .drop(
        columns="_Item_Order"
    )
    .reset_index(
        drop=True
    )
)


# =====================================================================
# 8. PARTICIPANT COLUMNS
# =====================================================================

participant_columns = [

    column
    for column in participant_matrix_df.columns
    if str(column).upper().startswith("P")
]


participant_columns = [

    column
    for column in participant_columns
    if str(column)[1:].isdigit()
]


participant_columns = sorted(
    participant_columns,
    key=lambda x: int(
        str(x)[1:]
    )
)


number_of_participants = len(
    participant_columns
)


print(
    "\nNumber of participants:",
    number_of_participants
)


# =====================================================================
# 9. REBUILD PARTICIPANT PREVALENCE DIRECTLY
# =====================================================================

matrix = participant_matrix_df.copy()


matrix_theme_column = (
    "Risk_Velocity_Domain"
)


# Make sure participant values are numeric

for column in participant_columns:

    matrix[column] = pd.to_numeric(
        matrix[column],
        errors="coerce"
    ).fillna(0)


# Calculate domain prevalence directly from
# the actual 26-participant matrix.

prevalence_records = []


for _, row in matrix.iterrows():

    domain = row[
        matrix_theme_column
    ]


    experts = int(
        (
            row[
                participant_columns
            ] > 0
        ).sum()
    )


    prevalence = round(
        experts /
        number_of_participants *
        100,
        1
    )


    if prevalence >= 75:

        category = "Very High"

    elif prevalence >= 50:

        category = "High"

    elif prevalence >= 25:

        category = "Moderate"

    else:

        category = "Low"


    prevalence_records.append({

        "Risk_Velocity_Domain":
            domain,

        "Experts_Mentioning":
            experts,

        "Expert_Prevalence_%":
            prevalence,

        "Prevalence_Category":
            category
    })


rebuilt_prevalence = pd.DataFrame(
    prevalence_records
)


# =====================================================================
# 10. MERGE PREVALENCE WITH RV ITEMS
# =====================================================================

rv = rv.drop(
    columns=[
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category"
    ]
)


rv = rv.merge(
    rebuilt_prevalence,
    on="Risk_Velocity_Domain",
    how="left"
)


# =====================================================================
# 11. ITEM → DOMAIN RATIONALE
# =====================================================================

rationales = {

    "RV1":
        "Captures the speed with which a supply-chain disruption develops after it first emerges.",

    "RV2":
        "Captures the speed with which disruption effects intensify and increasingly affect supply-chain operations.",

    "RV3":
        "Captures the speed with which disruption effects spread across the supply chain.",

    "RV4":
        "Captures the limited amount of time available for organizational response following disruption emergence.",

    "RV5":
        "Captures how quickly significant operational consequences materialize following a disruption."
}


rv[
    "Content_Domain_Rationale"
] = rv[
    "Item"
].map(
    rationales
)


rv[
    "Item_Status"
] = (
    "Candidate item — "
    "requires dedicated expert content validation"
)


# =====================================================================
# 12. RAW / NORMALIZED THEME COUNTS
# =====================================================================
#
# Because this Risk Velocity file contains five candidate domains
# rather than the original verbatim qualitative theme-level dataset,
# we explicitly distinguish:
#
#     candidate domain
#     normalized candidate theme
#
# We do NOT fabricate raw-theme counts.
#
# =====================================================================

rv[
    "Number_of_Raw_Themes"
] = np.nan


rv[
    "Raw_Themes_Kept"
] = np.nan


rv[
    "Raw_Themes_Merged"
] = np.nan


# =====================================================================
# 13. CREATE FINAL EVIDENCE TABLE
# =====================================================================

final_evidence = rv[
    [
        "Item",
        "Risk_Velocity_Domain",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Candidate_Questionnaire_Item",
        "Content_Domain_Rationale",
        "Item_Status",
        "Number_of_Raw_Themes",
        "Raw_Themes_Kept",
        "Raw_Themes_Merged"
    ]
].copy()


# =====================================================================
# 14. CREATE SAFE RANK
# =====================================================================
#
# IMPORTANT:
# We explicitly remove any existing Rank column before creating it.
# This eliminates the previous:
#
# ValueError: cannot insert Rank, already exists
#
# =====================================================================

if "Rank" in final_evidence.columns:

    final_evidence = final_evidence.drop(
        columns=["Rank"]
    )


final_evidence.insert(
    0,
    "Rank",
    range(
        1,
        len(final_evidence) + 1
    )
)


# =====================================================================
# 15. DIMENSION SUMMARY
# =====================================================================

dimension_summary = (
    rebuilt_prevalence
    .copy()
)


dimension_summary[
    "Number_of_Normalized_Themes"
] = 1


# Number of items mapped to each domain

domain_item_counts = (
    rv
    .groupby(
        "Risk_Velocity_Domain"
    )
    .size()
    .rename(
        "Number_of_Questionnaire_Items"
    )
    .reset_index()
)


dimension_summary = dimension_summary.merge(
    domain_item_counts,
    on="Risk_Velocity_Domain",
    how="left"
)


# =====================================================================
# 16. INCLUDED RV ITEMS
# =====================================================================

included_items = (
    rv
    .groupby(
        "Risk_Velocity_Domain"
    )[
        "Item"
    ]
    .apply(
        lambda x:
        "; ".join(
            x.astype(str)
        )
    )
    .reset_index()
)


dimension_summary = dimension_summary.merge(
    included_items,
    on="Risk_Velocity_Domain",
    how="left"
)


dimension_summary = dimension_summary.rename(
    columns={
        "Item":
            "Included_RV_Items"
    }
)


# =====================================================================
# 17. DIMENSION RANKING
# =====================================================================

dimension_summary = (
    dimension_summary
    .sort_values(
        [
            "Experts_Mentioning",
            "Expert_Prevalence_%",
            "Number_of_Normalized_Themes"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


if "Rank" in dimension_summary.columns:

    dimension_summary = dimension_summary.drop(
        columns=["Rank"]
    )


dimension_summary.insert(
    0,
    "Rank",
    range(
        1,
        len(dimension_summary) + 1
    )
)


# =====================================================================
# 18. DIMENSION THEMES
# =====================================================================

dimension_themes = rv[
    [
        "Risk_Velocity_Domain",
        "Item",
        "Candidate_Questionnaire_Item",
        "Experts_Mentioning",
        "Expert_Prevalence_%",
        "Prevalence_Category",
        "Content_Domain_Rationale"
    ]
].copy()


# =====================================================================
# 19. PARTICIPANT MATRIX
# =====================================================================

participant_matrix_final = matrix[
    [
        "Risk_Velocity_Domain"
    ]
    +
    participant_columns
].copy()


# =====================================================================
# 20. PARTICIPANT COVERAGE
# =====================================================================

coverage_records = []


for _, row in participant_matrix_final.iterrows():

    pass


for participant in participant_columns:

    number_domains = int(
        (
            participant_matrix_final[
                participant
            ] > 0
        ).sum()
    )


    percentage = round(
        number_domains /
        max(
            len(
                participant_matrix_final
            ),
            1
        )
        *
        100,
        1
    )


    if percentage >= 75:

        category = "Very High"

    elif percentage >= 50:

        category = "High"

    elif percentage >= 25:

        category = "Moderate"

    else:

        category = "Low"


    coverage_records.append({

        "Participant":
            participant,

        "Final_RV_Domains_Mentioned":
            number_domains,

        "Percentage_of_Final_Domains_Covered":
            percentage,

        "Coverage_Category":
            category
    })


participant_coverage_final = pd.DataFrame(
    coverage_records
)


# =====================================================================
# 21. UNMAPPED THEMES
# =====================================================================
#
# In this particular input, all five candidate domains are already
# mapped. Therefore the expected result is zero unmapped domains.
#
# =====================================================================

unmapped = rv[
    rv[
        "Risk_Velocity_Domain"
    ].isna()
    |
    (
        rv[
            "Risk_Velocity_Domain"
        ]
        .astype(str)
        .str.strip()
        == ""
    )
].copy()


if len(unmapped) == 0:

    unmapped_output = pd.DataFrame({

        "Status": [
            "PASS"
        ],

        "Unmapped_RV_Domains": [
            0
        ]
    })

else:

    unmapped_output = unmapped[
        [
            "Item",
            "Risk_Velocity_Domain"
        ]
    ].copy()


# =====================================================================
# 22. DECISION SUMMARY
# =====================================================================

decision_summary = pd.DataFrame({

    "Decision_Component": [

        "Candidate Risk Velocity items",

        "Risk Velocity content domains",

        "Highest-prevalence domain",

        "Lowest-prevalence domain",

        "All candidate domains mapped",

        "All five items have wording",

        "Next methodological stage"
    ],

    "Result": [

        len(rv),

        rv[
            "Risk_Velocity_Domain"
        ].nunique(),

        dimension_summary.iloc[0][
            "Risk_Velocity_Domain"
        ],

        dimension_summary.iloc[-1][
            "Risk_Velocity_Domain"
        ],

        len(unmapped) == 0,

        rv[
            "Candidate_Questionnaire_Item"
        ].notna().all(),

        "Literature triangulation + dedicated expert content validation"
    ]
})


# =====================================================================
# 23. CODING AUDIT
# =====================================================================

coding_audit = pd.DataFrame({

    "Audit_Item": [

        "Number of candidate items",

        "Number of candidate domains",

        "Participants",

        "Duplicate items",

        "Duplicate domains",

        "Items without questionnaire wording",

        "Domains without item mapping"
    ],

    "Result": [

        len(rv),

        rv[
            "Risk_Velocity_Domain"
        ].nunique(),

        number_of_participants,

        int(
            rv[
                "Item"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Risk_Velocity_Domain"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Candidate_Questionnaire_Item"
            ]
            .isna()
            .sum()
        ),

        int(
            dimension_summary[
                "Number_of_Questionnaire_Items"
            ]
            .eq(0)
            .sum()
        )
    ]
})


# =====================================================================
# 24. CONSTRUCT STATISTICS
# =====================================================================

construct_statistics = pd.DataFrame({

    "Construct": [
        "Risk Velocity"
    ],

    "Participants": [
        number_of_participants
    ],

    "Candidate_Items": [
        len(rv)
    ],

    "Candidate_Content_Domains": [
        rv[
            "Risk_Velocity_Domain"
        ].nunique()
    ],

    "Highest_Domain_Prevalence_%": [
        dimension_summary[
            "Expert_Prevalence_%"
        ].max()
    ],

    "Lowest_Domain_Prevalence_%": [
        dimension_summary[
            "Expert_Prevalence_%"
        ].min()
    ],

    "Mean_Domain_Prevalence_%": [
        round(
            dimension_summary[
                "Expert_Prevalence_%"
            ].mean(),
            1
        )
    ],

    "All_Domains_Mapped": [
        len(unmapped) == 0
    ],

    "All_Items_Worded": [
        rv[
            "Candidate_Questionnaire_Item"
        ].notna().all()
    ]
})


# =====================================================================
# 25. QUALITY CHECKS
# =====================================================================

quality_checks = pd.DataFrame({

    "Check": [

        "Input file exists",

        "Participant columns detected",

        "Number of participants",

        "Five candidate RV items present",

        "Five candidate RV domains present",

        "Duplicate item rows",

        "Duplicate domain rows",

        "Unmapped domains",

        "Missing questionnaire items",

        "Rank column duplicated",

        "Prevalence values available"
    ],

    "Result": [

        True,

        len(participant_columns) > 0,

        number_of_participants,

        len(rv) == 5,

        rv[
            "Risk_Velocity_Domain"
        ].nunique() == 5,

        int(
            rv[
                "Item"
            ]
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Risk_Velocity_Domain"
            ]
            .duplicated()
            .sum()
        ),

        len(unmapped),

        int(
            rv[
                "Candidate_Questionnaire_Item"
            ]
            .isna()
            .sum()
        ),

        int(
            final_evidence.columns
            .duplicated()
            .sum()
        ),

        int(
            rv[
                "Expert_Prevalence_%"
            ]
            .notna()
            .sum()
        )
    ]
})


# =====================================================================
# 26. METHODOLOGY NOTE
# =====================================================================

methodology_note = pd.DataFrame({

    "Component": [

        "Input",

        "Analytical unit",

        "Participants",

        "Risk Velocity domains",

        "Candidate items",

        "Participant prevalence",

        "Use of Python",

        "Important limitation",

        "Next stage"
    ],

    "Description": [

        "Risk_Velocity_Coding_20260827_124722.xlsx.",

        "Participant-level candidate Risk Velocity domains and their corresponding preliminary questionnaire items.",

        f"{number_of_participants} experts/participants represented in the participant matrix.",

        "Rapid Disruption Emergence; Rapid Disruption Escalation; Rapid Disruption Propagation; Limited Response Window; Rapid Operational Consequences / Short Time-to-Impact.",

        "Five preliminary items RV1–RV5 were mapped one-to-one to the five candidate content domains.",

        "Domain prevalence was recalculated directly from the 26-participant matrix rather than relying on manually entered prevalence values.",

        "Python was used for data organization, prevalence calculation, ranking, dimensional structuring, duplicate checks, quality assurance and reproducible Excel-file generation.",

        "The Risk Velocity material represents candidate content-development evidence. It should not be presented as equivalent to verbatim qualitative responses specifically elicited for a pre-established Risk Velocity construct.",

        "The next stage is literature triangulation of RV1–RV5 followed by dedicated expert content validation before the main quantitative PLS-SEM survey."
    ]
})


# =====================================================================
# 27. CREATE OUTPUT FILE
# =====================================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUTPUT_FILE = Path(
    f"Risk_Velocity_FINAL_QUALITATIVE_ANALYSIS_{timestamp}.xlsx"
)


# =====================================================================
# 28. WRITE ALL SHEETS
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:


    final_evidence.to_excel(
        writer,
        sheet_name="01_Original_RV_Evidence",
        index=False
    )


    item_development_df.to_excel(
        writer,
        sheet_name="02_Item_Development",
        index=False
    )


    final_evidence.to_excel(
        writer,
        sheet_name="03_Cleaned_RV_Evidence",
        index=False
    )


    coding_audit.to_excel(
        writer,
        sheet_name="04_Coding_Audit",
        index=False
    )


    participant_matrix_final.to_excel(
        writer,
        sheet_name="05_Participant_Matrix",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="06_Domain_Summary",
        index=False
    )


    rebuilt_prevalence.to_excel(
        writer,
        sheet_name="07_Domain_Prevalence",
        index=False
    )


    decision_summary.to_excel(
        writer,
        sheet_name="08_Decision_Summary",
        index=False
    )


    participant_coverage_final.to_excel(
        writer,
        sheet_name="09_Participant_Coverage",
        index=False
    )


    unmapped_output.to_excel(
        writer,
        sheet_name="10_Unmapped_Themes",
        index=False
    )


    final_evidence.to_excel(
        writer,
        sheet_name="11_Final_RV_Evidence",
        index=False
    )


    dimension_summary.to_excel(
        writer,
        sheet_name="12_RV_Dimension_Summary",
        index=False
    )


    dimension_themes.to_excel(
        writer,
        sheet_name="13_RV_Dimension_Themes",
        index=False
    )


    construct_statistics.to_excel(
        writer,
        sheet_name="14_Construct_Statistics",
        index=False
    )


    quality_checks.to_excel(
        writer,
        sheet_name="15_Quality_Checks",
        index=False
    )


    methodology_note.to_excel(
        writer,
        sheet_name="16_Methodology_Note",
        index=False
    )


# =====================================================================
# 29. PROFESSIONAL EXCEL FORMATTING
# =====================================================================

wb = load_workbook(
    OUTPUT_FILE
)


header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78"
)


header_font = Font(
    bold=True,
    color="FFFFFF"
)


for ws in wb.worksheets:


    ws.freeze_panes = "A2"


    # Header formatting

    for cell in ws[1]:

        cell.fill = header_fill

        cell.font = header_font

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )


    # Body formatting

    for row in ws.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )


    # Column widths

    for column_cells in ws.columns:

        column_letter = get_column_letter(
            column_cells[0].column
        )

        max_length = 0


        for cell in column_cells:

            if cell.value is None:
                continue


            max_length = max(
                max_length,
                len(
                    str(
                        cell.value
                    )
                )
            )


        ws.column_dimensions[
            column_letter
        ].width = min(
            max(
                max_length + 2,
                12
            ),
            65
        )


    # Autofilter

    if ws.max_row > 1:

        ws.auto_filter.ref = (
            ws.dimensions
        )


# Save

wb.save(
    OUTPUT_FILE
)


# =====================================================================
# 30. FINAL CONSOLE REPORT
# =====================================================================

print("\n")
print("=" * 90)
print("RISK VELOCITY FINAL FILE CREATED")
print("=" * 90)

print(
    "\nParticipants:",
    number_of_participants
)

print(
    "Candidate items:",
    len(rv)
)

print(
    "Candidate domains:",
    rv[
        "Risk_Velocity_Domain"
    ].nunique()
)

print(
    "Unmapped domains:",
    len(unmapped)
)

print(
    "Duplicate item rows:",
    int(
        rv[
            "Item"
        ]
        .duplicated()
        .sum()
    )
)

print(
    "Duplicate domain rows:",
    int(
        rv[
            "Risk_Velocity_Domain"
        ]
        .duplicated()
        .sum()
    )
)


print("\n")
print("=" * 90)
print("RISK VELOCITY DOMAIN SUMMARY")
print("=" * 90)

print(
    dimension_summary.to_string(
        index=False
    )
)


print("\n")
print("=" * 90)
print("FINAL OUTPUT")
print("=" * 90)

print(
    OUTPUT_FILE.resolve()
)

print("\nDone.")

RISK VELOCITY — FINAL QUALITATIVE EVIDENCE ANALYSIS

Input file:
C:\Users\1886199\OneDrive - UET\Drive G\Data science coding\VSCode programming\Code_Blockchain_Resilience_Risk_Sensing_Decision_Intelligence_2\Risk_Velocity_FINAL_EVIDENCE_20260828_132957.xlsx

Available sheets:
 - 01_Final_RV_Evidence
 - 02_Item_Development
 - 03_Domain_Prevalence
 - 04_Participant_Matrix
 - 05_Participant_Coverage
 - 06_Methodology_Note
 - 07_Quality_Checks

Number of participants: 26


RISK VELOCITY FINAL FILE CREATED

Participants: 26
Candidate items: 5
Candidate domains: 5
Unmapped domains: 0
Duplicate item rows: 0
Duplicate domain rows: 0


RISK VELOCITY DOMAIN SUMMARY
 Rank                                  Risk_Velocity_Domain  Experts_Mentioning  Expert_Prevalence_% Prevalence_Category  Number_of_Normalized_Themes  Number_of_Questionnaire_Items Included_RV_Items
    1                          Rapid Disruption Propagation                  24                 92.3           Very High                 